# Data Contract — Case Ambev Long Neck NENO
## Processo Seletivo Insper Junior | Engenharia Tecnológica — 50ª Gestão

**Grupo 3** | Mateus Loureiro
**Data:** 11 de março de 2026
**Fonte de Dados:** `Analise_LongNeck_WSNP - Sem repostas.xlsx` (6 abas)

---

### Metodologia
1. **Entendimento do Cliente** ✅
2. **Data Contract** ← *estamos aqui*
3. Análise Univariada
4. Árvore de Hipóteses (MECE)
5. Análise Bivariada
6. Entrega (PPTX, XLSX, Dashboard React+Vite+TS+Styled Components)

### O que é um Data Contract?
Documenta **cada campo** com 8 atributos:

| Atributo | Descrição |
|---|---|
| **Nome do Campo** | Identificador da variável |
| **Tipo** | int, float, string, datetime, boolean |
| **Formato** | Padrão de representação |
| **Descrição Semântica** | Significado no contexto de negócio |
| **Domínio de Valores** | Faixa ou conjunto de valores possíveis |
| **Regras de Negócio** | Restrições, fórmulas, dependências |
| **Nulabilidade** | Se aceita valores nulos/vazios |
| **Origem** | De onde o dado vem |

In [ ]:
%matplotlib inline
%pip install openpyxl --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import matplotlib.patheffects as pe
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from IPython.display import display, HTML

# Configuração visual global
plt.rcParams.update({
    'figure.facecolor': '#FAFAFA',
    'axes.facecolor': '#FAFAFA',
    'font.family': 'sans-serif',
    'font.size': 11,
    'figure.dpi': 100
})

CORES = {
    'azul_escuro': '#1B2A4A',
    'azul_medio': '#2D5F8A',
    'azul_claro': '#5BA4CF',
    'ambar': '#F5A623',
    'vermelho': '#E74C3C',
    'verde': '#27AE60',
    'cinza_claro': '#ECF0F1',
    'cinza_medio': '#95A5A6',
    'branco': '#FFFFFF',
    'bg': '#FAFAFA'
}

BASE_DIR = Path('.').resolve()
EXCEL_FILE = None
# Priorizar arquivo original "Sem repostas" sobre qualquer derivado (ignorar ~$ temporários)
for p in [BASE_DIR, BASE_DIR.parent]:
    candidates = [f for f in p.glob('*Sem*repostas*.xlsx') if not f.name.startswith('~$')]
    if candidates:
        EXCEL_FILE = candidates[0]
        break
if EXCEL_FILE is None:
    for p in [BASE_DIR, BASE_DIR.parent]:
        all_xlsx = list(p.glob('*Long*Neck*.xlsx')) + list(p.glob('*longneck*.xlsx'))
        candidates = [f for f in all_xlsx if not f.name.startswith('~$')]
        if candidates:
            EXCEL_FILE = candidates[0]
            break

if EXCEL_FILE is None:
    print("⚠️  Arquivo Excel não encontrado. Coloque na mesma pasta do notebook.")
else:
    print(f"✅ Arquivo encontrado: {EXCEL_FILE.name}")

def show_contract(df, title=""):
    display(HTML(f"<h4>{title}</h4>"))
    display(df)

print(f"Pandas version: {pd.__version__}")
print("📋 Setup concluído — com matplotlib para visualizações.")

---
---

# PARTE 1 — Data Contract

> **Objetivo:** Mapear e documentar formalmente cada campo das 6 abas do Excel com 8 atributos por variável — garantindo rastreabilidade e qualidade dos dados antes das análises.

## 1. Visão Geral das Fontes de Dados

| # | Aba | Descrição | Granularidade | Período |
|---|-----|-----------|---------------|---------|
| 1 | **Cenário atual BR** | Visão mensal consolidada por GEO/REG | Mensal × Região | Jan–Fev 2026 |
| 2 | **Custos de transferência** | Custos logísticos, MACOs, custos de produção | SKU × Rota | Estático |
| 3 | **Produção PCP** | Programação semanal AQ541 e PE541 | Semanal × SKU × Planta | Fev 2026 (4 sem) |
| 4 | **Transferências Programadas** | Transferências já planejadas SP→NE | Semanal × SKU × Destino | Fev 2026 (4 sem) |
| 5 | **Cenário Divulgado** | WSNP semanal detalhado (baseline) | Semanal × SKU × Sub-região | Fev 2026 (4 sem) |
| 6 | **Cenário com Nova Demanda** | Malzbier +30% (problema a resolver) | Semanal × SKU × Sub-região | Fev 2026 (4 sem) |

**SKUs:** Patagonia, Goose Island, Malzbier Brahma, Colorado Lager (todos LN)
**Região foco:** NENO — sub-regiões: Mapapi, NE Norte, NE Sul, NO Araguaia, NO Centro
**Choque:** Malzbier Brahma +30%

---
## 2. Data Contract — Aba 1: "Cenário Atual BR"

Visão macro mensal de todo o portfólio Long Neck SK269 por GEO/REG.
Janeiro (26 dias úteis) e Fevereiro (24 dias úteis).
Regiões: MG, SP, NENO, CO, RJ, SUL, Export, TOTAL.

In [ ]:
contract_aba1 = pd.DataFrame([
    {'Campo': 'GEO/REG', 'Tipo': 'string', 'Formato': 'Texto',
     'Descrição Semântica': 'Região geográfica de demanda/distribuição.',
     'Domínio': "MG, SP, NENO, CO, RJ, SUL, Export, TOTAL",
     'Regras de Negócio': 'TOTAL = soma regiões. NENO = NE+Norte (foco).',
     'Nulabilidade': 'NOT NULL', 'Origem': 'WSNP Ambev'},
    {'Campo': 'Demanda (HL)', 'Tipo': 'float', 'Formato': '#,##0',
     'Descrição Semântica': 'Volume previsto de venda mensal em hectolitros.',
     'Domínio': '[0, +∞) — NENO Jan: 200.754, Fev: 179.674',
     'Regras de Negócio': 'Soma regiões = TOTAL. BIAS GEOs: +9%.',
     'Nulabilidade': 'NOT NULL', 'Origem': 'GEOs (equipes regionais)'},
    {'Campo': 'PROD Real (HL)', 'Tipo': 'float', 'Formato': '#,##0',
     'Descrição Semântica': 'Volume realizado de produção no mês.',
     'Domínio': '[0, +∞) — só Jan. SP: 127.269, NENO: 70.399',
     'Regras de Negócio': 'Regiões sem planta = 0. Só Janeiro.',
     'Nulabilidade': 'Nullable (0)', 'Origem': 'MES/ERP'},
    {'Campo': 'PROD WSNP (HL)', 'Tipo': 'float', 'Formato': '#,##0',
     'Descrição Semântica': 'Volume programado conforme WSNP.',
     'Domínio': '[0, +∞) — NENO Fev: 158.000',
     'Regras de Negócio': 'Pode diferir do realizado.',
     'Nulabilidade': 'Nullable (0)', 'Origem': 'WSNP Ambev'},
    {'Campo': 'PROD 1W (HL)', 'Tipo': 'float', 'Formato': '#,##0',
     'Descrição Semântica': 'Vol. programado para produção (horizonte 1 semana, mais firme).',
     'Domínio': '[0, +∞) — NENO: 176.297',
     'Regras de Negócio': 'Mais firme que WSNP. Só Jan.',
     'Nulabilidade': 'Nullable', 'Origem': 'PCP'},
    {'Campo': 'EI Mês (HL)', 'Tipo': 'float', 'Formato': '#,##0',
     'Descrição Semântica': 'Estoque inicial do mês.',
     'Domínio': '[0, +∞) — NENO Jan: 136.190',
     'Regras de Negócio': 'EI mês N = EF mês N-1.',
     'Nulabilidade': 'NOT NULL', 'Origem': 'WMS'},
    {'Campo': 'Suf.ini (d)', 'Tipo': 'float', 'Formato': '#,##0 dias',
     'Descrição Semântica': 'Suficiência inicial = EI / demanda linear diária.',
     'Domínio': '[0, +∞) — NENO Jan: 15.39d',
     'Regras de Negócio': 'DOI mín = 12 dias. < 12 = alerta.',
     'Nulabilidade': 'NOT NULL', 'Origem': 'Calculado'},
    {'Campo': 'Transf. Malha (HL)', 'Tipo': 'float', 'Formato': '#,##0',
     'Descrição Semântica': 'Volume de transferência inter-regional. + recebe, - envia.',
     'Domínio': '(-∞, +∞) — SP envia, NENO recebe.',
     'Regras de Negócio': 'Soma ≈ 0 (conservação).',
     'Nulabilidade': 'NOT NULL', 'Origem': 'Planej. logístico'},
    {'Campo': 'EFM (HL)', 'Tipo': 'float', 'Formato': '#,##0',
     'Descrição Semântica': 'Estoque final projetado do mês.',
     'Domínio': '(-∞, +∞) — negativo = ruptura.',
     'Regras de Negócio': 'EFM = EI + PROD - Demanda + Transf.',
     'Nulabilidade': 'NOT NULL', 'Origem': 'Calculado'},
    {'Campo': 'Suf.f (d)', 'Tipo': 'float', 'Formato': '#,##0 dias',
     'Descrição Semântica': 'Suficiência final em dias de cobertura.',
     'Domínio': '(-∞, +∞) — NENO Jan: 11.54d (< DOI!)',
     'Regras de Negócio': 'DOI mín = 12d. NENO Jan fecha abaixo!',
     'Nulabilidade': 'NOT NULL', 'Origem': 'Calculado'}
])
show_contract(contract_aba1, '📋 Data Contract — Aba 1: Cenário Atual BR')

In [ ]:
# Dados Brutos — Aba 1

dados_jan = pd.DataFrame({
    'GEO/REG': ['MG', 'SP', 'NENO', 'CO', 'RJ', 'SUL', 'Export', 'TOTAL'],
    'Demanda': [65538.3, 173637.4, 200754.3, 104493.3, 111219.4, 116261.4, 67055.4, 838959.5],
    'PROD Real': [0, 127269, 70399, 50470, 130770, 41205, 0, 420113],
    'PROD WSNP': [0, 15300, 9342.9, 5142.9, 17571.4, 3985.7, 0, 51342.9],
    'PROD 1W': [0, 238579.2, 176297.4, 99590.4, 339789.6, 104891.4, 0, 959148],
    'EI Mês': [28580.4, 114472.8, 136190, 61952.4, 84834, 86758.2, 0, 495419.4],
    'Suf.ini(d)': [11.34, 17.14, 15.39, 15.41, 19.83, 19.40, 0, 15.35],
    'Transf.Malha': [11585.9, -12423.9, 7455.6, 7619.6, -29886.9, 12433.2, 3212.1, -4.5],
    'EFM': [32771.5, 120086.5, 86426.7, 81961.6, 87765.2, 77470.0, -21611.0, 486481.4],
    'Suf.f(d)': [10.80, 17.44, 11.54, 17.44, 16.12, 15.95, -10.55, 13.53]
})

dados_fev = pd.DataFrame({
    'GEO/REG': ['MG', 'SP', 'NENO', 'CO', 'RJ', 'SUL', 'Export', 'TOTAL'],
    'Demanda': [72849.3, 165262.4, 179673.8, 112802.3, 130649.4, 116585.2, 49179.6, 827002.0],
    'PROD WSNP': [0, 253602.3, 158000, 87685.7, 288282.9, 77748.4, 0, 863199.3],
    'Transf.Malha': [75118.4, -75583.1, 34507, 21794.0, -174400.9, 38221.7, 38158.1, -118.0],
    'EFM': [35040.6, 132843.2, 99259.9, 78639.0, 70997.7, 76855.0, -32632.5, 493635.4],
    'Suf.f(d)': [12.89, 25.53, 13.26, 21.10, 20.30, 17.45, -26.91, 14.33]
})

print("JANEIRO 2026 (26 dias úteis)")
display(dados_jan)
print()
print("FEVEREIRO 2026 (24 dias úteis)")
display(dados_fev)

### Insights — Aba 1
1. **NENO é a 2ª maior região** (~24% do Brasil)
2. **Suficiência NENO Janeiro: 11.54d** — abaixo do DOI mínimo de 12d ⚠️
3. **NENO é receptor líquido** de transferências (+7.456 Jan, +34.507 Fev)
4. **SP é o maior remetente** (-12.424 Jan, -75.583 Fev)
5. **Export tem EF negativo** (ruptura) — mercado interno priorizado

---
## 3. Data Contract — Aba 2: "Custos de Transferência"

3 seções: Custos de transferência (6 rotas SP→NE), MACOs (3 SKUs), Custos de produção (3 SKUs).
**Premissa:** Rodoviário = +60% sobre Cabotagem + 5% avaria.

In [ ]:
contract_aba2 = pd.DataFrame([
    {'Campo': 'SKU', 'Tipo': 'string', 'Formato': 'Nome completo',
     'Descrição Semântica': 'Stock Keeping Unit — produto no portfólio LN.',
     'Domínio': 'Colorado, Goose Island, Malzbier (Patagonia NÃO aparece)',
     'Regras de Negócio': 'Cada SKU tem custo e MACO específicos.',
     'Nulabilidade': 'NOT NULL', 'Origem': 'SAP/ERP'},
    {'Campo': 'Origem', 'Tipo': 'string', 'Formato': 'BR## - Nome - UF',
     'Descrição Semântica': 'Fábrica de origem da transferência.',
     'Domínio': 'BR16 Jacareí (SP), BR23 Jaguariúna (SP)',
     'Regras de Negócio': 'Colorado/Goose: Jacareí. Malzbier: Jaguariúna.',
     'Nulabilidade': 'NOT NULL', 'Origem': 'Cadastro Ambev'},
    {'Campo': 'Destino', 'Tipo': 'string', 'Formato': 'BR## - Nome - UF',
     'Descrição Semântica': 'CD/Fábrica de destino no NE.',
     'Domínio': 'BR04 Camaçari (BA), BR06 Fonte Mata (PB)',
     'Regras de Negócio': 'Fonte Mata é mais caro que Camaçari.',
     'Nulabilidade': 'NOT NULL', 'Origem': 'Cadastro Ambev'},
    {'Campo': 'Custo Transf (R$/HL)', 'Tipo': 'float', 'Formato': 'R$ #,##0.00',
     'Descrição Semântica': 'Custo unitário transferência via cabotagem.',
     'Domínio': '[R$76.59, R$95.33]',
     'Regras de Negócio': 'Cabotagem 25d lead time. Rodo = ×1.60 + 5% avaria.',
     'Nulabilidade': 'NOT NULL', 'Origem': 'Logística'},
    {'Campo': 'MACO (R$/HL)', 'Tipo': 'float', 'Formato': 'R$ #,##0',
     'Descrição Semântica': 'Margem de Contribuição por HL.',
     'Domínio': 'Malzbier R$285, Colorado R$300, Goose R$350',
     'Regras de Negócio': 'Goose maior margem. Malzbier menor.',
     'Nulabilidade': 'NOT NULL', 'Origem': 'Controladoria'},
    {'Campo': 'Custo Produção (R$/HL)', 'Tipo': 'float', 'Formato': 'R$ #,##0',
     'Descrição Semântica': 'Custo variável de produção por HL (plantas NE).',
     'Domínio': 'Malzbier R$149, Colorado R$150, Goose R$155',
     'Regras de Negócio': 'Custos próximos. Diferença de margem vem da receita.',
     'Nulabilidade': 'NOT NULL', 'Origem': 'Controladoria'}
])
show_contract(contract_aba2, '📋 Data Contract — Aba 2: Custos de Transferência')

In [ ]:
custos_transf = pd.DataFrame({
    'SKU': ['Colorado', 'Colorado', 'Goose Island', 'Goose Island', 'Malzbier', 'Malzbier'],
    'Origem': ['Jacareí SP', 'Jacareí SP', 'Jacareí SP', 'Jacareí SP', 'Jaguariúna SP', 'Jaguariúna SP'],
    'Destino': ['Camaçari BA', 'Fonte Mata PB', 'Camaçari BA', 'Fonte Mata PB', 'Camaçari BA', 'Fonte Mata PB'],
    'Cabo R$/HL': [76.59, 82.08, 82.40, 88.30, 84.58, 95.33],
    'Rodo R$/HL': [76.59*1.6, 82.08*1.6, 82.40*1.6, 88.30*1.6, 84.58*1.6, 95.33*1.6],
    'Lead Cabo': ['25d']*6, 'Lead Rodo': ['6d']*6
})
display(custos_transf)

macos = pd.DataFrame({
    'SKU': ['Colorado', 'Goose Island', 'Malzbier'],
    'MACO R$/HL': [300, 350, 285],
    'Custo Prod R$/HL': [150, 155, 149],
    'Margem Bruta R$/HL': [150, 195, 136]
})
print()
display(macos)

print()
print("Margem líquida Malzbier por cenário de supply:")
cenarios = pd.DataFrame({
    'Cenário': ['Produção Local NE', 'Cabo→Camaçari', 'Cabo→Fonte Mata', 'Rodo→Camaçari', 'Rodo→Fonte Mata'],
    'Custo Prod': [149, 149, 149, 149, 149],
    'Custo Transf': [0, 84.58, 95.33, 84.58*1.6, 95.33*1.6],
    'Custo Total': [149, 233.58, 244.33, 149+84.58*1.6, 149+95.33*1.6],
    'Margem': [136, 285-233.58, 285-244.33, 285-(149+84.58*1.6), 285-(149+95.33*1.6)],
    'Lead Time': ['0d', '25d', '25d', '6d', '6d'],
    'Avaria': ['0%', '0%', '0%', '5%', '5%']
})
display(cenarios)

### Insights — Aba 2
1. **Produção local sempre preferível**: margem R$136/HL vs R$51/HL (cabo) ou negativa (rodo)
2. **Rodoviário → Fonte Mata é inviável**: custo R$301.53/HL > MACO R$285/HL
3. **Lead time é o trade-off central**: Cabo 25d (barato) vs Rodo 6d (caro+avaria)
4. **Goose tem a maior MACO (R$350)**: cada HL deslocado dela custa mais

---
## 4. Data Contract — Aba 3: "Produção PCP"

| Planta | Código | Local | Nominal grf/h | Cap. Semanal HL |
|--------|--------|-------|--------------|----------------|
| AQ541 | BR03 Aquiraz | CE | 72.000 | 12.600 |
| PE541 | BR31 Pernambuco | PE | 108.000 | 27.000 |

4 semanas de Fev 2026. Ambas usam linha L541 compartilhada entre SKUs.

In [ ]:
contract_aba3 = pd.DataFrame([
    {'Campo': 'SCSLocation', 'Tipo': 'string',
     'Descrição Semântica': 'Planta de produção.',
     'Domínio': 'BR03 Aquiraz/CE, BR31 Pernambuco/PE',
     'Regras de Negócio': 'Únicas plantas LN no NENO.',
     'Nulabilidade': 'NOT NULL', 'Origem': 'SCS Ambev'},
    {'Campo': 'Linha', 'Tipo': 'string',
     'Descrição Semântica': 'Linha de produção.',
     'Domínio': 'L541 (única)',
     'Regras de Negócio': 'Compartilhada entre SKUs.',
     'Nulabilidade': 'NOT NULL', 'Origem': 'Engenharia'},
    {'Campo': 'Nominal grf/h', 'Tipo': 'int',
     'Descrição Semântica': 'Velocidade nominal garrafas/hora.',
     'Domínio': '72.000 (AQ), 108.000 (PE)',
     'Regras de Negócio': 'PE541 é 50% mais rápida.',
     'Nulabilidade': 'NOT NULL', 'Origem': 'Ficha técnica'},
    {'Campo': 'Cap Semanal HL', 'Tipo': 'int',
     'Descrição Semântica': 'Capacidade máxima semanal.',
     'Domínio': '12.600 (AQ), 27.000 (PE)',
     'Regras de Negócio': 'Mensal: AQ=50.400, PE=108.000.',
     'Nulabilidade': 'NOT NULL', 'Origem': 'Cálculo'},
    {'Campo': 'Item (SKU)', 'Tipo': 'string',
     'Descrição Semântica': 'Produto a ser produzido.',
     'Domínio': 'AQ: Malzbier/Patagonia/Colorado. PE: 6 SKUs.',
     'Regras de Negócio': 'Nem todo SKU roda toda semana (setup).',
     'Nulabilidade': 'NOT NULL', 'Origem': 'PCP'},
    {'Campo': 'Volume Semanal HL', 'Tipo': 'float',
     'Descrição Semântica': 'Volume programado por semana.',
     'Domínio': '[0, 27.000]',
     'Regras de Negócio': 'Soma por planta ≤ Capacidade.',
     'Nulabilidade': 'NOT NULL', 'Origem': 'PCP'}
])
show_contract(contract_aba3, '📋 Data Contract — Aba 3: Produção PCP')

In [ ]:
print("AQ541 — Aquiraz/CE | 72.000 grf/h | Cap: 12.600 HL/sem")
aq541 = pd.DataFrame({
    'SKU': ['Malzbier', 'Patagonia', 'Colorado', 'TOTAL'],
    'W0': [0, 12240, 0, 12240], 'W1': [9000, 1800, 0, 10800],
    'W2': [7560, 5040, 0, 12600], 'W3': [0, 12600, 0, 12600],
    'Total Fev': [16560, 31680, 0, 48240]
})
display(aq541)

print()
print("PE541 — Pernambuco/PE | 108.000 grf/h | Cap: 27.000 HL/sem")
pe541 = pd.DataFrame({
    'SKU': ['Brahma Zero', 'Goose Island', 'Malzbier', 'Colorado', 'Skol Beats', 'Budweiser Zero', 'TOTAL'],
    'W0': [0, 5400, 16200, 5400, 0, 0, 27000],
    'W1': [0, 14400, 0, 0, 0, 5400, 19800],
    'W2': [0, 0, 12960, 10800, 3240, 0, 27000],
    'W3': [3600, 12600, 0, 0, 0, 10800, 27000],
    'Total Fev': [3600, 32400, 29160, 16200, 3240, 16200, 100800]
})
display(pe541)

print()
print(f"Utilização AQ541: {48240:,}/{50400:,} = {48240/50400*100:.1f}%")
print(f"Utilização PE541: {100800:,}/{108000:,} = {100800/108000*100:.1f}%")
print(f"Ociosa: AQ {50400-48240:,} + PE {108000-100800:,} = {9360:,} HL")
print(f"Malzbier NE: AQ {16560:,} + PE {29160:,} = {45720:,} HL")

### Insights — Aba 3
1. **PE541 a 93.3%**, AQ541 a 95.7% — pouca folga
2. **Capacidade ociosa total: 9.360 HL** — insuficiente para gap de +11.680
3. **Colorado 0 em AQ541** — potencial realocação
4. **W1 em PE541 tem ociosidade** (19.800/27.000 = 73.3%)

---
## 5. Data Contract — Aba 4: "Transferências Programadas"

**Apenas Goose Island** tem transferências programadas!
Malzbier e Colorado: ZERO. 100% dependem de produção local.

In [ ]:
contract_aba4 = pd.DataFrame([
    {'Campo': 'Regional Origem', 'Tipo': 'string',
     'Descrição Semântica': 'Regional de origem.',
     'Domínio': 'REG SE (todas)', 'Regras de Negócio': 'Apenas SE→NE.',
     'Nulabilidade': 'NOT NULL', 'Origem': 'Logística'},
    {'Campo': 'Desc Destino', 'Tipo': 'string',
     'Descrição Semântica': 'CD/Planta de destino no NE.',
     'Domínio': 'Camaçari, Fonte Mata, CDR Bahia',
     'Regras de Negócio': 'Goose vai para os 3.',
     'Nulabilidade': 'NOT NULL', 'Origem': 'Cadastro'},
    {'Campo': 'Modal', 'Tipo': 'string',
     'Descrição Semântica': 'Modal de transporte.',
     'Domínio': 'Cabotagem (100%)',
     'Regras de Negócio': 'Lead 25d. Zero rodoviário.',
     'Nulabilidade': 'NOT NULL', 'Origem': 'Logística'},
    {'Campo': 'Cod Produto', 'Tipo': 'int',
     'Descrição Semântica': 'Código SAP.',
     'Domínio': '65758 (Goose apenas)',
     'Regras de Negócio': 'ZERO Malzbier, ZERO Colorado.',
     'Nulabilidade': 'NOT NULL', 'Origem': 'SAP'},
    {'Campo': 'Volume Semanal HL', 'Tipo': 'float',
     'Descrição Semântica': 'Volume transferido por semana.',
     'Domínio': '[0, 7.200]',
     'Regras de Negócio': 'Cam: 7.200/sem. FM: 4.107 só W0. CDR: 5.400/sem.',
     'Nulabilidade': 'NOT NULL', 'Origem': 'Logística'}
])
show_contract(contract_aba4, '📋 Data Contract — Aba 4: Transferências Programadas')

In [ ]:
transf = pd.DataFrame({
    'Destino': ['Camaçari BA', 'Fonte Mata PB', 'CDR Bahia'],
    'SKU': ['Goose Island']*3, 'Modal': ['Cabotagem']*3,
    'W0': [7200, 4107.4, 5400], 'W1': [7200, 0, 5400],
    'W2': [7200, 0, 5400], 'W3': [7200, 0, 5400],
    'Total': [28800, 4107.4, 21600]
})
display(transf)
print()
print("MALZBIER: 0 HL transferidos | COLORADO: 0 HL transferidos")
print("Malzbier depende 100% da produção local NE")

---
## 6. Data Contract — Aba 5: "Cenário Divulgado" (Baseline)

WSNP semanal: 4 SKUs × 5 sub-regiões × 4 semanas.
Métricas: Demanda, WSNP, EI, Suf.ini, Transf.Interna, Transf.Ext(Cabo/Rodo), Trânsito, EF, Suf.f

In [ ]:
contract_aba5 = pd.DataFrame([
    {'Campo': 'SKU (bloco)', 'Tipo': 'string',
     'Descrição Semântica': 'SKU agrupador.',
     'Domínio': 'Patagonia(70934), Goose(65758), Malzbier(41777), Colorado(51476)',
     'Regras de Negócio': 'Cada bloco: 5 sub-regiões + prod SP/NE.',
     'Nulabilidade': 'NOT NULL', 'Origem': 'WSNP'},
    {'Campo': 'GEO/REG', 'Tipo': 'string',
     'Descrição Semântica': 'Sub-região NENO.',
     'Domínio': 'Mapapi, NE Norte, NE Sul, NO Araguaia, NO Centro',
     'Regras de Negócio': 'Mapapi = maior. NO Araguaia = menor.',
     'Nulabilidade': 'NOT NULL', 'Origem': 'Estrutura Ambev'},
    {'Campo': 'Demanda HL/sem', 'Tipo': 'float',
     'Descrição Semântica': 'Demanda semanal por sub-região.',
     'Domínio': '[0, ~6.300]',
     'Regras de Negócio': 'Soma sub-regiões = total NENO.',
     'Nulabilidade': 'NOT NULL', 'Origem': 'GEOs'},
    {'Campo': 'EI Semana HL', 'Tipo': 'float',
     'Descrição Semântica': 'Estoque início da semana.',
     'Domínio': '[0, +∞)',
     'Regras de Negócio': 'EI_N = EF_{N-1}.',
     'Nulabilidade': 'NOT NULL', 'Origem': 'WMS'},
    {'Campo': 'Suf.ini (d)', 'Tipo': 'float',
     'Descrição Semântica': 'Suficiência inicial = EI/(Demanda/7).',
     'Domínio': '(-∞, +∞)',
     'Regras de Negócio': 'DOI mín 12d.',
     'Nulabilidade': 'NOT NULL', 'Origem': 'Calculado'},
    {'Campo': 'Transf. Interna HL', 'Tipo': 'float',
     'Descrição Semântica': 'Redistribuição dentro NENO.',
     'Domínio': '(-∞, +∞)',
     'Regras de Negócio': 'Soma ≈ 0.',
     'Nulabilidade': 'NOT NULL', 'Origem': 'Planejamento'},
    {'Campo': 'Transf. Ext Cabo HL', 'Tipo': 'float',
     'Descrição Semântica': 'Transferência SP→NE cabotagem.',
     'Domínio': '[0, +∞) — só Goose > 0',
     'Regras de Negócio': 'Lead 25d. Malzbier/Colorado = 0.',
     'Nulabilidade': 'NOT NULL', 'Origem': 'Logística'},
    {'Campo': 'Transf. Ext Rodo HL', 'Tipo': 'float',
     'Descrição Semântica': 'Transferência SP→NE rodoviário.',
     'Domínio': '{0} — ZERO em tudo',
     'Regras de Negócio': 'Nenhum rodoviário programado.',
     'Nulabilidade': 'NOT NULL', 'Origem': 'Logística'},
    {'Campo': 'EF Semana HL', 'Tipo': 'float',
     'Descrição Semântica': 'Estoque final projetado.',
     'Domínio': '(-∞, +∞)',
     'Regras de Negócio': 'EF = EI + Prod + Transf - Demanda.',
     'Nulabilidade': 'NOT NULL', 'Origem': 'Calculado'},
    {'Campo': 'Suf.f (d)', 'Tipo': 'float',
     'Descrição Semântica': 'Suficiência final em dias.',
     'Domínio': '(-∞, +∞)',
     'Regras de Negócio': 'DOI 12d.',
     'Nulabilidade': 'NOT NULL', 'Origem': 'Calculado'}
])
show_contract(contract_aba5, '📋 Data Contract — Aba 5: Cenário Divulgado')

In [ ]:
print("DEMANDA MALZBIER POR SUB-REGIÃO (HL)")
malz_dem = pd.DataFrame({
    'Sub-região': ['Mapapi', 'NE Norte', 'NE Sul', 'NO Araguaia', 'NO Centro', 'TOTAL'],
    'W0': [4672.9, 1516.7, 2484.9, 60.7, 1713.5, 10448.7],
    'W1': [4835.7, 1742.9, 2705.9, 71.4, 1894.7, 11250.6],
    'W2': [3275.4, 1313.8, 1991.6, 48.4, 1376.6, 8005.8],
    'W3': [4003.5, 1418.5, 2197.9, 50.7, 1558.4, 9229.0],
    'Total': [16787.5, 5991.9, 9380.3, 231.2, 6543.2, 38934.1]
})
display(malz_dem)

print()
print("SUFICIÊNCIA FINAL MALZBIER (dias) — DOI mín = 12d")
malz_suf = pd.DataFrame({
    'Sub-região': ['Mapapi', 'NE Norte', 'NE Sul', 'NO Araguaia', 'NO Centro'],
    'Suf W0': [2.93, 19.64, 10.98, 0.21, 1.16],
    'Suf W1': [6.76, 14.79, 7.72, 0.28, 7.51],
    'Suf W2': [4.81, 18.68, 22.48, 0.28, 30.51],
    'Suf W3': [3.37, 4.64, 20.41, 0.28, 14.34],
    'Sem <12d': [4, 3, 2, 4, 2]
})
display(malz_suf)
print()
print("Mesmo no baseline, Malzbier já tem problemas:")
print("  Mapapi: TODAS as semanas abaixo do DOI")
print("  NO Araguaia: ~0 dias (praticamente sem estoque)")

In [ ]:
print("RESUMO DEMANDA TOTAL NENO — Cenário Divulgado")
resumo = pd.DataFrame({
    'SKU': ['Goose Island', 'Malzbier', 'Patagonia', 'Colorado', 'TOTAL'],
    'Demanda Fev HL': [43328.9, 38934.1, 31879.6, 19728.6, 133871.2],
    '% Total': [32.4, 29.1, 23.8, 14.7, 100.0]
})
display(resumo)

print()
print("Goose Island — 43.329 HL (maior SKU)")
goose = pd.DataFrame({
    'Sub-região': ['Mapapi', 'NE Norte', 'NE Sul', 'NO Araguaia', 'NO Centro'],
    'Total HL': [16454.0, 7627.8, 10160.8, 717.3, 8369.0]
})
display(goose)

print()
print("Colorado — 19.729 HL (menor SKU)")
colorado = pd.DataFrame({
    'Sub-região': ['Mapapi', 'NE Norte', 'NE Sul', 'NO Araguaia', 'NO Centro'],
    'Total HL': [7358.6, 3313.2, 5390.0, 216.7, 3450.1]
})
display(colorado)

---
## 7. Data Contract — Aba 6: "Cenário com Nova Demanda"

Estrutura **idêntica** à Aba 5, mas Malzbier +30%.
Demais SKUs inalterados. Produção/transferências NÃO ajustadas → GAP.

In [ ]:
print("MALZBIER: Comparativo Divulgado vs Nova Demanda (+30%)")
comp = pd.DataFrame({
    'Sub-região': ['Mapapi', 'NE Norte', 'NE Sul', 'NO Araguaia', 'NO Centro', 'TOTAL'],
    'Div Total': [16787.5, 5991.9, 9380.3, 231.2, 6543.2, 38934.1],
    'Nova Total': [21823.8, 7789.5, 12194.3, 300.6, 8506.1, 50614.3],
    'Gap HL': [5036.3, 1797.6, 2814.0, 69.4, 1962.9, 11680.2]
})
display(comp)

print()
print("SUFICIÊNCIA COMPARATIVA (dias)")
suf_comp = pd.DataFrame({
    'Sub-região': ['Mapapi', 'NE Norte', 'NE Sul', 'NO Araguaia', 'NO Centro'],
    'Div W0': [2.93, 19.64, 10.98, 0.21, 1.16],
    'Nova W0': [0.92, 13.90, 7.18, -1.18, -0.36],
    'Div W1': [6.76, 14.79, 7.72, 0.28, 7.51],
    'Nova W1': [1.18, 7.94, 2.33, -3.78, 1.89],
    'Div W2': [4.81, 18.68, 22.48, 0.28, 30.51],
    'Nova W2': [-0.73, 9.90, 12.77, -4.93, 18.66],
    'Div W3': [3.37, 4.64, 20.41, 0.28, 14.34],
    'Nova W3': [-3.21, -2.28, 9.79, -6.32, 4.84]
})
display(suf_comp)
print()
print("RUPTURAS na Nova Demanda:")
print("  NO Araguaia: W0(-1.18), W1(-3.78), W2(-4.93), W3(-6.32) — 4 sem!")
print("  NO Centro: W0(-0.36) | Mapapi: W2(-0.73), W3(-3.21)")
print("  NE Norte: W3(-2.28)")
print(f"  GAP TOTAL: +{11680.2:,.1f} HL | Ociosa NE: ~{9360:,} HL")
print(f"  Déficit: ~{11680-9360:,} HL → transferência SP→NE obrigatória")

---
## 8. Relacionamento entre Abas

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(14, 7))
ax.set_xlim(0, 14)
ax.set_ylim(0, 7)
ax.axis('off')
fig.patch.set_facecolor('#FAFAFA')
ax.set_facecolor('#FAFAFA')

# Cores dos blocos
c_input  = '#3498DB'  # azul
c_core   = '#2ECC71'  # verde
c_shock  = '#E74C3C'  # vermelho
c_cost   = '#F39C12'  # ambar

def draw_box(ax, x, y, w, h, text, color, fontsize=9, bold=False):
    box = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.15",
                         facecolor=color, edgecolor='white', linewidth=2, alpha=0.92)
    ax.add_patch(box)
    weight = 'bold' if bold else 'normal'
    ax.text(x + w/2, y + h/2, text, ha='center', va='center',
            fontsize=fontsize, color='white', fontweight=weight,
            linespacing=1.4)

def draw_arrow(ax, x1, y1, x2, y2, label='', color='#7F8C8D'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=2.0,
                                connectionstyle='arc3,rad=0.0'))
    if label:
        mx, my = (x1+x2)/2, (y1+y2)/2 + 0.2
        ax.text(mx, my, label, ha='center', va='bottom', fontsize=7.5,
                color='#555555', fontstyle='italic',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='none', alpha=0.8))

# Blocos INPUT (esquerda)
draw_box(ax, 0.3, 5.0, 3.5, 1.2, 'Aba 1\nCenário Atual BR\n(macro mensal)', c_input, 9)
draw_box(ax, 0.3, 3.2, 3.5, 1.2, 'Aba 3\nProdução PCP\n(AQ541 + PE541)', c_input, 9)
draw_box(ax, 0.3, 1.4, 3.5, 1.2, 'Aba 4\nTransf. Programadas\n(só Goose)', c_input, 9)

# Bloco CORE (centro)
draw_box(ax, 5.2, 3.5, 3.8, 2.5, 'Aba 5\nCenário Divulgado\n(WSNP Baseline)\n4 SKUs × 5 sub-regiões', c_core, 10, bold=True)

# Bloco SHOCK (direita)
draw_box(ax, 10.2, 3.5, 3.5, 2.5, 'Aba 6\nNova Demanda\nMalzbier +30%\nGAP: +11.680 HL', c_shock, 10, bold=True)

# Bloco CUSTOS (abaixo)
draw_box(ax, 5.2, 0.5, 3.8, 1.5, 'Aba 2\nCustos de Transferência\nMACOs + Prod + Logística', c_cost, 9)

# Setas INPUT → CORE
draw_arrow(ax, 3.8, 5.6, 5.2, 5.2, 'NENO = Σ sub-regiões')
draw_arrow(ax, 3.8, 3.8, 5.2, 4.5, 'vol semanal = WSNP')
draw_arrow(ax, 3.8, 2.0, 5.2, 3.8, 'Goose = Transf.Ext(Cabo)')

# Seta CORE → SHOCK
draw_arrow(ax, 9.0, 4.75, 10.2, 4.75, 'Malzbier × 1.30')

# Seta CUSTOS → SHOCK
draw_arrow(ax, 9.0, 1.25, 11.95, 3.5, 'custo por cenário')

# Legenda
legend_items = [
    mpatches.Patch(color=c_input, label='Dados de Entrada'),
    mpatches.Patch(color=c_core, label='Baseline (WSNP)'),
    mpatches.Patch(color=c_shock, label='Cenário +30%'),
    mpatches.Patch(color=c_cost, label='Custos & Margens'),
]
ax.legend(handles=legend_items, loc='upper right', fontsize=9, framealpha=0.9,
          edgecolor='#CCCCCC', fancybox=True)

ax.set_title('Relacionamento entre as 6 Abas da Planilha', fontsize=14,
             fontweight='bold', color='#2C3E50', pad=15)

plt.tight_layout()
plt.show()

In [ ]:
checks = [
    ('Produção PCP vs WSNP', f'Malzbier NE: AQ 16.560 + PE 29.160 = 45.720 HL\nDemanda divulgada: 38.934 HL', True, 'Produção > Demanda no baseline'),
    ('Transferências', 'Goose: 54.507 HL programados\nMalzbier: 0 HL | Colorado: 0 HL', True, 'Malzbier depende 100% produção local'),
    ('Gap Malzbier +30%', f'Nova demanda: 50.614 HL\nGap incremental: +11.680 HL', True, '+30% consistente com dados'),
    ('Capacidade vs Necessidade', f'Ociosa NE: 9.360 HL\nGap: 11.680 HL → Déficit: 2.320 HL', False, 'Capacidade insuficiente!'),
    ('Impacto Financeiro', f'MACO perdido: R$ 3.328.800\nCusto prod local: R$ 1.740.320\nCusto prod+cabo: R$ 2.727.744', True, 'Valores consistentes'),
]

fig, ax = plt.subplots(figsize=(13, 6))
ax.axis('off')
fig.patch.set_facecolor('#FAFAFA')

ax.set_title('Verificações de Consistência entre Abas', fontsize=14,
             fontweight='bold', color='#2C3E50', y=1.02)

y_start = 0.92
row_h = 0.17

for i, (titulo, detalhe, ok, nota) in enumerate(checks):
    y = y_start - i * row_h
    cor_bg = '#E8F8F5' if ok else '#FDEDEC'
    cor_borda = '#27AE60' if ok else '#E74C3C'
    icone = 'OK' if ok else '!!'

    rect = FancyBboxPatch((0.02, y - 0.06), 0.96, row_h - 0.02,
                          boxstyle="round,pad=0.01",
                          facecolor=cor_bg, edgecolor=cor_borda,
                          linewidth=1.5, transform=ax.transAxes)
    ax.add_patch(rect)

    ax.text(0.05, y + 0.04, f'{icone}  {titulo}', transform=ax.transAxes,
            fontsize=11, fontweight='bold', color='#2C3E50', va='center')
    ax.text(0.05, y - 0.02, detalhe, transform=ax.transAxes,
            fontsize=8.5, color='#555555', va='top', linespacing=1.3)
    ax.text(0.95, y + 0.04, nota, transform=ax.transAxes,
            fontsize=8.5, color=cor_borda, va='center', ha='right', fontstyle='italic')

plt.tight_layout()
plt.show()

---
## 9. Dicionário de Variáveis Derivadas

Variáveis **calculadas** essenciais para a análise, que não existem diretamente na planilha.

In [ ]:
variaveis = pd.DataFrame([
    {'Variável': 'Gap Demanda (HL)', 'Fórmula': 'Demanda_Nova - Demanda_Divulgada',
     'Granularidade': 'SKU × Sub-região × Semana',
     'Interpretação': 'Volume incremental do choque +30%.',
     'Exemplo': 'Mapapi W0: 6074.8 - 4672.9 = 1401.9 HL'},
    {'Variável': 'Utilização Planta (%)', 'Fórmula': 'Σ(Prod SKUs) / Cap Semanal × 100',
     'Granularidade': 'Planta × Semana',
     'Interpretação': '>90% = gargalo.',
     'Exemplo': 'PE541 W0: 27000/27000 = 100%'},
    {'Variável': 'Capacidade Ociosa (HL)', 'Fórmula': 'Cap Semanal - Σ(Prod SKUs)',
     'Granularidade': 'Planta × Semana',
     'Interpretação': 'Espaço livre para realocar.',
     'Exemplo': 'PE541 W1: 27000-19800 = 7200 HL'},
    {'Variável': 'Margem Líquida (R$/HL)', 'Fórmula': 'MACO - CustoProd - CustoTransf',
     'Granularidade': 'SKU × Modal × Destino',
     'Interpretação': 'Negativa = inviável.',
     'Exemplo': 'Malzbier Rodo→FM: 285-149-152.53 = -16.53'},
    {'Variável': 'DOI Delta (dias)', 'Fórmula': 'Suf.f(Nova) - Suf.f(Div)',
     'Granularidade': 'Sub-região × Semana',
     'Interpretação': 'Deterioração da suficiência. Sempre ≤ 0.',
     'Exemplo': 'Mapapi W0: 0.92 - 2.93 = -2.01 dias'},
    {'Variável': 'Custo Oportunidade (R$)', 'Fórmula': 'Gap_não_atendido × MACO',
     'Granularidade': 'SKU × Sub-região',
     'Interpretação': 'Receita perdida se não cobrir.',
     'Exemplo': 'Total: 11680 × 285 = R$ 3.328.800'},
    {'Variável': 'BIAS Ajustado (HL)', 'Fórmula': 'Demanda × (1 - 0.09)',
     'Granularidade': 'Sub-região × Semana',
     'Interpretação': 'Demanda corrigida pelo sobre-forecast.',
     'Exemplo': '10000 HL → 9100 HL'},
    {'Variável': 'Vol Líquido Rodo (HL)', 'Fórmula': 'Vol_bruto × (1 - 0.05)',
     'Granularidade': 'Transferência',
     'Interpretação': 'Volume efetivo após 5% avaria.',
     'Exemplo': '1000 HL → 950 HL'}
])
display(HTML("<h4>Variáveis Derivadas</h4>"))
display(variaveis)

---
## 10. Premissas e Restrições Operacionais

In [ ]:
premissas = pd.DataFrame([
    {'Cat': 'Demanda', 'Premissa': 'Malzbier +30% sobre cenário divulgado',
     'Fonte': 'Enunciado', 'Impacto': 'Gap +11.680 HL'},
    {'Cat': 'Demanda', 'Premissa': 'BIAS de +9% no forecast dos GEOs',
     'Fonte': 'Onboarding', 'Impacto': 'Demanda real pode ser 9% menor'},
    {'Cat': 'Demanda', 'Premissa': 'Demais SKUs inalterados',
     'Fonte': 'Enunciado', 'Impacto': 'Goose, Patagonia, Colorado = mesma demanda'},
    {'Cat': 'Produção', 'Premissa': 'Linha L541 compartilhada entre SKUs',
     'Fonte': 'Aba 3', 'Impacto': 'Não rodam 2 SKUs simultâneos'},
    {'Cat': 'Produção', 'Premissa': 'Cap AQ541: 12.600/sem, PE541: 27.000/sem',
     'Fonte': 'Aba 3', 'Impacto': 'Limite físico'},
    {'Cat': 'Logística', 'Premissa': 'Cabotagem: 25 dias lead time',
     'Fonte': 'Onboarding', 'Impacto': 'Não chega a tempo se iniciar em Fev'},
    {'Cat': 'Logística', 'Premissa': 'Rodoviário: 6 dias, +60% custo, 5% avaria',
     'Fonte': 'Onboarding', 'Impacto': 'Alternativa rápida mas cara'},
    {'Cat': 'Estoque', 'Premissa': 'DOI mínimo: 12 dias',
     'Fonte': 'Onboarding', 'Impacto': 'Abaixo = risco de ruptura'},
    {'Cat': 'Financeiro', 'Premissa': 'MACOs e custos fixos (sem economia de escala)',
     'Fonte': 'Aba 2', 'Impacto': 'Análise marginal é linear'},
    {'Cat': 'Temporal', 'Premissa': 'Horizonte = Fev 2026 (4 semanas)',
     'Fonte': 'Enunciado', 'Impacto': 'Solução de curto prazo'}
])
display(HTML("<h4>Premissas e Restrições</h4>"))
display(premissas)

---
## 11. Matriz de Completude dos Dados

In [ ]:
campos = ['Demanda', 'Produção', 'Estoque EI', 'Estoque EF', 'Suficiência',
          'Transf.Interna', 'Transf.Ext Cabo', 'Transf.Ext Rodo', 'Trânsito',
          'Custo Transf.', 'MACO', 'Custo Produção', 'Cap. Planta']
abas = ['Aba 1\nCenário BR', 'Aba 2\nCustos', 'Aba 3\nPCP', 'Aba 4\nTransf.', 'Aba 5\nDivulgado', 'Aba 6\nNova Dem.']

matrix = np.array([
    [1,0,1,1,1, 0,0,0,0, 0,0,0,0],
    [0,0,0,0,0, 0,0,0,0, 1,1,1,0],
    [0,1,0,0,0, 0,0,0,0, 0,0,0,1],
    [0,0,0,0,0, 0,1,0,0, 0,0,0,0],
    [1,1,1,1,1, 1,1,1,1, 0,0,0,0],
    [1,1,1,1,1, 1,1,1,1, 0,0,0,0],
])

fig, ax = plt.subplots(figsize=(14, 5.5))

# Cores: verde para disponível, vermelho claro para indisponível
colors = np.where(matrix == 1, '#27AE60', '#F5B7B1')
for i in range(len(abas)):
    for j in range(len(campos)):
        ax.add_patch(plt.Rectangle((j, len(abas)-1-i), 1, 1,
                     facecolor=colors[i][j], edgecolor='white', linewidth=2))
        symbol = '\u2713' if matrix[i][j] == 1 else '\u2717'
        txt_color = 'white' if matrix[i][j] == 1 else '#C0392B'
        ax.text(j + 0.5, len(abas)-1-i + 0.5, symbol, ha='center', va='center',
                fontsize=14, fontweight='bold', color=txt_color)

ax.set_xlim(0, len(campos))
ax.set_ylim(0, len(abas))
ax.set_xticks([j + 0.5 for j in range(len(campos))])
ax.set_xticklabels(campos, rotation=45, ha='right', fontsize=9)
ax.set_yticks([i + 0.5 for i in range(len(abas))])
ax.set_yticklabels(list(reversed(abas)), fontsize=9)
ax.tick_params(length=0)

for spine in ax.spines.values():
    spine.set_visible(False)

legend_items = [
    mpatches.Patch(color='#27AE60', label='Disponível'),
    mpatches.Patch(color='#F5B7B1', label='Não disponível'),
]
ax.legend(handles=legend_items, loc='upper right', fontsize=9, bbox_to_anchor=(1.0, 1.15))

ax.set_title('Matriz de Completude: Campos × Abas', fontsize=14,
             fontweight='bold', color='#2C3E50', pad=20)

plt.tight_layout()
plt.show()

---
## 12. Qualidade dos Dados e Limitações

In [ ]:
aspectos = ['Valores Nulos', 'Consistência\nTemporal', 'Unidade de\nMedida',
            'Granularidade\nGeo', 'SKUs entre\nAbas', 'BIAS de\nDemanda', 'Dados\nFaltantes']
scores  = [1.0, 0.6, 1.0, 0.5, 0.6, 0.4, 0.2]
colors  = ['#27AE60','#F39C12','#27AE60','#E67E22','#F39C12','#E74C3C','#C0392B']
labels  = ['OK', 'Atenção', 'OK', 'Mismatch', 'Inconsist.', 'Viés +9%', 'Gap']

fig, ax = plt.subplots(figsize=(13, 4.5))

bars = ax.barh(range(len(aspectos)), scores, color=colors, height=0.6,
               edgecolor='white', linewidth=1.5)

for i, (bar, label) in enumerate(zip(bars, labels)):
    ax.text(bar.get_width() + 0.03, bar.get_y() + bar.get_height()/2,
            label, va='center', fontsize=10, fontweight='bold', color=colors[i])

ax.set_yticks(range(len(aspectos)))
ax.set_yticklabels(aspectos, fontsize=10)
ax.set_xlim(0, 1.3)
ax.set_xticks([0, 0.25, 0.5, 0.75, 1.0])
ax.set_xticklabels(['Crítico', 'Ruim', 'Médio', 'Bom', 'Excelente'], fontsize=9)
ax.axvline(x=0.5, color='#BDC3C7', linestyle='--', linewidth=1, alpha=0.7)

for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

ax.set_title('Avaliação de Qualidade dos Dados', fontsize=14,
             fontweight='bold', color='#2C3E50', pad=15)
ax.invert_yaxis()

detalhes = [
    'Nenhum campo essencial é nulo. Zeros = sem produção/transferência.',
    'Aba 1 mensal, Abas 3-6 semanais. Agregação necessária.',
    'Tudo em HL e R$. Sem conversão necessária.',
    'Aba 1: NENO agregado. Abas 5/6: 5 sub-regiões.',
    'Aba 1 consolida todos LN. Patagonia ausente Aba 2.',
    'GEOs sobre-estimam +9%. Demanda real ~ 91%.',
    'Falta: custo setup, tempo troca, estoque mín, cap armazenagem.'
]

fig.text(0.02, -0.02, '\n'.join([f'  {asp.replace(chr(10)," ")}: {det}' for asp, det in zip(aspectos, detalhes)]),
         fontsize=8, color='#7F8C8D', va='top', linespacing=1.5,
         fontfamily='monospace')

plt.tight_layout()
plt.show()

---
## 13. Mapa de Fluxo da Cadeia de Supply

In [ ]:
fig, ax = plt.subplots(figsize=(16, 9))
ax.set_xlim(0, 16)
ax.set_ylim(0, 9)
ax.axis('off')
fig.patch.set_facecolor('#FAFAFA')

# ── Título de seção ──
ax.text(8, 8.6, 'Fluxo de Supply Chain — NENO', ha='center', fontsize=16,
        fontweight='bold', color='#2C3E50')
ax.text(8, 8.2, 'Produção → Distribuição → Sub-regiões de Consumo', ha='center',
        fontsize=11, color='#7F8C8D', fontstyle='italic')

# ── COLUNA 1: ORIGENS DE PRODUÇÃO ──
# SP (Sudeste) — origem de transferências
sp_box = FancyBboxPatch((0.3, 5.5), 3.2, 2.2, boxstyle="round,pad=0.15",
                        facecolor='#8E44AD', edgecolor='#6C3483', linewidth=2)
ax.add_patch(sp_box)
ax.text(1.9, 7.1, 'SP (Sudeste)', ha='center', fontsize=12, fontweight='bold', color='white')
ax.text(1.9, 6.6, 'BR16 Jacareí', ha='center', fontsize=9, color='#D7BDE2')
ax.text(1.9, 6.3, 'BR23 Jaguariúna', ha='center', fontsize=9, color='#D7BDE2')
ax.text(1.9, 5.85, 'Goose: SIM  |  Malzbier: NAO  |  Colorado: NAO', ha='center', fontsize=8, color='#F9E79F')

# AQ541
aq_box = FancyBboxPatch((0.3, 3.2), 3.2, 1.6, boxstyle="round,pad=0.15",
                         facecolor='#2980B9', edgecolor='#1F618D', linewidth=2)
ax.add_patch(aq_box)
ax.text(1.9, 4.35, 'AQ541 Aquiraz', ha='center', fontsize=11, fontweight='bold', color='white')
ax.text(1.9, 3.9, 'CE | 12.600 HL/sem', ha='center', fontsize=9.5, color='#AED6F1')
ax.text(1.9, 3.5, 'Utilização: 95.7%', ha='center', fontsize=9, color='#F9E79F')

# PE541
pe_box = FancyBboxPatch((0.3, 1.0), 3.2, 1.6, boxstyle="round,pad=0.15",
                         facecolor='#2980B9', edgecolor='#1F618D', linewidth=2)
ax.add_patch(pe_box)
ax.text(1.9, 2.15, 'PE541 Pernambuco', ha='center', fontsize=11, fontweight='bold', color='white')
ax.text(1.9, 1.7, 'PE | 27.000 HL/sem', ha='center', fontsize=9.5, color='#AED6F1')
ax.text(1.9, 1.3, 'Utilização: 93.3%', ha='center', fontsize=9, color='#F9E79F')

# ── COLUNA 2: CDs DE DISTRIBUIÇÃO ──
cds = [('Camaçari BA', 7.0), ('Fonte Mata PB', 5.3), ('CDR Bahia', 3.6)]
for nome, y in cds:
    cd_box = FancyBboxPatch((6.0, y - 0.45), 2.8, 0.9, boxstyle="round,pad=0.12",
                             facecolor='#F39C12', edgecolor='#D68910', linewidth=2)
    ax.add_patch(cd_box)
    ax.text(7.4, y, nome, ha='center', va='center', fontsize=10, fontweight='bold', color='white')

# ── COLUNA 3: SUB-REGIÕES (CONSUMO) ──
subregs = [('Mapapi', 7.8, '43% gap'), ('NE Norte', 6.6, '15% gap'),
           ('NE Sul', 5.4, '24% gap'), ('NO Araguaia', 4.2, '1% gap'),
           ('NO Centro', 3.0, '17% gap')]
for nome, y, pct in subregs:
    sr_box = FancyBboxPatch((11.0, y - 0.35), 2.8, 0.7, boxstyle="round,pad=0.1",
                             facecolor='#27AE60', edgecolor='#1E8449', linewidth=1.5)
    ax.add_patch(sr_box)
    ax.text(12.4, y, f'{nome}', ha='center', va='center', fontsize=10, fontweight='bold', color='white')
    ax.text(14.0, y, pct, ha='left', va='center', fontsize=8.5, color='#7F8C8D', fontstyle='italic')

# ── SETAS: SP → CDs ──
arrow_kw_cabo = dict(arrowstyle='->', color='#8E44AD', lw=2.5, connectionstyle='arc3,rad=0.15')
arrow_kw_rodo = dict(arrowstyle='->', color='#E74C3C', lw=2.0, linestyle='dashed', connectionstyle='arc3,rad=-0.1')

ax.annotate('', xy=(6.0, 7.0), xytext=(3.5, 6.8), arrowprops=arrow_kw_cabo)
ax.text(4.75, 7.35, 'Cabo 25d', fontsize=8.5, color='#8E44AD', fontweight='bold', ha='center',
        bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='#8E44AD', alpha=0.9))

ax.annotate('', xy=(6.0, 5.3), xytext=(3.5, 6.3), arrowprops=arrow_kw_cabo)

ax.annotate('', xy=(6.0, 3.6), xytext=(3.5, 6.0), arrowprops=arrow_kw_cabo)

# Rodo (alternativa emergencial)
ax.annotate('', xy=(6.0, 6.7), xytext=(3.5, 6.5), arrowprops=arrow_kw_rodo)
ax.text(4.75, 6.25, 'Rodo 6d', fontsize=8.5, color='#E74C3C', fontweight='bold', ha='center',
        bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='#E74C3C', alpha=0.9))

# ── SETAS: Plantas NE → Sub-regiões (direto) ──
arrow_kw_local = dict(arrowstyle='->', color='#2980B9', lw=2.0, connectionstyle='arc3,rad=0.0')
for _, y, _ in subregs:
    ax.annotate('', xy=(11.0, y), xytext=(3.5, 4.0), arrowprops=dict(
        arrowstyle='->', color='#2980B9', lw=1.2, alpha=0.4, connectionstyle='arc3,rad=0.1'))
    ax.annotate('', xy=(11.0, y), xytext=(3.5, 1.8), arrowprops=dict(
        arrowstyle='->', color='#2980B9', lw=1.2, alpha=0.4, connectionstyle='arc3,rad=-0.1'))

ax.text(7.0, 2.0, 'Produção Local\n(direto)', ha='center', fontsize=9.5, color='#2980B9',
        fontweight='bold', bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='#2980B9', alpha=0.9))

# ── SETAS: CDs → Sub-regiões ──
for _, cd_y in cds:
    for _, sr_y, _ in subregs:
        ax.annotate('', xy=(11.0, sr_y), xytext=(8.8, cd_y),
                    arrowprops=dict(arrowstyle='->', color='#F39C12', lw=0.8, alpha=0.3))

# ── Legenda ──
legend_items = [
    mpatches.Patch(color='#8E44AD', label='SP — Origem transferências'),
    mpatches.Patch(color='#2980B9', label='Plantas NE — Produção local'),
    mpatches.Patch(color='#F39C12', label='CDs — Distribuição'),
    mpatches.Patch(color='#27AE60', label='Sub-regiões — Consumo'),
]
ax.legend(handles=legend_items, loc='lower center', ncol=4, fontsize=9,
          framealpha=0.9, edgecolor='#CCCCCC', bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
plt.show()

---
## 14. Resumo Executivo do Data Contract

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
fig.patch.set_facecolor('#FAFAFA')
fig.suptitle('Resumo Executivo — Data Contract', fontsize=16, fontweight='bold',
             color='#2C3E50', y=1.02)

cards = [
    ('DEMANDA', '#3498DB',
     ['Malzbier Divulgada:', '38.934 HL', '', 'Malzbier Nova (+30%):', '50.614 HL', '', 'Gap:', '+11.680 HL']),
    ('OFERTA NE', '#2ECC71',
     ['Cap total NE/mês:', '158.400 HL', '', 'Em uso:', '149.040 HL (94.1%)', '', 'Ociosa:', '9.360 HL']),
    ('GAP vs CAPACIDADE', '#E74C3C',
     ['Gap:', '+11.680 HL', '', 'Ociosa:', '9.360 HL', '', 'Déficit (SP):', '2.320 HL']),
    ('CUSTOS MALZBIER', '#F39C12',
     ['MACO: R$285/HL', 'Prod local: R$149 → mg R$136', '',
      'Cabo(Cam): R$234 → mg R$51', 'Rodo(Cam): R$284 → mg R$1', 'Rodo(FM): R$302 → mg -R$17']),
    ('TIMING', '#9B59B6',
     ['Cabo 25d:', 'NÃO resolve em Fev', '',
      'Rodo 6d:', 'ÚNICA opção emergencial', '', '+60% custo + 5% avaria']),
    ('IMPACTO FINANCEIRO', '#1ABC9C',
     ['MACO perdido:', 'R$ 3.328.800', '',
      'Melhor cenário (mix):', 'R$ 1.936.305', '', 'Margem preservada:', 'R$ 1.392.495']),
]

for idx, (titulo, cor, linhas) in enumerate(cards):
    ax = axes[idx // 3][idx % 3]
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')

    # Card background
    card = FancyBboxPatch((0.02, 0.02), 0.96, 0.96, boxstyle="round,pad=0.03",
                          facecolor='white', edgecolor=cor, linewidth=2.5)
    ax.add_patch(card)

    # Header bar
    header = FancyBboxPatch((0.02, 0.78), 0.96, 0.20, boxstyle="round,pad=0.03",
                            facecolor=cor, edgecolor=cor, linewidth=0)
    ax.add_patch(header)
    ax.text(0.5, 0.88, titulo, ha='center', va='center', fontsize=11,
            fontweight='bold', color='white')

    # Content
    y_pos = 0.68
    for linha in linhas:
        if linha == '':
            y_pos -= 0.03
            continue
        is_value = any(c.isdigit() for c in linha) or 'R$' in linha or 'mg' in linha
        ax.text(0.1, y_pos, linha, ha='left', va='center',
                fontsize=9 if is_value else 8.5,
                fontweight='bold' if is_value else 'normal',
                color='#2C3E50' if is_value else '#7F8C8D')
        y_pos -= 0.09

plt.tight_layout()
plt.show()

---
## 15. Insights do Data Contract — Síntese Executiva

> Esta síntese consolida as regras de negócio, restrições estruturais e variáveis críticas identificadas no mapeamento das 6 abas do Excel. Os valores são exatamente os documentados no Data Contract.

---

### 1. Escopo: 6 Fontes de Dados, 48 Variáveis Mapeadas

O Data Contract cobre todas as abas do Excel do case:

| Aba | Conteúdo | Campos mapeados |
|---|---|---|
| **Cenário Atual BR** | Demanda e DOI por região e SKU | 8 |
| **Custos de Transferência** | Fretes por modal, rota e SKU | 8 |
| **Produção PCP** | Capacidade e utilização por planta | 8 |
| **Transferências Programadas** | Fluxos inter-regionais planejados | 8 |
| **Cenário Divulgado** | WSNP baseline por sub-região | 8 |
| **Cenário com Nova Demanda** | WSNP com +30% Malzbier | 8 |

Cada campo documentado com: Nome, Tipo, Formato, Descrição Semântica, Domínio, Regras de Negócio, Nulabilidade e Origem.

---

### 2. A Regra de Negócio Mais Crítica: DOI Mínimo de 12 Dias

O **DOI (Days of Inventory) mínimo de 12 dias** é a principal restrição operacional do sistema. Abaixo desse limiar, há risco formal de ruptura de estoque. Essa regra determina:

- Quais sub-regiões exigem ação imediata (DOI < 12d)
- O volume de transferência mínimo necessário
- A priorização entre sub-regiões em caso de capacidade insuficiente

Com o +30% de Malzbier, o DOI de **3 das 5 sub-regiões cai abaixo de zero** na W3 — o que significa que o estoque existente não cobre nem o mês inteiro sem reabastecimento.

---

### 3. O BIAS de +9% é um Dado, Não uma Estimativa

Os dados da Aba 1 revelam que os GEOs (gestores comerciais regionais) historicamente **sobre-preveem a demanda em +9%** (BIAS positivo). Isso impacta diretamente a análise:

| Cenário | Volume Malzbier NENO |
|---|---|
| Demanda divulgada (com BIAS) | **38.934 HL** |
| Gap com +30% (com BIAS) | **11.680 HL** |
| Gap corrigido (descontando BIAS) | **~10.630 HL** |

A decisão de incorporar ou desconsiderar o BIAS é **julgamento gerencial** — os dados documentam o fato, mas não prescrevem a resposta.

---

### 4. MACO por SKU Define a Priorização Econômica

Os dados de **MACO (Margem de Contribuição)** por SKU determinam qual produto deve ser priorizado em situações de restrição de capacidade:

| SKU | MACO (R\$/HL) | Margem Bruta | Posição |
|---|---|---|---|
| **Goose Island** | R\$350/HL | — | Maior valor unitário |
| **Colorado** | R\$300/HL | — | Segundo maior |
| **Malzbier** | R\$285/HL | R\$136/HL (47,7%) | Objeto do case |

Isso significa que, em caso de conflito por capacidade, **Goose e Colorado têm prioridade econômica sobre o Malzbier** — o que limita ainda mais a capacidade disponível para realocar.

---

### 5. Duas Restrições Estruturais que Limitam a Solução

O mapeamento identificou duas restrições que não aparecem nas planilhas brutas, mas estão documentadas no case:

**Restrição 1 — Goose Island em PE541:**
Goose Island não pode ter sua produção aumentada na planta de Pernambuco por limitação de elaboração de líquido. Qualquer realocação de capacidade em PE541 deve vir de outros SKUs (Brahma Chopp Zero, Budweiser Zero, Colorado).

**Restrição 2 — NO Araguaia (100% Revendedores):**
A sub-região de NO Araguaia opera integralmente com revendedores autônomos, que gerenciam seu próprio estoque e fazem **retirada direta em Uberlândia**. O DOI negativo registrado para essa sub-região é estoque dos revendedores — não da Ambev. Isso reduz a urgência de atender NO Araguaia em relação às demais sub-regiões.

---

### 6. A Estrutura Logística: Dois CDRs, Duas Rotas de Entrada

Os dados de transferência revelam a arquitetura da malha de distribuição no NENO:

| CDR | Sub-regiões atendidas | Rota cabotagem |
|---|---|---|
| **CDR João Pessoa** | Mapapi, NO Centro, NE Norte | Jaguariúna → Fonte Mata (R\$95,3/HL) |
| **CDR Camaçari** | NE Sul | Jaguariúna → Camaçari (R\$84,6/HL) |

Lead time de cabotagem: **25 dias**. Para fevereiro (28 dias), o prazo de acionamento já estava vencendo no início do mês.

---

### Resumo das Variáveis-Chave do Data Contract

| Variável | Valor Documentado |
|---|---|
| Abas mapeadas | **6** |
| Atributos por campo | **8** |
| DOI mínimo operacional | **12 dias** |
| BIAS histórico GEOs | **+9% (sobre-previsão)** |
| MACO Malzbier | **R\$285/HL** |
| Margem bruta Malzbier | **R\$136/HL (47,7%)** |
| Capacidade AQ541 | **12.600 HL/semana** |
| Capacidade PE541 | **27.000 HL/semana** |
| Lead time cabotagem | **25 dias** |
| Sub-regiões NENO | **5 (Mapapi, NE Norte, NE Sul, NO Centro, NO Araguaia)** |

---

> **Próximo passo — Análise Univariada:** examinar cada variável individualmente (distribuição, tendência central, dispersão, outliers) para construir fundamento sólido antes de cruzar dados na análise bivariada.

---
---

# PARTE 2 — Análise Univariada

> **Objetivo:** Examinar **cada variável individualmente** — distribuição, tendência central, dispersão, outliers — para construir fundamento sólido antes de cruzar dados na análise bivariada.

**Escopo:** Todas as 6 abas do Excel + informações contextuais dos PDFs do case.

| Conceito | Significado |
|---|---|
| **Média** | Tendência central — valor "típico" |
| **Mediana** | Valor central (robusto a outliers) |
| **Desvio Padrão** | Dispersão — quanto os valores variam |
| **CV (Coeficiente de Variação)** | Dispersão relativa — CV > 30% = alta variabilidade |
| **Min / Max** | Extremos — indicam amplitude |
| **DOI (Days of Inventory)** | Dias de estoque — mínimo operacional: 12 dias |

---
## 16. Análise Univariada — Aba 1: Cenário Atual BR (Visão Nacional)

> **O que é esta aba?** Panorama mensal (Janeiro/Fevereiro) de **demanda, produção e estoque** para todas as 7 regiões da Ambev no Brasil. Permite entender onde está o volume e onde há folga ou pressão logística.

**Variáveis analisadas:** Demanda (HL), Produção Real (HL), WSNP (HL), Suficiência final (dias), Transferência de malha (HL), Estoque Final (HL)

## 🎯 Síntese — Data Contract

Dos 6 contratos mapeados, **3 achados são críticos** e guiam toda a análise:

1. **Demanda NENO +30%** (Malzbier): Nova demanda de 11.680 HL/semana vs capacidade atual de ~9.360 HL ociosos — gap estrutural de ≈1.100–1.500 HL/semana
2. **Linha L541 compartilhada**: PE541 (Recife) e AQ541 (Fortaleza) operam >95% de utilização — sem folga para absorver pico sem realocação de SKUs
3. **BIAS GEOs de +9%**: Demanda divulgada historicamente superestimada — análise de cenários deve contemplar sensibilidade de −9%

Esses três pontos fundamentam a árvore de hipóteses e a análise bivariada nas seções seguintes.

In [ ]:

import subprocess, sys as _sys
%pip install openpyxl --quiet

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import matplotlib.patheffects as pe
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from IPython.display import display, HTML

# ── Configuração visual global ──────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#FAFAFA',
    'axes.facecolor': '#FAFAFA',
    'font.family': 'sans-serif',
    'font.size': 11,
    'figure.dpi': 100
})

CORES = {
    'azul_escuro': '#1B2A4A',
    'azul_medio': '#2D5F8A',
    'azul_claro': '#5BA4CF',
    'ambar': '#F5A623',
    'vermelho': '#E74C3C',
    'verde': '#27AE60',
    'cinza_claro': '#ECF0F1',
    'cinza_medio': '#95A5A6',
    'branco': '#FFFFFF',
    'bg': '#FAFAFA'
}

# ── Constantes ───────────────────────────────────────────────────────────────
cap_aq = 12600   # HL/semana AQ541 (Aquiraz/CE)
cap_pe = 27000   # HL/semana PE541 (Nassau/PE)
DOI_MIN = 12     # dias de inventário mínimo NENO
semanas = ['W1 (02/02)', 'W2 (09/02)', 'W3 (16/02)', 'W4 (23/02)']

# ── Localização do Excel ─────────────────────────────────────────────────────
BASE_DIR = Path('.').resolve()
EXCEL_FILE = None
for p in [BASE_DIR, BASE_DIR.parent]:
    candidates = [f for f in p.glob('*Sem*repostas*.xlsx') if not f.name.startswith('~$')]
    if candidates:
        EXCEL_FILE = candidates[0]
        break
if EXCEL_FILE is None:
    for p in [BASE_DIR, BASE_DIR.parent]:
        all_xlsx = list(p.glob('*Long*Neck*.xlsx')) + list(p.glob('*longneck*.xlsx'))
        candidates = [f for f in all_xlsx if not f.name.startswith('~$')]
        if candidates:
            EXCEL_FILE = candidates[0]
            break

if EXCEL_FILE is None:
    raise FileNotFoundError("Excel não encontrado! Coloque o arquivo na mesma pasta do notebook.")

print(f"✅ Excel: {EXCEL_FILE.name}")
xls = pd.ExcelFile(EXCEL_FILE)
print(f"Abas: {xls.sheet_names}")

# ── Aba 1: Cenário Atual BR ──────────────────────────────────────────────────
raw1 = pd.read_excel(xls, 'Cenário atual BR', header=None)
regioes_br = ['MG', 'SP', 'NENO', 'CO', 'RJ', 'SUL']
jan_data, fev_data = [], []
for i, reg in enumerate(regioes_br):
    row = 5 + i
    jan_data.append({
        'Regiao': reg,
        'Demanda_Jan': float(raw1.iloc[row, 3]) if pd.notna(raw1.iloc[row, 3]) else 0,
        'PROD_Real_Jan': float(raw1.iloc[row, 5]) if pd.notna(raw1.iloc[row, 5]) else 0,
        'WSNP_Jan': float(raw1.iloc[row, 6]) if pd.notna(raw1.iloc[row, 6]) else 0,
        'Prog_1W_Jan': float(raw1.iloc[row, 7]) if pd.notna(raw1.iloc[row, 7]) else 0,
    })
    fev_data.append({
        'Regiao': reg,
        'Demanda_Fev': float(raw1.iloc[row, 17]) if pd.notna(raw1.iloc[row, 17]) else 0,
        'WSNP_Fev': float(raw1.iloc[row, 19]) if pd.notna(raw1.iloc[row, 19]) else 0,
        'Transf_Malha_Fev': float(raw1.iloc[row, 21]) if pd.notna(raw1.iloc[row, 21]) else 0,
        'EFM_Fev': float(raw1.iloc[row, 22]) if pd.notna(raw1.iloc[row, 22]) else 0,
        'Suf_Fev_dias': float(raw1.iloc[row, 23]) if pd.notna(raw1.iloc[row, 23]) else 0,
    })
df_jan = pd.DataFrame(jan_data)
df_fev = pd.DataFrame(fev_data)
df_cenario_br = pd.merge(df_jan, df_fev, on='Regiao')
print(f"✅ Aba 1 (Cenário Atual BR): {len(df_cenario_br)} regiões")

# ── Aba 2: Custos de Transferência ───────────────────────────────────────────
raw2 = pd.read_excel(xls, 'Custos de transferência', header=None)
custos_transf = []
for row in range(3, 9):
    custos_transf.append({
        'SKU': str(raw2.iloc[row, 1]).strip(),
        'Origem': str(raw2.iloc[row, 2]).strip(),
        'Destino': str(raw2.iloc[row, 3]).strip(),
        'Custo_R_HL': float(raw2.iloc[row, 4])
    })
df_custos_transf = pd.DataFrame(custos_transf)

maco_data = []
for row in [13, 14, 15]:
    maco_data.append({'SKU': str(raw2.iloc[row, 1]).strip(), 'MACO_R_HL': float(raw2.iloc[row, 4])})
df_maco = pd.DataFrame(maco_data)

cprod_data = []
for row in [20, 21, 22]:
    cprod_data.append({'SKU': str(raw2.iloc[row, 1]).strip(), 'Custo_Prod_R_HL': float(raw2.iloc[row, 4])})
df_cprod = pd.DataFrame(cprod_data)

df_sku_economics = pd.merge(df_maco, df_cprod, on='SKU')
df_sku_economics['Margem_Bruta'] = df_sku_economics['MACO_R_HL'] - df_sku_economics['Custo_Prod_R_HL']
df_sku_economics['Margem_Pct'] = (df_sku_economics['Margem_Bruta'] / df_sku_economics['MACO_R_HL'] * 100).round(1)
print(f"✅ Aba 2 (Custos): {len(df_custos_transf)} rotas + {len(df_sku_economics)} SKUs")

# ── Aba 3: Produção PCP ──────────────────────────────────────────────────────
raw3 = pd.read_excel(xls, 'Produção PCP', header=None)
aq_skus = ['Malzbier', 'Patagonia', 'Colorado']
aq_rows = [2, 3, 4]
prod_pcp = []
for sku, row in zip(aq_skus, aq_rows):
    for w, col in enumerate([6, 7, 8, 9]):
        val = float(raw3.iloc[row, col]) if pd.notna(raw3.iloc[row, col]) else 0
        prod_pcp.append({'Planta': 'AQ541 (CE)', 'SKU': sku, 'Semana': semanas[w], 'Semana_Num': w+1, 'Volume_HL': val})
pe_skus = ['Brahma Chopp Zero', 'Goose Island', 'Malzbier', 'Colorado', 'Skol Beats', 'Budweiser Zero']
pe_rows = [9, 10, 11, 12, 13, 14]
for sku, row in zip(pe_skus, pe_rows):
    for w, col in enumerate([6, 7, 8, 9]):
        val = float(raw3.iloc[row, col]) if pd.notna(raw3.iloc[row, col]) else 0
        prod_pcp.append({'Planta': 'PE541 (PE)', 'SKU': sku, 'Semana': semanas[w], 'Semana_Num': w+1, 'Volume_HL': val})
df_pcp = pd.DataFrame(prod_pcp)
print(f"✅ Aba 3 (Produção PCP): {len(df_pcp)} registros")

# ── Aba 4: Transferências Programadas ────────────────────────────────────────
raw4 = pd.read_excel(xls, 'Transferências Programadas', header=None)
transf_prog = []
for row in [3, 4, 5]:
    dest = str(raw4.iloc[row, 3]).strip()
    for w, col in enumerate([7, 8, 9, 10]):
        val = float(raw4.iloc[row, col]) if pd.notna(raw4.iloc[row, col]) else 0
        transf_prog.append({
            'Destino': dest, 'SKU': 'Goose Island', 'Modal': 'Cabotagem',
            'Semana': semanas[w], 'Semana_Num': w+1, 'Volume_HL': val
        })
df_transf = pd.DataFrame(transf_prog)
print(f"✅ Aba 4 (Transferências): {len(df_transf)} registros")

# ── Abas 5 e 6: Cenário Divulgado e Nova Demanda ─────────────────────────────
def parse_cenario_neno(sheet_name):
    raw = pd.read_excel(xls, sheet_name, header=None)
    skus_info = [
        ('Patagonia', 4, 9),
        ('Goose Island', 12, 17),
        ('Malzbier', 20, 25),
        ('Colorado', 28, 33),
    ]
    sub_regioes = ['Mapapi', 'NE Norte', 'NE Sul', 'NO Araguaia', 'NO Centro']
    week_cols = [
        {'dem': 3, 'wsnp': 5, 'ei': 7, 'suf_ini': 8, 'tr_int': 9, 'ef': 13, 'suf_f': 14},
        {'dem': 16, 'wsnp': 18, 'ei': None, 'suf_ini': None, 'tr_int': 20, 'ef': 24, 'suf_f': 25},
        {'dem': 27, 'wsnp': 29, 'ei': None, 'suf_ini': None, 'tr_int': 31, 'ef': 35, 'suf_f': 36},
        {'dem': 38, 'wsnp': 40, 'ei': None, 'suf_ini': None, 'tr_int': 42, 'ef': 46, 'suf_f': 47},
    ]
    def safe_float(val):
        try:
            if pd.notna(val):
                return float(val)
        except (ValueError, TypeError):
            pass
        return 0.0
    records = []
    for sku_name, start_row, total_row in skus_info:
        for sr_idx, sr_name in enumerate(sub_regioes):
            row = start_row + sr_idx
            for w_idx, wc in enumerate(week_cols):
                records.append({
                    'SKU': sku_name, 'Sub_Regiao': sr_name,
                    'Semana': semanas[w_idx], 'Semana_Num': w_idx + 1,
                    'Demanda': safe_float(raw.iloc[row, wc['dem']]),
                    'WSNP': safe_float(raw.iloc[row, wc['wsnp']]),
                    'EI_Semana': safe_float(raw.iloc[row, wc['ei']]) if wc['ei'] is not None else 0.0,
                    'Suf_Ini_dias': safe_float(raw.iloc[row, wc['suf_ini']]) if wc['suf_ini'] is not None else 0.0,
                    'Transf_Interna': safe_float(raw.iloc[row, wc['tr_int']]),
                    'EF_Semana': safe_float(raw.iloc[row, wc['ef']]),
                    'Suf_Final_dias': safe_float(raw.iloc[row, wc['suf_f']])
                })
    sp_skus = [('Goose Island', 39), ('Malzbier', 40), ('Colorado', 41)]
    for sku_name, row in sp_skus:
        for w_idx, wc in enumerate(week_cols):
            records.append({
                'SKU': sku_name, 'Sub_Regiao': 'SP (Origem)',
                'Semana': semanas[w_idx], 'Semana_Num': w_idx + 1,
                'Demanda': safe_float(raw.iloc[row, wc['dem']]),
                'WSNP': safe_float(raw.iloc[row, wc['wsnp']]),
                'EI_Semana': safe_float(raw.iloc[row, wc['ei']]) if wc['ei'] is not None else 0.0,
                'Suf_Ini_dias': safe_float(raw.iloc[row, wc['suf_ini']]) if wc['suf_ini'] is not None else 0.0,
                'Transf_Interna': 0,
                'EF_Semana': safe_float(raw.iloc[row, wc['ef']]),
                'Suf_Final_dias': safe_float(raw.iloc[row, wc['suf_f']])
            })
    return pd.DataFrame(records)

df_divulgado = parse_cenario_neno('Cenário Divulgado')
df_nova_dem = parse_cenario_neno('Cenário com Nova Demanda')
print(f"✅ Aba 5 (Cenário Divulgado): {len(df_divulgado)} registros")
print(f"✅ Aba 6 (Nova Demanda): {len(df_nova_dem)} registros")

# ── Sub-DataFrames NENO ───────────────────────────────────────────────────────
div_neno = df_divulgado[df_divulgado['Sub_Regiao'] != 'SP (Origem)'].copy()
nova_neno = df_nova_dem[df_nova_dem['Sub_Regiao'] != 'SP (Origem)'].copy()

total = len(df_cenario_br) + len(df_custos_transf) + len(df_pcp) + len(df_transf) + len(df_divulgado) + len(df_nova_dem)
print(f"\n{'='*60}")
print(f"TOTAL: {total} registros carregados com sucesso")
print(f"{'='*60}")


In [ ]:
# ABA 1: CENÁRIO ATUAL BR — ANÁLISE UNIVARIADA

# 1.1 Estatísticas descritivas — Demanda por Região
print("=" * 70)
print("1.1 ESTATÍSTICAS DESCRITIVAS — DEMANDA REGIONAL (HL)")
print("=" * 70)

stats_dem = pd.DataFrame({
    'Regiao': df_cenario_br['Regiao'],
    'Demanda_Jan': df_cenario_br['Demanda_Jan'].round(0),
    'Demanda_Fev': df_cenario_br['Demanda_Fev'].round(0),
    'Variacao_Pct': ((df_cenario_br['Demanda_Fev'] - df_cenario_br['Demanda_Jan']) / df_cenario_br['Demanda_Jan'].replace(0, np.nan) * 100).round(1)
})
display(stats_dem)

# Resumo estatístico (excluindo TOTAL e Export)
regioes_operacionais = df_cenario_br[~df_cenario_br['Regiao'].isin(['Export'])]
dem_jan_vals = regioes_operacionais['Demanda_Jan']
dem_fev_vals = regioes_operacionais['Demanda_Fev']

print(f"\nJaneiro  — Media: {dem_jan_vals.mean():,.0f} HL | Mediana: {dem_jan_vals.median():,.0f} | DP: {dem_jan_vals.std():,.0f} | CV: {dem_jan_vals.std()/dem_jan_vals.mean()*100:.1f}%")
print(f"Fevereiro — Media: {dem_fev_vals.mean():,.0f} HL | Mediana: {dem_fev_vals.median():,.0f} | DP: {dem_fev_vals.std():,.0f} | CV: {dem_fev_vals.std()/dem_fev_vals.mean()*100:.1f}%")

# 1.2 Visualização: Demanda Jan vs Fev por Região
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

regioes_plot = df_cenario_br[df_cenario_br['Regiao'] != 'Export'].copy()
regioes_plot = regioes_plot.sort_values('Demanda_Jan', ascending=True)

x = np.arange(len(regioes_plot))
w = 0.35

axes[0].barh(x - w/2, regioes_plot['Demanda_Jan']/1000, w, color=CORES['azul_medio'], label='Janeiro', edgecolor='white')
axes[0].barh(x + w/2, regioes_plot['Demanda_Fev']/1000, w, color=CORES['ambar'], label='Fevereiro', edgecolor='white')
axes[0].set_yticks(x)
axes[0].set_yticklabels(regioes_plot['Regiao'])
axes[0].set_xlabel('Demanda (mil HL)')
axes[0].set_title('Demanda LN por Regiao — Jan vs Fev', fontweight='bold')
axes[0].legend()
axes[0].grid(axis='x', alpha=0.3)

# Share de cada região no total
total_fev = regioes_plot['Demanda_Fev'].sum()
shares = (regioes_plot['Demanda_Fev'] / total_fev * 100).values
colors_pie = [CORES['azul_escuro'], CORES['azul_medio'], CORES['azul_claro'], CORES['ambar'], CORES['verde'], CORES['vermelho']]
wedges, texts, autotexts = axes[1].pie(
    shares, labels=regioes_plot['Regiao'].values,
    autopct='%1.1f%%', colors=colors_pie[:len(shares)],
    startangle=90, textprops={'fontsize': 9}
)
axes[1].set_title('Share de Demanda Fev (% do total, sem Export)', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# 1.3 SUFICIÊNCIA (DOI) POR REGIÃO — FEVEREIRO

print("=" * 70)
print("1.3 SUFICIÊNCIA FINAL (DOI) POR REGIÃO — FEVEREIRO")
print("=" * 70)

suf_data = df_cenario_br[['Regiao', 'Suf_Fev_dias']].copy()
suf_data = suf_data.sort_values('Suf_Fev_dias', ascending=True)
display(suf_data)

DOI_MIN = 12  # dias

fig, ax = plt.subplots(figsize=(12, 5))

cores_barra = []
for val in suf_data['Suf_Fev_dias']:
    if val < 0:
        cores_barra.append(CORES['vermelho'])
    elif val < DOI_MIN:
        cores_barra.append(CORES['ambar'])
    else:
        cores_barra.append(CORES['verde'])

bars = ax.barh(suf_data['Regiao'], suf_data['Suf_Fev_dias'], color=cores_barra, edgecolor='white', height=0.6)
ax.axvline(x=DOI_MIN, color=CORES['vermelho'], linestyle='--', linewidth=2, label=f'DOI minimo = {DOI_MIN} dias')
ax.axvline(x=0, color='black', linewidth=0.5)

for bar, val in zip(bars, suf_data['Suf_Fev_dias']):
    x_pos = max(val + 0.5, 1)
    ax.text(x_pos, bar.get_y() + bar.get_height()/2, f'{val:.1f}d', va='center', fontsize=9, fontweight='bold')

ax.set_xlabel('Suficiência (dias)')
ax.set_title('Suficiencia Final Fev por Regiao — DOI minimo 12 dias', fontweight='bold', fontsize=13)
ax.legend(fontsize=10)
ax.grid(axis='x', alpha=0.3)

# Insights
abaixo_doi = suf_data[suf_data['Suf_Fev_dias'] < DOI_MIN]
print(f"\nRegioes ABAIXO do DOI minimo ({DOI_MIN} dias): {len(abaixo_doi)}")
for _, row in abaixo_doi.iterrows():
    status = "CRITICO" if row['Suf_Fev_dias'] < 0 else "ALERTA"
    print(f"  [{status}] {row['Regiao']}: {row['Suf_Fev_dias']:.1f} dias")

plt.tight_layout()
plt.show()

In [ ]:
# 1.4 PRODUÇÃO vs DEMANDA — BALANÇO REGIONAL

print("=" * 70)
print("1.4 TRANSFERÊNCIA DE MALHA E ESTOQUE FINAL — FEVEREIRO")
print("=" * 70)

# Transferência de malha: positivo = recebe, negativo = envia
transf_data = df_cenario_br[['Regiao', 'Transf_Malha_Fev', 'EFM_Fev']].copy()
transf_data = transf_data[transf_data['Regiao'] != 'Export']
transf_data = transf_data.sort_values('Transf_Malha_Fev')
display(transf_data)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Transferência de Malha
cores_transf = [CORES['verde'] if v >= 0 else CORES['vermelho'] for v in transf_data['Transf_Malha_Fev']]
axes[0].barh(transf_data['Regiao'], transf_data['Transf_Malha_Fev']/1000, color=cores_transf, edgecolor='white')
axes[0].axvline(x=0, color='black', linewidth=1)
axes[0].set_xlabel('Volume (mil HL)')
axes[0].set_title('Transferencia de Malha Fev\n(+recebe / -envia)', fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)

for i, (_, row) in enumerate(transf_data.iterrows()):
    val = row['Transf_Malha_Fev']
    axes[0].text(val/1000 + (2 if val >= 0 else -2), i, f'{val/1000:.1f}k', va='center', fontsize=8, fontweight='bold',
                 ha='left' if val >= 0 else 'right')

# Estoque Final
ef_sorted = transf_data.sort_values('EFM_Fev')
cores_ef = [CORES['vermelho'] if v < 0 else CORES['azul_medio'] for v in ef_sorted['EFM_Fev']]
axes[1].barh(ef_sorted['Regiao'], ef_sorted['EFM_Fev']/1000, color=cores_ef, edgecolor='white')
axes[1].axvline(x=0, color='black', linewidth=1)
axes[1].set_xlabel('Volume (mil HL)')
axes[1].set_title('Estoque Final Mes Fev (EFM)', fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

# Insight chave
print("\n--- INSIGHT ABA 1 ---")
print("SP e RJ sao EXPORTADORES liquidos (enviam para outras regioes)")
print("NENO e CO sao IMPORTADORES (recebem de outras regioes)")
print(f"NENO: Suf. Fev = {df_cenario_br[df_cenario_br['Regiao']=='NENO']['Suf_Fev_dias'].values[0]:.1f} dias — PROXIMO do limite de 12 dias")

---
## 17. Análise Univariada — Aba 2: Custos de Transferência e Economia dos SKUs

> **O que é esta aba?** Mapeia os **custos logísticos por rota** (R\$/HL) e a **economia unitária** de cada SKU (MACO, custo de produção, margem). É a base para avaliar se vale transferir volume de SP para o Nordeste.

**Variáveis:** Custo por rota (R\$/HL), MACO (R\$/HL), Custo de Produção (R\$/HL), Margem Bruta (R\$/HL)

In [ ]:
# ABA 2: CUSTOS — ANÁLISE UNIVARIADA

print("=" * 70)
print("2.1 CUSTOS DE TRANSFERÊNCIA POR ROTA")
print("=" * 70)

# Simplificar nomes
df_custos_viz = df_custos_transf.copy()
df_custos_viz['SKU_curto'] = df_custos_viz['SKU'].str.split(' ').str[0]
df_custos_viz['Rota'] = (df_custos_viz['Origem'].str.extract(r'F\. (\w+)')[0] +
                          ' -> ' +
                          df_custos_viz['Destino'].str.extract(r'F\. (\w+)')[0])

display(df_custos_viz[['SKU_curto', 'Rota', 'Custo_R_HL']].sort_values('Custo_R_HL'))

# Estatísticas
print(f"\nCusto medio: R$ {df_custos_transf['Custo_R_HL'].mean():.2f}/HL")
print(f"Custo min:   R$ {df_custos_transf['Custo_R_HL'].min():.2f}/HL (Colorado, Jacarei->Camacari)")
print(f"Custo max:   R$ {df_custos_transf['Custo_R_HL'].max():.2f}/HL (Malzbier, Jaguariuna->Fonte Mata)")
print(f"Amplitude:   R$ {df_custos_transf['Custo_R_HL'].max() - df_custos_transf['Custo_R_HL'].min():.2f}/HL")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de custos por rota
df_sorted = df_custos_viz.sort_values('Custo_R_HL')
colors_sku = {'COLORADO': CORES['verde'], 'GOOSE': CORES['azul_medio'], 'MALZBIER': CORES['ambar']}
bar_colors = [colors_sku.get(s, CORES['cinza_medio']) for s in df_sorted['SKU_curto']]

bars = axes[0].barh(range(len(df_sorted)), df_sorted['Custo_R_HL'], color=bar_colors, edgecolor='white')
axes[0].set_yticks(range(len(df_sorted)))
axes[0].set_yticklabels([f"{r['SKU_curto']}\n{r['Rota']}" for _, r in df_sorted.iterrows()], fontsize=8)
axes[0].set_xlabel('R$/HL')
axes[0].set_title('Custo de Transferencia por Rota', fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)

for bar, val in zip(bars, df_sorted['Custo_R_HL']):
    axes[0].text(val + 0.5, bar.get_y() + bar.get_height()/2, f'R${val:.1f}', va='center', fontsize=8)

# Economia dos SKUs (MACO, Custo Prod, Margem)
x_pos = np.arange(len(df_sku_economics))
w = 0.25

axes[1].bar(x_pos - w, df_sku_economics['MACO_R_HL'], w, color=CORES['azul_escuro'], label='MACO', edgecolor='white')
axes[1].bar(x_pos, df_sku_economics['Custo_Prod_R_HL'], w, color=CORES['vermelho'], label='Custo Prod.', edgecolor='white')
axes[1].bar(x_pos + w, df_sku_economics['Margem_Bruta'], w, color=CORES['verde'], label='Margem Bruta', edgecolor='white')

axes[1].set_xticks(x_pos)
sku_labels = [s.split()[0] for s in df_sku_economics['SKU']]
axes[1].set_xticklabels(sku_labels, fontsize=9)
axes[1].set_ylabel('R$/HL')
axes[1].set_title('Economia Unitaria por SKU', fontweight='bold')
axes[1].legend(fontsize=8)
axes[1].grid(axis='y', alpha=0.3)

for i, row in df_sku_economics.iterrows():
    axes[1].text(i + w, row['Margem_Bruta'] + 3, f'R${row["Margem_Bruta"]:.0f}\n({row["Margem_Pct"]}%)',
                 ha='center', fontsize=8, fontweight='bold', color=CORES['verde'])

plt.tight_layout()
plt.show()

print("\n--- INSIGHT ABA 2 ---")
print(f"Goose Island tem MAIOR MACO (R$350/HL) mas custo transf. tambem eh alto")
print(f"Malzbier: margem R$136/HL ({df_sku_economics[df_sku_economics['SKU'].str.contains('MALZ')]['Margem_Pct'].values[0]}%) — custo transf. R$84-95/HL")
print(f"Transferir Malzbier consome {84.58/136*100:.0f}%-{95.33/136*100:.0f}% da margem bruta (rota Camacari vs Fonte Mata)")

---
## 18. Análise Univariada — Aba 3: Produção PCP (Fevereiro, 4 semanas)

> **O que é esta aba?** Programação semanal de produção para as **2 plantas do Nordeste** (AQ541 em Aquiraz/CE e PE541 em Pernambuco). Mostra quais SKUs são produzidos quando e em que volume.

**Informação-chave do case:** Goose Island MIDWAY tem **restrição de elaboração de líquido** em Pernambuco — não pode aumentar produção. Capacidade: AQ541 = 12.600 HL/sem | PE541 = 27.000 HL/sem.

In [ ]:
# ABA 3: PRODUÇÃO PCP — ANÁLISE UNIVARIADA

print("=" * 70)
print("3.1 VOLUME SEMANAL POR PLANTA")
print("=" * 70)

# Agregação por planta e semana
prod_planta_sem = df_pcp.groupby(['Planta', 'Semana', 'Semana_Num'])['Volume_HL'].sum().reset_index()
prod_planta_sem = prod_planta_sem.sort_values(['Planta', 'Semana_Num'])

# Utilização
for planta in ['AQ541 (CE)', 'PE541 (PE)']:
    cap = cap_aq if 'AQ' in planta else cap_pe
    mask = prod_planta_sem['Planta'] == planta
    prod_planta_sem.loc[mask, 'Capacidade'] = cap
    prod_planta_sem.loc[mask, 'Utilizacao_Pct'] = (prod_planta_sem.loc[mask, 'Volume_HL'] / cap * 100)

display(prod_planta_sem[['Planta', 'Semana', 'Volume_HL', 'Capacidade', 'Utilizacao_Pct']].round(1))

# Estatísticas por planta
for planta in ['AQ541 (CE)', 'PE541 (PE)']:
    dados = prod_planta_sem[prod_planta_sem['Planta'] == planta]['Utilizacao_Pct']
    print(f"\n{planta}:")
    print(f"  Utilizacao media: {dados.mean():.1f}% | Min: {dados.min():.1f}% | Max: {dados.max():.1f}%")
    print(f"  Desvio padrao: {dados.std():.1f}% | CV: {dados.std()/dados.mean()*100:.1f}%")

# Visualização
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 3.1a Utilização semanal por planta
for idx, planta in enumerate(['AQ541 (CE)', 'PE541 (PE)']):
    data = prod_planta_sem[prod_planta_sem['Planta'] == planta]
    cap = cap_aq if 'AQ' in planta else cap_pe

    bars = axes[0, idx].bar(data['Semana'], data['Volume_HL']/1000,
                            color=CORES['azul_medio'], edgecolor='white', width=0.6)
    axes[0, idx].axhline(y=cap/1000, color=CORES['vermelho'], linestyle='--', linewidth=2,
                         label=f'Capacidade: {cap/1000:.1f}k HL')
    axes[0, idx].set_ylabel('Volume (mil HL)')
    axes[0, idx].set_title(f'{planta} — Producao Semanal', fontweight='bold')
    axes[0, idx].legend(fontsize=8)
    axes[0, idx].tick_params(axis='x', rotation=20, labelsize=8)
    axes[0, idx].grid(axis='y', alpha=0.3)

    for bar, val in zip(bars, data['Utilizacao_Pct']):
        axes[0, idx].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                         f'{val:.0f}%', ha='center', fontsize=9, fontweight='bold')

# 3.1b Mix de SKU por planta (stacked bar)
for idx, planta in enumerate(['AQ541 (CE)', 'PE541 (PE)']):
    pdata = df_pcp[df_pcp['Planta'] == planta].pivot_table(
        index='Semana_Num', columns='SKU', values='Volume_HL', aggfunc='sum', fill_value=0)

    pdata = pdata.reindex(columns=pdata.sum().sort_values(ascending=False).index)

    bottom = np.zeros(len(pdata))
    sku_colors = [CORES['ambar'], CORES['azul_medio'], CORES['verde'], CORES['vermelho'], CORES['azul_claro'], CORES['cinza_medio']]

    for col_idx, col in enumerate(pdata.columns):
        vals = pdata[col].values
        c = sku_colors[col_idx % len(sku_colors)]
        axes[1, idx].bar(pdata.index, vals/1000, bottom=bottom/1000, label=col, color=c, edgecolor='white', width=0.6)
        bottom += vals

    cap = cap_aq if 'AQ' in planta else cap_pe
    axes[1, idx].axhline(y=cap/1000, color=CORES['vermelho'], linestyle='--', linewidth=2)
    axes[1, idx].set_xlabel('Semana')
    axes[1, idx].set_ylabel('Volume (mil HL)')
    axes[1, idx].set_title(f'{planta} — Mix de SKU por Semana', fontweight='bold')
    axes[1, idx].legend(fontsize=7, loc='upper right')
    axes[1, idx].set_xticks([1,2,3,4])
    axes[1, idx].set_xticklabels(['W1', 'W2', 'W3', 'W4'])
    axes[1, idx].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# SKU com maior volume por planta
print("\n--- INSIGHT ABA 3 ---")
sku_totals = df_pcp.groupby(['Planta', 'SKU'])['Volume_HL'].sum().reset_index()
for planta in ['AQ541 (CE)', 'PE541 (PE)']:
    top = sku_totals[sku_totals['Planta'] == planta].sort_values('Volume_HL', ascending=False)
    print(f"\n{planta} — Volume total 4 semanas:")
    for _, r in top.iterrows():
        if r['Volume_HL'] > 0:
            print(f"  {r['SKU']}: {r['Volume_HL']:,.0f} HL")

ociosidade_aq = prod_planta_sem[prod_planta_sem['Planta'] == 'AQ541 (CE)']
ociosas = ociosidade_aq[ociosidade_aq['Utilizacao_Pct'] < 100]
if len(ociosas) > 0:
    print(f"\nAQ541 tem ociosidade em {len(ociosas)}/4 semanas — potencial para realocar producao")
    gap_total = (ociosidade_aq['Capacidade'] - ociosidade_aq['Volume_HL']).sum()
    print(f"Gap de capacidade total AQ541 no mes: {gap_total:,.0f} HL")

---
## 19. Análise Univariada — Aba 4: Transferências Programadas

> **O que é esta aba?** Lista as transferências **já planejadas** (antes de qualquer nova ação). Atualmente, apenas **Goose Island** tem transferências programadas, todas via **cabotagem** de SP para o Nordeste.

**Nota:** Nenhum outro SKU (Malzbier, Colorado, Patagonia) tem transferência programada para o NE neste momento.

In [ ]:
# ABA 4: TRANSFERÊNCIAS PROGRAMADAS — ANÁLISE UNIVARIADA

print("=" * 70)
print("4.1 TRANSFERÊNCIAS PROGRAMADAS (SOMENTE GOOSE ISLAND)")
print("=" * 70)

# Pivot: Destino x Semana
transf_pivot = df_transf.pivot_table(index='Destino', columns='Semana', values='Volume_HL', aggfunc='sum')
transf_pivot['TOTAL'] = transf_pivot.sum(axis=1)
display(transf_pivot)

total_transf = df_transf['Volume_HL'].sum()
print(f"\nVolume total programado: {total_transf:,.1f} HL (todas 4 semanas)")
print(f"Volume medio semanal: {total_transf/4:,.1f} HL/semana")

# Visualização
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Por destino
dest_totals = df_transf.groupby('Destino')['Volume_HL'].sum().sort_values(ascending=True)
cores_dest = [CORES['azul_claro'], CORES['azul_medio'], CORES['azul_escuro']]
axes[0].barh(dest_totals.index, dest_totals.values/1000, color=cores_dest[:len(dest_totals)], edgecolor='white')
axes[0].set_xlabel('Volume Total 4 semanas (mil HL)')
axes[0].set_title('Goose Island — Transf. Programadas por Destino', fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)
for i, (dest, val) in enumerate(dest_totals.items()):
    axes[0].text(val/1000 + 0.2, i, f'{val:,.0f} HL', va='center', fontsize=9)

# Evolução semanal
transf_sem = df_transf.groupby(['Semana_Num', 'Destino'])['Volume_HL'].sum().reset_index()
for dest in df_transf['Destino'].unique():
    data = transf_sem[transf_sem['Destino'] == dest]
    axes[1].plot(data['Semana_Num'], data['Volume_HL']/1000, marker='o', linewidth=2, label=dest)

axes[1].set_xlabel('Semana')
axes[1].set_ylabel('Volume (mil HL)')
axes[1].set_title('Evolucao Semanal das Transferencias', fontweight='bold')
axes[1].set_xticks([1,2,3,4])
axes[1].set_xticklabels(['W1', 'W2', 'W3', 'W4'])
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n--- INSIGHT ABA 4 ---")
print("Apenas Goose Island tem transferencias programadas")
print("Malzbier NAO tem nenhuma transferencia — todo gap deve ser coberto com NOVAS acoes")
print(f"F. Camacari recebe volume constante (7.200 HL/sem)")
print(f"F. Fonte Mata recebe volume apenas na W1 ({df_transf[(df_transf['Destino'].str.contains('FONTE')) & (df_transf['Semana_Num']==1)]['Volume_HL'].sum():,.0f} HL)")

---
## 20. Análise Univariada — Aba 5: Cenário Divulgado NENO (Baseline)

> **O que é esta aba?** O cenário **antes** do aumento de demanda de Malzbier. Mostra semana a semana (W0–W3 de Fevereiro) como ficam **demanda, estoque e suficiência** para cada SKU em cada sub-região do NENO.

**Sub-regiões:** Mapapi, NE Norte, NE Sul, NO Araguaia, NO Centro
**SKUs:** Patagonia, Goose Island, Malzbier, Colorado

In [ ]:
# ABA 5: CENÁRIO DIVULGADO — ANÁLISE UNIVARIADA

# Filtrar apenas NENO (excluir SP)
div_neno = df_divulgado[df_divulgado['Sub_Regiao'] != 'SP (Origem)'].copy()

print("=" * 70)
print("5.1 DEMANDA TOTAL POR SKU NO NENO — CENÁRIO DIVULGADO")
print("=" * 70)

dem_por_sku = div_neno.groupby('SKU')['Demanda'].sum().sort_values(ascending=False)
print("\nDemanda total 4 semanas (HL):")
for sku, val in dem_por_sku.items():
    share = val / dem_por_sku.sum() * 100
    print(f"  {sku}: {val:,.0f} HL ({share:.1f}%)")

print(f"\nTOTAL NENO: {dem_por_sku.sum():,.0f} HL")

# Demanda por sub-região (agregando todos os SKUs)
dem_por_sr = div_neno.groupby('Sub_Regiao')['Demanda'].sum().sort_values(ascending=False)
print(f"\nDemanda por sub-regiao (todos os SKUs):")
for sr, val in dem_por_sr.items():
    share = val / dem_por_sr.sum() * 100
    print(f"  {sr}: {val:,.0f} HL ({share:.1f}%)")

# Visualização 5.1
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# a) Demanda por SKU
colors_sku_list = [CORES['ambar'], CORES['azul_medio'], CORES['verde'], CORES['azul_claro']]
axes[0,0].bar(range(len(dem_por_sku)), dem_por_sku.values/1000, color=colors_sku_list[:len(dem_por_sku)], edgecolor='white')
axes[0,0].set_xticks(range(len(dem_por_sku)))
axes[0,0].set_xticklabels(dem_por_sku.index, fontsize=9)
axes[0,0].set_ylabel('Demanda (mil HL)')
axes[0,0].set_title('Demanda Total NENO por SKU (4 semanas)', fontweight='bold')
axes[0,0].grid(axis='y', alpha=0.3)
for i, val in enumerate(dem_por_sku.values):
    axes[0,0].text(i, val/1000 + 0.3, f'{val/1000:.1f}k', ha='center', fontsize=9, fontweight='bold')

# b) Demanda por sub-região
axes[0,1].barh(dem_por_sr.index, dem_por_sr.values/1000, color=CORES['azul_medio'], edgecolor='white')
axes[0,1].set_xlabel('Demanda (mil HL)')
axes[0,1].set_title('Demanda Total por Sub-Regiao', fontweight='bold')
axes[0,1].grid(axis='x', alpha=0.3)

# c) Heatmap: Demanda SKU × Sub-região (W0-W3 somado)
heat_data = div_neno.groupby(['SKU', 'Sub_Regiao'])['Demanda'].sum().reset_index()
heat_pivot = heat_data.pivot(index='Sub_Regiao', columns='SKU', values='Demanda').fillna(0)

im = axes[1,0].imshow(heat_pivot.values, cmap='YlOrRd', aspect='auto')
axes[1,0].set_xticks(range(len(heat_pivot.columns)))
axes[1,0].set_xticklabels(heat_pivot.columns, fontsize=8, rotation=20)
axes[1,0].set_yticks(range(len(heat_pivot.index)))
axes[1,0].set_yticklabels(heat_pivot.index, fontsize=9)
axes[1,0].set_title('Heatmap: Demanda SKU x Sub-Regiao (HL)', fontweight='bold')

for i in range(len(heat_pivot.index)):
    for j in range(len(heat_pivot.columns)):
        val = heat_pivot.iloc[i, j]
        color = 'white' if val > heat_pivot.values.max() * 0.6 else 'black'
        axes[1,0].text(j, i, f'{val:,.0f}', ha='center', va='center', fontsize=7, color=color)

plt.colorbar(im, ax=axes[1,0], shrink=0.8)

# d) Evolução semanal da demanda total
dem_semanal = div_neno.groupby('Semana_Num')['Demanda'].sum()
axes[1,1].plot(dem_semanal.index, dem_semanal.values/1000, marker='s', linewidth=2.5,
               color=CORES['azul_escuro'], markersize=8)
axes[1,1].fill_between(dem_semanal.index, dem_semanal.values/1000, alpha=0.15, color=CORES['azul_medio'])
axes[1,1].set_xlabel('Semana')
axes[1,1].set_ylabel('Demanda Total (mil HL)')
axes[1,1].set_title('Evolucao Semanal da Demanda NENO', fontweight='bold')
axes[1,1].set_xticks([1,2,3,4])
axes[1,1].set_xticklabels(['W1', 'W2', 'W3', 'W4'])
axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 5.2 SUFICIÊNCIA (DOI) POR SKU × SUB-REGIÃO — CENÁRIO DIVULGADO

print("=" * 70)
print("5.2 SUFICIÊNCIA FINAL (DOI) — CENÁRIO DIVULGADO (W3)")
print("=" * 70)

# Pegar suficiência da última semana (W3) como indicador final
suf_w3 = div_neno[div_neno['Semana_Num'] == 4][['SKU', 'Sub_Regiao', 'Suf_Final_dias']].copy()
suf_pivot = suf_w3.pivot(index='Sub_Regiao', columns='SKU', values='Suf_Final_dias').round(1)
display(suf_pivot)

DOI_MIN = 12

# Contagem de células críticas
total_cells = suf_pivot.size
criticas = (suf_pivot < DOI_MIN).sum().sum()
negativas = (suf_pivot < 0).sum().sum()
print(f"\nCelulas com DOI < {DOI_MIN} dias: {criticas}/{total_cells} ({criticas/total_cells*100:.0f}%)")
print(f"Celulas com DOI NEGATIVO (ruptura): {negativas}/{total_cells}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap de suficiência
import matplotlib.colors as mcolors
# Custom colormap: vermelho -> amarelo -> verde
cmap = mcolors.LinearSegmentedColormap.from_list('doi', ['#E74C3C', '#F5A623', '#27AE60'])
norm = mcolors.TwoSlopeNorm(vmin=suf_pivot.values.min(), vcenter=DOI_MIN, vmax=max(suf_pivot.values.max(), 30))

im = axes[0].imshow(suf_pivot.values, cmap=cmap, norm=norm, aspect='auto')
axes[0].set_xticks(range(len(suf_pivot.columns)))
axes[0].set_xticklabels(suf_pivot.columns, fontsize=8, rotation=20)
axes[0].set_yticks(range(len(suf_pivot.index)))
axes[0].set_yticklabels(suf_pivot.index, fontsize=9)
axes[0].set_title(f'Suficiencia Final W3 (dias) — Min: {DOI_MIN}d', fontweight='bold')

for i in range(len(suf_pivot.index)):
    for j in range(len(suf_pivot.columns)):
        val = suf_pivot.iloc[i, j]
        color = 'white' if val < 5 else 'black'
        marker = ' !!' if val < DOI_MIN else ''
        axes[0].text(j, i, f'{val:.1f}d{marker}', ha='center', va='center', fontsize=8,
                    fontweight='bold' if val < DOI_MIN else 'normal', color=color)

plt.colorbar(im, ax=axes[0], shrink=0.8, label='DOI (dias)')

# Grouped bar: Suf final W3 por sub-região, agrupado por SKU
skus = suf_pivot.columns.tolist()
x = np.arange(len(suf_pivot.index))
w = 0.18
sku_cols = [CORES['ambar'], CORES['azul_medio'], CORES['verde'], CORES['azul_claro']]

for i, sku in enumerate(skus):
    vals = suf_pivot[sku].values
    axes[1].bar(x + i*w - w*1.5, vals, w, label=sku, color=sku_cols[i % len(sku_cols)], edgecolor='white')

axes[1].axhline(y=DOI_MIN, color=CORES['vermelho'], linestyle='--', linewidth=2, label=f'DOI min = {DOI_MIN}d')
axes[1].set_xticks(x)
axes[1].set_xticklabels(suf_pivot.index, fontsize=8)
axes[1].set_ylabel('Suficiencia (dias)')
axes[1].set_title('DOI Final W3 por Sub-Regiao e SKU', fontweight='bold')
axes[1].legend(fontsize=7, loc='upper right')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Identificar pontos críticos
print("\n--- PONTOS CRITICOS (DOI < 12 dias no cenario DIVULGADO) ---")
for sr in suf_pivot.index:
    for sku in suf_pivot.columns:
        val = suf_pivot.loc[sr, sku]
        if val < DOI_MIN:
            status = "RUPTURA" if val < 0 else "CRITICO" if val < 5 else "ALERTA"
            print(f"  [{status}] {sr} / {sku}: {val:.1f} dias")

---
## 21. Análise Univariada — Aba 6: Cenário com Nova Demanda (+30% Malzbier)

> **O que é esta aba?** O cenário com o **aumento de +30% na demanda de Malzbier Brahma** para Fevereiro. É o cenário que precisa ser resolvido. Comparar com Aba 5 revela exatamente **onde** e **quanto** a pressão aumenta.

**Mudança:** Malzbier +30% em todas as sub-regiões. Demais SKUs: **inalterados**.
**Gap a cobrir:** ~4.500 HL (simplificado) ou até ~11.629 HL considerando bias de +9%.

In [ ]:
# ABA 6: NOVA DEMANDA — ANÁLISE UNIVARIADA + COMPARAÇÃO

nova_neno = df_nova_dem[df_nova_dem['Sub_Regiao'] != 'SP (Origem)'].copy()

print("=" * 70)
print("6.1 IMPACTO DO +30% MALZBIER NA DEMANDA NENO")
print("=" * 70)

# Comparar demanda total por SKU: Divulgado vs Nova
dem_div = div_neno.groupby('SKU')['Demanda'].sum()
dem_nova = nova_neno.groupby('SKU')['Demanda'].sum()

comp = pd.DataFrame({
    'Divulgado_HL': dem_div,
    'Nova_Dem_HL': dem_nova,
})
comp['Delta_HL'] = comp['Nova_Dem_HL'] - comp['Divulgado_HL']
comp['Delta_Pct'] = (comp['Delta_HL'] / comp['Divulgado_HL'] * 100).round(1)
display(comp)

gap_malzbier = comp.loc['Malzbier', 'Delta_HL']
print(f"\nGap total Malzbier: {gap_malzbier:,.0f} HL (4 semanas)")
print(f"Gap semanal medio: {gap_malzbier/4:,.0f} HL/semana")

# Impacto por sub-região (apenas Malzbier)
print(f"\n{'='*70}")
print(f"6.2 GAP DE MALZBIER POR SUB-REGIÃO")
print(f"{'='*70}")

malz_div = div_neno[div_neno['SKU']=='Malzbier'].groupby('Sub_Regiao')['Demanda'].sum()
malz_nova = nova_neno[nova_neno['SKU']=='Malzbier'].groupby('Sub_Regiao')['Demanda'].sum()

gap_sr = pd.DataFrame({
    'Divulgado': malz_div,
    'Nova_Dem': malz_nova,
    'Gap_HL': malz_nova - malz_div,
    'Gap_Pct': ((malz_nova - malz_div) / malz_div * 100).round(1)
})
gap_sr = gap_sr.sort_values('Gap_HL', ascending=False)
display(gap_sr)

# Visualização
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# a) Demanda por SKU: antes vs depois
x = np.arange(len(comp))
w = 0.35
axes[0,0].bar(x - w/2, comp['Divulgado_HL']/1000, w, color=CORES['azul_medio'], label='Divulgado', edgecolor='white')
axes[0,0].bar(x + w/2, comp['Nova_Dem_HL']/1000, w, color=CORES['vermelho'], label='Nova Demanda', edgecolor='white')
axes[0,0].set_xticks(x)
axes[0,0].set_xticklabels(comp.index, fontsize=9)
axes[0,0].set_ylabel('Demanda (mil HL)')
axes[0,0].set_title('Demanda NENO: Divulgado vs Nova (+30% Malz)', fontweight='bold')
axes[0,0].legend()
axes[0,0].grid(axis='y', alpha=0.3)

# Anotar delta
for i, (sku, row) in enumerate(comp.iterrows()):
    if abs(row['Delta_HL']) > 10:
        axes[0,0].text(i + w/2, row['Nova_Dem_HL']/1000 + 0.3, f'+{row["Delta_HL"]/1000:.1f}k',
                       ha='center', fontsize=8, fontweight='bold', color=CORES['vermelho'])

# b) Gap Malzbier por sub-região
gap_sorted = gap_sr.sort_values('Gap_HL', ascending=True)
colors_gap = [CORES['vermelho'] if g > 0 else CORES['verde'] for g in gap_sorted['Gap_HL']]
axes[0,1].barh(gap_sorted.index, gap_sorted['Gap_HL'], color=colors_gap, edgecolor='white')
axes[0,1].set_xlabel('Gap (HL)')
axes[0,1].set_title('Gap de Malzbier por Sub-Regiao (+30%)', fontweight='bold')
axes[0,1].grid(axis='x', alpha=0.3)
for i, (sr, row) in enumerate(gap_sorted.iterrows()):
    axes[0,1].text(row['Gap_HL'] + 20, i, f'+{row["Gap_HL"]:.0f} HL', va='center', fontsize=8)

# c) Suficiência W3: Divulgado vs Nova Demanda (Malzbier only)
suf_div_malz = div_neno[(div_neno['SKU']=='Malzbier') & (div_neno['Semana_Num']==4)][['Sub_Regiao','Suf_Final_dias']]
suf_nova_malz = nova_neno[(nova_neno['SKU']=='Malzbier') & (nova_neno['Semana_Num']==4)][['Sub_Regiao','Suf_Final_dias']]

suf_comp = pd.merge(suf_div_malz, suf_nova_malz, on='Sub_Regiao', suffixes=('_Div', '_Nova'))
suf_comp = suf_comp.sort_values('Suf_Final_dias_Nova')

x = np.arange(len(suf_comp))
axes[1,0].bar(x - 0.2, suf_comp['Suf_Final_dias_Div'], 0.35, color=CORES['azul_medio'], label='Divulgado', edgecolor='white')
axes[1,0].bar(x + 0.2, suf_comp['Suf_Final_dias_Nova'], 0.35, color=CORES['vermelho'], label='Nova Demanda', edgecolor='white')
axes[1,0].axhline(y=DOI_MIN, color='black', linestyle='--', linewidth=1.5, label=f'DOI min = {DOI_MIN}d')
axes[1,0].set_xticks(x)
axes[1,0].set_xticklabels(suf_comp['Sub_Regiao'], fontsize=8, rotation=15)
axes[1,0].set_ylabel('DOI (dias)')
axes[1,0].set_title('Malzbier: DOI Final W3 — Antes vs Depois', fontweight='bold')
axes[1,0].legend(fontsize=8)
axes[1,0].grid(axis='y', alpha=0.3)

# d) Evolução semanal Malzbier: div vs nova
malz_div_sem = div_neno[div_neno['SKU']=='Malzbier'].groupby('Semana_Num')['Demanda'].sum()
malz_nova_sem = nova_neno[nova_neno['SKU']=='Malzbier'].groupby('Semana_Num')['Demanda'].sum()

axes[1,1].plot(malz_div_sem.index, malz_div_sem.values/1000, marker='o', linewidth=2,
               color=CORES['azul_medio'], label='Divulgado')
axes[1,1].plot(malz_nova_sem.index, malz_nova_sem.values/1000, marker='s', linewidth=2,
               color=CORES['vermelho'], label='Nova Demanda (+30%)')
axes[1,1].fill_between(malz_nova_sem.index, malz_div_sem.values/1000, malz_nova_sem.values/1000,
                        alpha=0.2, color=CORES['vermelho'], label='Gap')
axes[1,1].set_xlabel('Semana')
axes[1,1].set_ylabel('Demanda Malzbier (mil HL)')
axes[1,1].set_title('Malzbier NENO: Evolucao Semanal do Gap', fontweight='bold')
axes[1,1].set_xticks([1,2,3,4])
axes[1,1].set_xticklabels(['W1', 'W2', 'W3', 'W4'])
axes[1,1].legend(fontsize=8)
axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 6.3 ANÁLISE DE SUFICIÊNCIA COMPLETA — NOVA DEMANDA (TODOS SKUs)

print("=" * 70)
print("6.3 SUFICIÊNCIA FINAL W3 — CENÁRIO NOVA DEMANDA (TODOS SKUs)")
print("=" * 70)

suf_nova_w3 = nova_neno[nova_neno['Semana_Num'] == 4][['SKU', 'Sub_Regiao', 'Suf_Final_dias']].copy()
suf_nova_pivot = suf_nova_w3.pivot(index='Sub_Regiao', columns='SKU', values='Suf_Final_dias').round(1)
display(suf_nova_pivot)

# Comparação com cenário divulgado
print(f"\n{'='*70}")
print(f"6.4 DELTA DE DOI: (Nova Demanda - Divulgado)")
print(f"{'='*70}")

delta_doi = suf_nova_pivot - suf_pivot
display(delta_doi.round(1))

# Visualização final: Heatmap de suficiência Nova Demanda
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

import matplotlib.colors as mcolors
cmap = mcolors.LinearSegmentedColormap.from_list('doi', ['#E74C3C', '#F5A623', '#27AE60'])
norm = mcolors.TwoSlopeNorm(vmin=min(suf_nova_pivot.values.min(), -7), vcenter=DOI_MIN, vmax=max(suf_nova_pivot.values.max(), 30))

im = axes[0].imshow(suf_nova_pivot.values, cmap=cmap, norm=norm, aspect='auto')
axes[0].set_xticks(range(len(suf_nova_pivot.columns)))
axes[0].set_xticklabels(suf_nova_pivot.columns, fontsize=8, rotation=20)
axes[0].set_yticks(range(len(suf_nova_pivot.index)))
axes[0].set_yticklabels(suf_nova_pivot.index, fontsize=9)
axes[0].set_title('DOI Final W3 — Nova Demanda (+30% Malz)', fontweight='bold')

for i in range(len(suf_nova_pivot.index)):
    for j in range(len(suf_nova_pivot.columns)):
        val = suf_nova_pivot.iloc[i, j]
        color = 'white' if val < 5 else 'black'
        marker = ' !!' if val < DOI_MIN else ''
        axes[0].text(j, i, f'{val:.1f}d{marker}', ha='center', va='center', fontsize=8,
                    fontweight='bold' if val < DOI_MIN else 'normal', color=color)

plt.colorbar(im, ax=axes[0], shrink=0.8, label='DOI (dias)')

# Delta DOI heatmap (impacto)
max_abs = max(abs(delta_doi.values.min()), abs(delta_doi.values.max()), 1)
im2 = axes[1].imshow(delta_doi.values, cmap='RdYlGn', vmin=-max_abs, vmax=max_abs, aspect='auto')
axes[1].set_xticks(range(len(delta_doi.columns)))
axes[1].set_xticklabels(delta_doi.columns, fontsize=8, rotation=20)
axes[1].set_yticks(range(len(delta_doi.index)))
axes[1].set_yticklabels(delta_doi.index, fontsize=9)
axes[1].set_title('Delta DOI (Nova - Divulgado)', fontweight='bold')

for i in range(len(delta_doi.index)):
    for j in range(len(delta_doi.columns)):
        val = delta_doi.iloc[i, j]
        if abs(val) > 0.1:
            color = 'white' if abs(val) > max_abs * 0.6 else 'black'
            sign = '+' if val > 0 else ''
            axes[1].text(j, i, f'{sign}{val:.1f}d', ha='center', va='center', fontsize=8, color=color)

plt.colorbar(im2, ax=axes[1], shrink=0.8, label='Delta DOI (dias)')

plt.tight_layout()
plt.show()

# Resumo de pontos críticos
print("\n--- PONTOS CRITICOS NA NOVA DEMANDA ---")
for sr in suf_nova_pivot.index:
    for sku in suf_nova_pivot.columns:
        val = suf_nova_pivot.loc[sr, sku]
        if val < DOI_MIN:
            delta = delta_doi.loc[sr, sku]
            agravou = " (AGRAVOU)" if delta < -0.5 else ""
            status = "RUPTURA" if val < 0 else "CRITICO" if val < 5 else "ALERTA"
            print(f"  [{status}] {sr} / {sku}: {val:.1f} dias (delta: {delta:+.1f}d){agravou}")

---
## 22. Resumo Estatístico Consolidado — Todas as Variáveis

> Visão geral das **estatísticas descritivas** de cada variável numérica relevante, consolidando insights de todas as 6 abas para facilitar a tomada de decisão.

In [ ]:
# RESUMO ESTATÍSTICO CONSOLIDADO

print("=" * 70)
print("ESTATÍSTICAS DESCRITIVAS CONSOLIDADAS")
print("=" * 70)

# 1. Demanda NENO Nova por SKU
print("\n[1] DEMANDA NENO SEMANAL (Nova Demanda) — HL/semana:")
for sku in ['Patagonia', 'Goose Island', 'Malzbier', 'Colorado']:
    vals = nova_neno[nova_neno['SKU']==sku].groupby('Semana_Num')['Demanda'].sum()
    print(f"  {sku}: Media={vals.mean():,.0f} | DP={vals.std():,.0f} | Min={vals.min():,.0f} | Max={vals.max():,.0f} | CV={vals.std()/vals.mean()*100:.1f}%")

# 2. Suficiência NENO Nova
print("\n[2] SUFICIÊNCIA FINAL (DOI) — Nova Demanda (todos SKU x Sub-Regiao x Semana):")
suf_all = nova_neno['Suf_Final_dias']
print(f"  Media: {suf_all.mean():.1f} dias | Mediana: {suf_all.median():.1f} | DP: {suf_all.std():.1f}")
print(f"  Min: {suf_all.min():.1f} | Max: {suf_all.max():.1f} | < 12 dias: {(suf_all < 12).sum()}/{len(suf_all)} ({(suf_all < 12).mean()*100:.1f}%)")

# 3. Custos
print("\n[3] CUSTOS DE TRANSFERÊNCIA — R$/HL:")
print(f"  Media: R${df_custos_transf['Custo_R_HL'].mean():.2f} | DP: R${df_custos_transf['Custo_R_HL'].std():.2f}")
print(f"  Min: R${df_custos_transf['Custo_R_HL'].min():.2f} | Max: R${df_custos_transf['Custo_R_HL'].max():.2f}")

# 4. Produção
print("\n[4] PRODUÇÃO PCP — Utilização Semanal:")
for planta in ['AQ541 (CE)', 'PE541 (PE)']:
    ut = prod_planta_sem[prod_planta_sem['Planta']==planta]['Utilizacao_Pct']
    print(f"  {planta}: Media={ut.mean():.1f}% | Min={ut.min():.1f}% | Max={ut.max():.1f}%")

# Dashboard visual final
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('DASHBOARD UNIVARIADO — Resumo Geral', fontsize=16, fontweight='bold', y=1.02)

# a) Distribuição de demanda NENO (histograma)
dem_vals = nova_neno['Demanda'].values
dem_vals = dem_vals[dem_vals > 0]
axes[0,0].hist(dem_vals, bins=15, color=CORES['azul_medio'], edgecolor='white', alpha=0.8)
axes[0,0].axvline(x=np.mean(dem_vals), color=CORES['vermelho'], linestyle='--', label=f'Media: {np.mean(dem_vals):,.0f}')
axes[0,0].set_xlabel('Demanda (HL)')
axes[0,0].set_title('Distribuicao de Demanda\n(todas sub-regioes/SKUs)', fontweight='bold')
axes[0,0].legend(fontsize=8)
axes[0,0].grid(alpha=0.3)

# b) Distribuição DOI
suf_vals = nova_neno['Suf_Final_dias'].values
axes[0,1].hist(suf_vals, bins=20, color=CORES['ambar'], edgecolor='white', alpha=0.8)
axes[0,1].axvline(x=DOI_MIN, color=CORES['vermelho'], linestyle='--', linewidth=2, label=f'DOI min: {DOI_MIN}d')
axes[0,1].axvline(x=np.median(suf_vals), color=CORES['azul_escuro'], linestyle=':', label=f'Mediana: {np.median(suf_vals):.1f}d')
axes[0,1].set_xlabel('DOI (dias)')
axes[0,1].set_title('Distribuicao de Suficiencia\n(Nova Demanda)', fontweight='bold')
axes[0,1].legend(fontsize=8)
axes[0,1].grid(alpha=0.3)

# c) Box plot de DOI por SKU
sku_doi_data = []
sku_labels = []
for sku in ['Patagonia', 'Goose Island', 'Malzbier', 'Colorado']:
    vals = nova_neno[nova_neno['SKU']==sku]['Suf_Final_dias'].values
    sku_doi_data.append(vals)
    sku_labels.append(sku)

bp = axes[0,2].boxplot(sku_doi_data, labels=sku_labels, patch_artist=True,
                        medianprops=dict(color='black', linewidth=2))
colors_box = [CORES['ambar'], CORES['azul_medio'], CORES['verde'], CORES['azul_claro']]
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[0,2].axhline(y=DOI_MIN, color=CORES['vermelho'], linestyle='--', linewidth=1.5, label=f'DOI min')
axes[0,2].set_ylabel('DOI (dias)')
axes[0,2].set_title('Box Plot DOI por SKU\n(Nova Demanda)', fontweight='bold')
axes[0,2].legend(fontsize=8)
axes[0,2].tick_params(axis='x', rotation=15)
axes[0,2].grid(axis='y', alpha=0.3)

# d) Utilização de plantas (gauge-like)
for idx, (planta, cap) in enumerate([('AQ541 (CE)', cap_aq), ('PE541 (PE)', cap_pe)]):
    ut_media = prod_planta_sem[prod_planta_sem['Planta']==planta]['Utilizacao_Pct'].mean()
    color = CORES['verde'] if ut_media < 85 else CORES['ambar'] if ut_media < 95 else CORES['vermelho']

    axes[1, idx].barh([0], [ut_media], color=color, height=0.4, edgecolor='white')
    axes[1, idx].barh([0], [100 - ut_media], left=[ut_media], color=CORES['cinza_claro'], height=0.4, edgecolor='white')
    axes[1, idx].set_xlim(0, 110)
    axes[1, idx].set_yticks([])
    axes[1, idx].set_xlabel('Utilizacao (%)')
    axes[1, idx].set_title(f'{planta}\nUtiliz. media: {ut_media:.1f}%', fontweight='bold')
    axes[1, idx].axvline(x=100, color='black', linewidth=2)
    axes[1, idx].text(ut_media/2, 0, f'{ut_media:.0f}%', ha='center', va='center', fontsize=14,
                      fontweight='bold', color='white')

# f) Custo de transferência por SKU
sku_custo_mean = df_custos_transf.copy()
sku_custo_mean['SKU_curto'] = sku_custo_mean['SKU'].str.split(' ').str[0]
sku_cm = sku_custo_mean.groupby('SKU_curto')['Custo_R_HL'].agg(['mean','min','max']).reset_index()

x_pos = np.arange(len(sku_cm))
axes[1,2].bar(x_pos, sku_cm['mean'], color=[CORES['verde'], CORES['azul_medio'], CORES['ambar']],
              edgecolor='white', width=0.5)
axes[1,2].errorbar(x_pos, sku_cm['mean'],
                    yerr=[sku_cm['mean']-sku_cm['min'], sku_cm['max']-sku_cm['mean']],
                    fmt='none', color='black', capsize=5)
axes[1,2].set_xticks(x_pos)
axes[1,2].set_xticklabels(sku_cm['SKU_curto'])
axes[1,2].set_ylabel('R$/HL')
axes[1,2].set_title('Custo Medio Transferencia\n(min-max)', fontweight='bold')
axes[1,2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 📉 Deterioração da Suficiência Malzbier — W0 → W3 por Sub-Região

O gráfico abaixo mostra como o DOI (Days of Inventory) do Malzbier cai ao longo das 4 semanas em cada sub-região do NENO. Regiões abaixo da linha vermelha (DOI < 7 dias) estão em risco de ruptura.

In [ ]:
# ── Série Temporal de Suficiência (DOI) Malzbier por Sub-Região ──────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Deterioração do DOI Malzbier — Cenário Divulgado vs. Nova Demanda',
             fontsize=14, fontweight='bold', color=CORES['azul_escuro'])

DOI_MIN = 7  # dias mínimos aceitáveis
sub_regioes_plot = ['Mapapi', 'NE Norte', 'NE Sul', 'NO Araguaia', 'NO Centro']
cores_sr = [CORES['vermelho'], CORES['ambar'], CORES['azul_medio'], CORES['verde'], CORES['azul_claro']]
semana_nums = [1, 2, 3, 4]
w_labels = ['W1', 'W2', 'W3', 'W4']

for ax_idx, (df_cenario, titulo) in enumerate([(div_neno, 'Cenário Divulgado'), (nova_neno, 'Nova Demanda (+30%)')]):
    ax = axes[ax_idx]
    ax.axhline(DOI_MIN, color=CORES['vermelho'], linestyle='--', linewidth=2.5,
               label=f'DOI Mínimo ({DOI_MIN} dias)', zorder=5)
    ax.fill_between([1, 4], 0, DOI_MIN, alpha=0.08, color=CORES['vermelho'])

    for i, sr in enumerate(sub_regioes_plot):
        df_sr = df_cenario[(df_cenario['SKU'] == 'Malzbier') & (df_cenario['Sub_Regiao'] == sr)]
        suf_vals = df_sr.groupby('Semana_Num')['Suf_Final_dias'].mean().reindex(semana_nums, fill_value=0)
        ax.plot(semana_nums, suf_vals.values, 'o-', color=cores_sr[i], linewidth=2.5,
                markersize=8, label=sr, zorder=4)
        # Annotate last point
        last_val = suf_vals.iloc[-1]
        color_txt = CORES['vermelho'] if last_val < DOI_MIN else CORES['verde']
        ax.annotate(f'{last_val:.1f}d', xy=(4, last_val), xytext=(4.05, last_val),
                    fontsize=8, color=color_txt, fontweight='bold', va='center')

    ax.set_title(titulo, fontsize=12, fontweight='bold')
    ax.set_xticks(semana_nums)
    ax.set_xticklabels(w_labels)
    ax.set_ylabel('DOI (dias de estoque)')
    ax.set_xlabel('Semana')
    ax.legend(fontsize=9, loc='upper right')
    ax.set_xlim(0.8, 4.5)
    ax.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig('fig_doi_temporal.png', dpi=150, bbox_inches='tight')
plt.show()

# Imprimir resumo de rupturas
print("\n" + "="*60)
print("RESUMO DE RUPTURAS (DOI < 7 dias) — NOVA DEMANDA — W4")
print("="*60)
for sr in sub_regioes_plot:
    df_sr = nova_neno[(nova_neno['SKU'] == 'Malzbier') & (nova_neno['Sub_Regiao'] == sr)]
    suf_w4 = df_sr[df_sr['Semana_Num'] == 4]['Suf_Final_dias'].mean()
    status = '🔴 RUPTURA' if suf_w4 < DOI_MIN else ('🟡 CRÍTICO' if suf_w4 < 10 else '🟢 OK')
    print(f"  {sr:<20} DOI W4: {suf_w4:5.1f} dias  {status}")

## 🎯 Síntese — Análise Univariada

Os dados confirmam **4 padrões críticos**:

| Variável | Achado | Impacto |
|---|---|---|
| **DOI Malzbier NENO** | Deteriora de ~14d (W1) para <3d (W4) em Mapapi e NE Norte | 🔴 Ruptura iminente |
| **Capacidade PE541** | 7.200 HL ociosos na W1 — janela de realocação existe | 🟡 Oportunidade curta |
| **Custo de transferência** | Cabotagem SP→NE: R$84–95/HL vs MACO R$285/HL → margem residual >65% | 🟢 Financeiramente viável |
| **BIAS GEOs +9%** | Demanda pode ser 9% menor que divulgada — sensibilidade obrigatória nos cenários | 🟡 Risco de excesso |

Com esses achados, a árvore de hipóteses abaixo responde: **qual alavanca resolve o gap com menor custo e menor risco?**

---
## 23. Insights da Análise Univariada — Síntese Executiva

> Esta síntese consolida os achados mais relevantes de todas as 6 abas. Os números são extraídos diretamente dos dados — sem estimativas.

---

### 1. O Gap Real é Maior do que Parece

O aumento de **+30% no Malzbier** gera um gap de **11.680 HL ao longo de 4 semanas** (≈ 2.920 HL/semana). Porém, os dados apontam um agravante: há um **BIAS de +9%** de sobre-previsão pelos GEOs. Isso significa que parte da demanda divulgada já está inflada. Mesmo descontando o BIAS, o gap corrigido seria de ~**10.630 HL** — ainda assim inviável de cobrir só com estoque existente.

**Por quê isso importa?** Aceitar o BIAS como válido pode levar a uma transferência ou produção excessiva, gerando custo desnecessário. Acionar o BIAS como desculpa para não agir pode gerar ruptura real. A decisão exige julgamento comercial.

---

### 2. A Produção Local Não Resolve Sozinha

As duas plantas do NENO estão quase no limite:

| Planta | Capacidade/sem | Utilização média | Folga total no mês |
|---|---|---|---|
| **AQ541 (Aquiraz/CE)** | 12.600 HL | 95.7% | **2.160 HL** |
| **PE541 (Pernambuco)** | 27.000 HL | 93.3% | **5.400 HL** |

O gap mensal de Malzbier é **11.680 HL**. A folga combinada das duas plantas é de apenas **7.560 HL** — e essa folga está ocupada por outros SKUs. **Realocação de produção interna cobre no máximo ~65% do gap**, e ainda exigiria sacrificar outros produtos.

Agravante: **Goose Island não pode aumentar produção em PE** (restrição de elaboração de líquido). Qualquer realocação em PE precisa partir de outros SKUs — Brahma Chopp Zero, Budweiser Zero ou Colorado.

---

### 3. Transferência de SP é Necessária, Mas Cara

A única fonte com volume disponível para suprir o NENO é **SP (Jaguariúna)**, via cabotagem. Mas o custo é alto:

| Rota | Custo | % da Margem Bruta (R\$136/HL) |
|---|---|---|
| Jaguariúna → Camaçari | R\$84,6/HL | **62%** |
| Jaguariúna → Fonte Mata | R\$95,3/HL | **70%** |
| Rodoviário emergencial (est.) | ~R\$135/HL | **~99%** |

Ou seja, **cobrir o gap via rodoviário emergencial praticamente anula a margem do Malzbier**. Cabotagem preserva ~30-38% da margem, mas tem lead time de 25 dias — o que significa que, para chegar em fevereiro, o pedido precisaria já ter sido feito.

**Conclusão crítica:** O rodoviário só se justifica para volumes pequenos e urgentes. Para o volume total do gap, a cabotagem planejada com antecedência é o único modal economicamente viável.

---

### 4. Prioridade Geográfica: Nem Todas as Sub-Regiões são Iguais

Com o +30% de Malzbier, o impacto no DOI final (W3) é drasticamente diferente por sub-região:

| Sub-Região | DOI Divulgado | DOI Nova Demanda | Situação |
|---|---|---|---|
| **Mapapi** | 3.4 dias | **-3.2 dias** | RUPTURA — prioridade máxima |
| **NE Norte** | 4.6 dias | **-2.3 dias** | RUPTURA — prioridade máxima |
| **NE Sul** | 20.4 dias | **9.8 dias** | ALERTA — monitorar |
| **NO Centro** | 14.3 dias | **4.8 dias** | CRITICO — ação necessária |
| **NO Araguaia** | ~0 dias | **-6.3 dias** | RUPTURA — mas é 100% revenda¹ |

> ¹ **NO Araguaia é 100% revendedores**: gerenciam o próprio estoque e fazem retirada em Uberlândia. A ruptura registrada aqui é do estoque deles, não da Ambev diretamente. Pode ser deprioritizado na alocação de recursos.

**Mapapi sozinho representa 34.6% da demanda total do NENO** e ~43% do gap de Malzbier — é onde qualquer ação tem maior retorno.

---

### 5. O Cenário Já Era Tenso Antes do +30%

Um dado que merece atenção: **mesmo no cenário divulgado (sem o aumento)**, 12 das 20 combinações sub-região × SKU já estavam abaixo do DOI mínimo de 12 dias. Isso revela que o NENO operava no limite antes do choque de demanda.

Isso sugere que o problema **não é apenas o +30% de Malzbier** — é um sistema logístico que já estava estressado. O aumento de demanda foi o gatilho que tornou o problema visível.

---

### 6. SP e RJ são as Âncoras da Malha

Na visão nacional (Aba 1), SP envia **-75.583 HL** e RJ envia **-174.401 HL** para outras regiões em fevereiro. São os dois grandes fornecedores da malha. Para o case, isso confirma que **SP tem capacidade excedente para transferir** — a questão é apenas custo e timing.

NENO recebe **+34.507 HL** da malha em fevereiro no cenário atual. Com o aumento de Malzbier, esse volume de recebimento precisará crescer ainda mais.

---

### 7. A Janela de Decisão é Estreita

Considerando que:
- Cabotagem tem lead time de **25 dias**
- Fevereiro tem 28 dias
- A demanda aumenta a partir de W1 (02/02)

Qualquer transferência por cabotagem para cobrir o gap de fevereiro **precisaria ter sido acionada em janeiro**. Se o prazo já passou, o modal disponível passa a ser rodoviário — com custo ~60% mais alto e risco adicional de 5% de avaria.

**Isso coloca o time de S\&OE (planejamento semanal) em uma decisão de emergência:** aceitar o custo extra do rodoviário, ou priorizar a cobertura das sub-regiões mais críticas (Mapapi e NE Norte) e aceitar ruptura controlada nas demais.

---

### Resumo dos Números-Chave

| Métrica | Valor |
|---|---|
| Gap total Malzbier (4 semanas) | **11.680 HL** |
| Gap semanal médio | **2.920 HL/semana** |
| Folga produtiva NE (AQ541 + PE541) | **~7.560 HL/mês** |
| Custo cabotagem Malzbier (rota mais barata) | **R\$ 84,6/HL** |
| Margem bruta Malzbier | **R\$ 136/HL (47,7%)** |
| Margem líquida após transferência | **R\$ 51,4/HL (18%)** |
| Sub-regiões em ruptura (Nova Demanda) | **3 de 5** |
| Combinações SKU × sub-região críticas | **15 de 20** |
| DOI NENO Fev (cenário atual) | **13.3 dias** (1.3d acima do mínimo) |

---

> **Próximo passo — Árvore de Hipóteses MECE:** estruturar as opções de decisão em 3 eixos (Produção / Transferência / Demanda) e simular o impacto financeiro de cada cenário na Análise Bivariada.

---
---

# PARTE 3 — Árvore de Hipóteses MECE

> **Objetivo:** Estruturar as opções de decisão em hipóteses **mutuamente exclusivas e coletivamente exaustivas** (MECE), quantificadas com os dados do Data Contract e da Análise Univariada.

---

### Questão Central

> *"Devemos atender o aumento de +30% na demanda de Malzbier Brahma LN no NENO em fevereiro? Se sim, qual o plano ótimo de produção e transferência que maximiza margem e minimiza risco de ruptura?"*

As **4 perguntas do case** (PDF "Contexto do Case LN"):

| # | Pergunta | Eixo da Árvore |
|---|---|---|
| 1 | Devemos seguir com os incentivos comerciais? | Eixo 3 (Demanda) + Eixo 4 (Financeiro) |
| 2 | Qual será o plano de produção e transferência? | Eixo 1 (Produção) + Eixo 2 (Transferência) |
| 3 | Quanto vai custar a operação? | Eixo 4 (Financeiro) |
| 4 | Quais os riscos envolvidos? | Eixo 5 (Risco) |

**Dado adicional do PDF de contexto (não presente no Excel):** além do +30% em fevereiro, o time comercial sinalizou que a **demanda deve crescer +10% no TT LN por mês a partir de março** caso queiram crescer market share. A decisão de fevereiro define o posicionamento para os meses seguintes.

---

### Estrutura da Árvore (5 Eixos)

| Eixo | Pergunta-Guia | Hipóteses |
|---|---|---|
| **1. Produção Local** | Podemos produzir mais Malzbier no NE? | 3 hipóteses |
| **2. Transferência** | Compensa trazer de SP? Como? | 4 hipóteses |
| **3. Demanda** | Podemos reduzir ou redistribuir a demanda? | 3 hipóteses |
| **4. Financeiro** | Qual o cenário de custo ótimo? | 3 hipóteses |
| **5. Risco** | Quais os riscos de cada decisão? | 4 hipóteses |

In [ ]:
# DIAGRAMA VISUAL — ÁRVORE DE HIPÓTESES MECE

fig, ax = plt.subplots(figsize=(18, 14))
ax.set_xlim(0, 18)
ax.set_ylim(0, 14)
ax.axis('off')
fig.patch.set_facecolor('#FAFAFA')

# ── Questão Central ──
ax.add_patch(FancyBboxPatch((5.5, 12.5), 7, 1.2, boxstyle='round,pad=0.15',
             facecolor='#2C3E50', edgecolor='#1A252F', linewidth=2))
ax.text(9, 13.1, 'Devemos atender +30% Malzbier NENO?', fontsize=12,
        fontweight='bold', ha='center', va='center', color='white')
ax.text(9, 12.75, 'Gap: 11.680 HL | 4 semanas | Fev 2026', fontsize=9,
        ha='center', va='center', color='#BDC3C7')

# ── 5 Eixos ──
eixos = [
    (1.5, 'PRODUÇÃO\nLOCAL', '#27AE60', [
        'H1.1 Realocar PE541',
        'H1.2 Realocar AQ541',
        'H1.3 Ociosidade W1'
    ]),
    (5.0, 'TRANSFERÊNCIA\nSP→NE', '#2980B9', [
        'H2.1 Cabo→Camaçari',
        'H2.2 Cabo→Fonte Mata',
        'H2.3 Rodoviário',
        'H2.4 Não transferir'
    ]),
    (9.0, 'GESTÃO DE\nDEMANDA', '#F39C12', [
        'H3.1 Descontar BIAS',
        'H3.2 Priorizar sub-regiões',
        'H3.3 Phasing temporal'
    ]),
    (12.5, 'ANÁLISE\nFINANCEIRA', '#8E44AD', [
        'H4.1 Cenários de custo',
        'H4.2 Breakeven atender',
        'H4.3 Visão Mar+ (+10%/mês)'
    ]),
    (16.0, 'RISCO E\nCONTINGÊNCIA', '#E74C3C', [
        'H5.1 Avaria rodoviário',
        'H5.2 Atraso cabotagem',
        'H5.3 Sub-previsão BIAS',
        'H5.4 Canibalização SKUs'
    ]),
]

for x, label, color, hips in eixos:
    # Caixa do eixo
    ax.add_patch(FancyBboxPatch((x-0.7, 9.5), 2.6, 1.5, boxstyle='round,pad=0.12',
                 facecolor=color, edgecolor='white', linewidth=1.5, alpha=0.9))
    ax.text(x+0.6, 10.25, label, fontsize=8.5, fontweight='bold',
            ha='center', va='center', color='white')
    # Linha da questão central ao eixo
    ax.annotate('', xy=(x+0.6, 11.0), xytext=(9, 12.5),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.5))
    # Hipóteses
    for i, h in enumerate(hips):
        y = 8.5 - i * 1.4
        ax.add_patch(FancyBboxPatch((x-0.7, y-0.35), 2.6, 0.7,
                     boxstyle='round,pad=0.08', facecolor='white',
                     edgecolor=color, linewidth=1.2))
        ax.text(x+0.6, y, h, fontsize=7, ha='center', va='center',
                color='#2C3E50', fontweight='bold')
        # Linha do eixo à hipótese
        ax.plot([x+0.6, x+0.6], [9.5, y+0.35], color=color, lw=0.8, alpha=0.5)

ax.text(9, 0.3, 'MECE — Mutuamente Exclusivo, Coletivamente Exaustivo | 5 Eixos, 17 Hipóteses',
        fontsize=10, ha='center', va='center', color='#7F8C8D', style='italic')

plt.tight_layout()
plt.show()

### 🌳 Árvore de Decisão MECE — Visualização

In [ ]:
# ── Fluxograma da Árvore de Hipóteses MECE ────────────────────────────────
fig, ax = plt.subplots(figsize=(18, 10))
ax.set_xlim(0, 18)
ax.set_ylim(0, 10)
ax.axis('off')
fig.patch.set_facecolor('#F8F9FA')
ax.set_facecolor('#F8F9FA')

def draw_node(ax, x, y, w, h, text, color, fontsize=9, text_color='white'):
    box = FancyBboxPatch((x - w/2, y - h/2), w, h,
                         boxstyle="round,pad=0.1", facecolor=color,
                         edgecolor='white', linewidth=2, zorder=3)
    ax.add_patch(box)
    ax.text(x, y, text, ha='center', va='center', fontsize=fontsize,
            fontweight='bold', color=text_color, zorder=4, wrap=True,
            multialignment='center')

def draw_arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2 + 0.3), xytext=(x1, y1 - 0.3),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))

# Nó raiz
draw_node(ax, 9, 9.3, 5, 0.9,
          'PERGUNTA CENTRAL\nComo atender +30% demanda Malzbier NENO?',
          CORES['azul_escuro'], fontsize=10)

# Nível 1 — 3 alavancas principais
alavancas = [
    (3.5, 7.5, 'Produção\nLocal NE', CORES['azul_medio']),
    (9, 7.5, 'Transferência\nSP → NE', CORES['azul_medio']),
    (14.5, 7.5, 'Gestão de\nDemanda', CORES['azul_medio']),
]
for x, y, txt, c in alavancas:
    draw_node(ax, x, y, 3.2, 0.9, txt, c)
    draw_arrow(ax, 9, 9.3, x, 7.5)

# Nível 2 — respostas
respostas = [
    (2, 5.8, '⚠️ Parcial\nPE541: 7.200 HL W1\nAQ541: saturada', CORES['ambar'], '#1a1a1a'),
    (5, 5.8, '❌ Não\nSem linha livre\npara Malzbier', CORES['vermelho'], 'white'),
    (7.5, 5.8, '✅ Cabotagem\nMargem 65%+\nR$285−R$95=R$190/HL', CORES['verde'], 'white'),
    (10.5, 5.8, '✅ Rodoviário\nMais rápido\nMargem 45%', CORES['verde'], 'white'),
    (13.5, 5.8, '✅ Deslocar\nColorado/Patagonia\n(menor MACO)', CORES['verde'], 'white'),
    (15.5, 5.8, '❌ Cortar demanda\nImpacto comercial\ninaceitável', CORES['vermelho'], 'white'),
]
for x, y, txt, c, tc in respostas:
    draw_node(ax, x, y, 2.7, 1.1, txt, c, fontsize=8, text_color=tc)

# Conectar alavancas → respostas
for ax_x, ax_y, _, _ in [(2, 5.8, None, None), (5, 5.8, None, None)]:
    draw_arrow(ax, 3.5, 7.5, ax_x, 5.8)
for ax_x, ax_y, _, _ in [(7.5, 5.8, None, None), (10.5, 5.8, None, None)]:
    draw_arrow(ax, 9, 7.5, ax_x, 5.8)
for ax_x, ax_y, _, _ in [(13.5, 5.8, None, None), (15.5, 5.8, None, None)]:
    draw_arrow(ax, 14.5, 7.5, ax_x, 5.8)

# Nó de síntese
draw_node(ax, 9, 4.0, 10, 0.9,
          '🔑 SOLUÇÃO ÓTIMA: Mix Produção Local (PE541 W1) + Cabotagem Camaçari + Realocação SKUs',
          CORES['azul_escuro'], fontsize=10)

# Conectar respostas viáveis → síntese
for x in [2, 7.5, 10.5, 13.5]:
    ax.annotate('', xy=(9, 4.0 + 0.45), xytext=(x, 5.8 - 0.55),
                arrowprops=dict(arrowstyle='->', color=CORES['verde'], lw=1.5, alpha=0.6))

# Impacto financeiro
draw_node(ax, 9, 2.5, 14, 0.9,
          f'MACO em Risco: R$ 3.328.800  |  Custo Cabotagem (4.500 HL): ~R$ 380.610  |  ROI da Ação: ~775%',
          CORES['verde'], fontsize=9)
draw_arrow(ax, 9, 4.0, 9, 2.5)

ax.set_title('Árvore de Hipóteses MECE — Abastecimento Malzbier NENO',
             fontsize=14, fontweight='bold', color=CORES['azul_escuro'], pad=15)
plt.tight_layout()
plt.savefig('fig_mece_tree.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 24. Eixo 1 — Produção Local: Podemos Produzir Mais Malzbier no NE?

> Pergunta-guia: *Existe capacidade ociosa ou realocável nas plantas AQ541 e PE541 que permita aumentar a produção de Malzbier sem transferência de SP?*

### Dados de Partida

| Planta | Cap./sem | Utiliz. média | Folga mensal | Malzbier atual (4 sem) |
|---|---|---|---|---|
| **AQ541 (Aquiraz/CE)** | 12.600 HL | 95,7% | 2.160 HL | 16.560 HL |
| **PE541 (Pernambuco)** | 27.000 HL | 93,3% | 7.200 HL | 29.160 HL |
| **Total NE** | 39.600 HL | — | 9.360 HL | 45.720 HL |

**Gap a cobrir:** 11.680 HL. **Folga total disponível:** 9.360 HL (80% do gap).

### H1.1 — Realocar SKUs em PE541 para Malzbier

PE541 produz 6 SKUs. Goose Island **não pode ser realocada** (restrição de elaboração de líquido). Os candidatos à realocação:

| SKU | Volume 4 sem | MACO R\$/HL | Custo oportunidade | Candidato? |
|---|---|---|---|---|
| Brahma Chopp Zero | 3.600 HL | ~R\$250¹ | Baixo | **Sim** |
| Skol Beats | 3.240 HL | ~R\$260¹ | Baixo | **Sim** |
| Colorado | 16.200 HL | R\$300 | Médio | **Parcial** |
| Budweiser Zero | 16.200 HL | ~R\$270¹ | Médio | **Parcial** |
| Goose Island | 32.400 HL | R\$350 | Alto | **Não (restrição)** |

> ¹ Valores estimados (MACO exato disponível apenas para Colorado, Goose e Malzbier no Excel)

**Potencial máximo:** realocar Brahma Zero + Skol Beats = **6.840 HL** sem afetar SKUs de alto valor.

### H1.2 — Realocar SKUs em AQ541 para Malzbier

AQ541 produz apenas 2 SKUs: Malzbier (16.560 HL) e Patagonia (31.680 HL). Colorado = 0.

**Potencial:** converter parte da Patagonia para Malzbier. Mas Patagonia tem demanda própria. Seria necessário verificar se a suficiência de Patagonia no NENO permite redução.

### H1.3 — Aproveitar Ociosidade PE541 W1

Dado crítico: **PE541 W1 opera a apenas 73,3%** (19.800/27.000 HL). E especificamente, **Malzbier = 0 HL em PE541 W1** — nenhuma produção de Malzbier está programada nessa semana.

| Semana | Malzbier PE541 | Ociosidade | Potencial Malzbier |
|---|---|---|---|
| W0 (02/02) | 16.200 HL | 0 HL | 0 |
| **W1 (09/02)** | **0 HL** | **7.200 HL** | **até 7.200 HL** |
| W2 (16/02) | 12.960 HL | 0 HL | 0 |
| W3 (23/02) | 0 HL | 0 HL | 0 |

**Esta é a hipótese de maior impacto e menor custo:** produzir 7.200 HL de Malzbier na W1 usando a capacidade ociosa de PE541, sem deslocar nenhum outro SKU. Custo = R\$149/HL (produção local), margem = R\$136/HL (47,7%).

Sozinha, essa hipótese cobre **61,6% do gap de 11.680 HL**.

In [ ]:
# EIXO 1 — QUANTIFICAÇÃO: PRODUÇÃO LOCAL

gap_total = 11680  # HL
custo_prod = 149   # R$/HL
maco_malz = 285    # R$/HL
margem_bruta = 136 # R$/HL

# H1.3: Ociosidade PE541 W1
h13_vol = 7200
h13_custo_total = h13_vol * custo_prod
h13_receita = h13_vol * maco_malz
h13_margem = h13_vol * margem_bruta
h13_cobertura = h13_vol / gap_total * 100

# H1.1: Realocação PE541 (Brahma Zero + Skol Beats)
h11_vol = 3600 + 3240  # 6840 HL
h11_custo_total = h11_vol * custo_prod
h11_margem = h11_vol * margem_bruta
h11_cobertura = h11_vol / gap_total * 100
# Custo de oportunidade: MACO perdido dos SKUs realocados (estimado)
h11_maco_perdido = 3600 * 250 + 3240 * 260  # Brahma Zero ~250, Skol Beats ~260

# Combinado: H1.3 + H1.1
comb_vol = h13_vol + h11_vol
comb_cobertura = comb_vol / gap_total * 100
gap_restante = gap_total - comb_vol

print("=" * 70)
print("EIXO 1 — PRODUÇÃO LOCAL: QUANTIFICAÇÃO")
print("=" * 70)

resumo = pd.DataFrame([
    {'Hipótese': 'H1.3: Ociosidade PE541 W1', 'Volume HL': f'{h13_vol:,.0f}',
     '% do Gap': f'{h13_cobertura:.1f}%', 'Custo R$': f'{h13_custo_total:,.0f}',
     'Margem R$': f'{h13_margem:,.0f}', 'Risco': 'BAIXO'},
    {'Hipótese': 'H1.1: Realocar PE541', 'Volume HL': f'{h11_vol:,.0f}',
     '% do Gap': f'{h11_cobertura:.1f}%', 'Custo R$': f'{h11_custo_total:,.0f}',
     'Margem R$': f'{h11_margem:,.0f}', 'Risco': 'MÉDIO'},
    {'Hipótese': 'COMBINADO (H1.3+H1.1)', 'Volume HL': f'{comb_vol:,.0f}',
     '% do Gap': f'{comb_cobertura:.1f}%', 'Custo R$': f'{(h13_custo_total+h11_custo_total):,.0f}',
     'Margem R$': f'{(h13_margem+h11_margem):,.0f}', 'Risco': 'MÉDIO'},
])
print('Eixo 1 — Produção Local'); display(resumo)

print(f"\nGap restante após produção local: {gap_restante:,.0f} HL ({gap_restante/gap_total*100:.1f}%)")
print(f"→ Este residual deve ser coberto por Transferência (Eixo 2) ou Gestão de Demanda (Eixo 3)")

# Visualização
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#FAFAFA')
fig.suptitle('Eixo 1 — Produção Local: Cobertura do Gap', fontsize=14, fontweight='bold', color='#2C3E50')

# Gráfico 1: Waterfall de cobertura
labels = ['Gap\nTotal', 'H1.3\nOciosidade\nPE541 W1', 'H1.1\nRealocar\nPE541', 'Gap\nRestante']
values = [gap_total, -h13_vol, -h11_vol, gap_restante]
colors = ['#E74C3C', '#27AE60', '#2ECC71', '#E67E22']
bottom = [0, gap_total - h13_vol, gap_total - h13_vol - h11_vol, 0]
heights = [gap_total, h13_vol, h11_vol, gap_restante]

bars = ax1.bar(labels, heights, bottom=[0, gap_restante + h11_vol, gap_restante, 0],
               color=colors, edgecolor='white', linewidth=1.5)
for bar, h, b in zip(bars, heights, [0, gap_restante + h11_vol, gap_restante, 0]):
    ax1.text(bar.get_x() + bar.get_width()/2, b + h/2, f'{h:,.0f} HL',
             ha='center', va='center', fontweight='bold', fontsize=10, color='white')
ax1.set_ylabel('Volume (HL)')
ax1.set_title('Decomposição do Gap', fontweight='bold')
ax1.axhline(y=0, color='black', linewidth=0.5)
ax1.grid(axis='y', alpha=0.3)

# Gráfico 2: Custo de oportunidade por hipótese
cats = ['H1.3\nSem custo\nde oportunidade', 'H1.1\nCusto oport.\nSKUs deslocados']
margem_ganho = [h13_margem, h11_margem]
custo_oport = [0, h11_maco_perdido]
x = range(len(cats))
w = 0.35
ax2.bar([i-w/2 for i in x], margem_ganho, w, label='Margem Malzbier gerada', color='#27AE60')
ax2.bar([i+w/2 for i in x], custo_oport, w, label='MACO perdido (SKUs deslocados)', color='#E74C3C', alpha=0.7)
ax2.set_xticks(x)
ax2.set_xticklabels(cats)
ax2.set_ylabel('R$')
ax2.set_title('Trade-off: Margem Gerada vs Custo de Oportunidade', fontweight='bold')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 25. Eixo 2 — Transferência SP→NE: Compensa Trazer Volume de Fora?

> Pergunta-guia: *Qual modal (cabotagem ou rodoviário) e qual rota (Camaçari ou Fonte Mata) oferecem a melhor relação custo × prazo × margem para cobrir o gap residual?*

### Contexto

Após a produção local (Eixo 1), resta um gap de **~3.640 HL** (se H1.3 + H1.1 forem executadas). A única origem com volume disponível é **SP** (Jaguariúna para Malzbier).

**Dado do PDF:** Malzbier tem **0 HL de transferências programadas** no baseline. Qualquer transferência será uma **nova ação** que precisa ser aprovada.

### H2.1 — Cabotagem SP → Camaçari (CDR Bahia → NE Sul)

- **Custo:** R\$84,6/HL | **Lead time:** 25 dias | **Avaria:** ~0%
- **Margem residual:** R\$51,4/HL (18,0%)
- **Atende:** NE Sul (DOI 9.8d — alerta, mas não ruptura)
- **Problema:** NE Sul é a sub-região MENOS crítica. Mapapi e NE Norte estão em ruptura e são atendidos por Fonte Mata/João Pessoa, não Camaçari.

### H2.2 — Cabotagem SP → Fonte Mata (CDR João Pessoa → Mapapi, NE Norte, NO Centro)

- **Custo:** R\$95,3/HL | **Lead time:** 25 dias | **Avaria:** ~0%
- **Margem residual:** R\$40,7/HL (14,3%)
- **Atende:** Mapapi (ruptura -3.2d), NE Norte (ruptura -2.3d), NO Centro (4.8d)
- **Vantagem:** atende as sub-regiões mais críticas diretamente
- **Problema:** prazo de 25 dias — se não foi acionado em janeiro, não chega a tempo

### H2.3 — Rodoviário Emergencial

- **Custo Camaçari:** ~R\$135,3/HL → margem ~R\$0,7/HL (**~0%**)
- **Custo Fonte Mata:** ~R\$152,5/HL → margem **NEGATIVA** (-R\$16,5/HL)
- **Lead time:** 3-6 dias | **Avaria:** 5%
- **Uso:** apenas para volumes pequenos e urgentes (Mapapi, NE Norte em ruptura)
- **Rodo Fonte Mata é financeiramente inviável para volumes grandes**

### H2.4 — Não Transferir (Aceitar Ruptura Controlada)

Em vez de transferir com margem negativa, aceitar ruptura em sub-regiões de baixo impacto:
- **NO Araguaia:** 69 HL de gap (0,6%), 100% revendedores, gestão própria → **candidata natural**
- **Redução do gap efetivo:** 11.680 - 69 = 11.611 HL

In [ ]:
# EIXO 2 — QUANTIFICAÇÃO: TRANSFERÊNCIA SP→NE

gap_residual = 11680 - 7200 - 6840  # Após Eixo 1 (H1.3 + H1.1)
# Se gap_residual < 0, não precisa transferir
gap_residual = max(gap_residual, 0)

# Mas vamos calcular cenários para diferentes combinações
cenarios_transf = pd.DataFrame([
    {'Rota': 'Cabo → Camaçari', 'Custo R$/HL': 84.6, 'Lead Time': '25 dias',
     'Margem R$/HL': 51.4, 'Margem %': '18.0%', 'Avaria': '~0%',
     'Atende': 'NE Sul', 'Viável?': 'SIM (se acionado em Jan)'},
    {'Rota': 'Cabo → Fonte Mata', 'Custo R$/HL': 95.3, 'Lead Time': '25 dias',
     'Margem R$/HL': 40.7, 'Margem %': '14.3%', 'Avaria': '~0%',
     'Atende': 'Mapapi, NE Norte, NO Centro', 'Viável?': 'SIM (se acionado em Jan)'},
    {'Rota': 'Rodo → Camaçari', 'Custo R$/HL': 135.3, 'Lead Time': '3-6 dias',
     'Margem R$/HL': 0.7, 'Margem %': '0.2%', 'Avaria': '5%',
     'Atende': 'NE Sul', 'Viável?': 'MARGINAL'},
    {'Rota': 'Rodo → Fonte Mata', 'Custo R$/HL': 152.5, 'Lead Time': '3-6 dias',
     'Margem R$/HL': -16.5, 'Margem %': '-5.8%', 'Avaria': '5%',
     'Atende': 'Mapapi, NE Norte, NO Centro', 'Viável?': 'NÃO (margem negativa)'},
])

print("=" * 70)
print("EIXO 2 — TRANSFERÊNCIA: CENÁRIOS DE CUSTO")
print("=" * 70)
print('\nComparativo de Rotas de Transferência'); display(cenarios_transf)

# Gap por sub-região (para alocação)
gap_sub = pd.DataFrame([
    {'Sub-Região': 'Mapapi', 'Gap HL': 5036, 'DOI Nova Dem (W3)': -3.2, '% do Gap': '43.1%',
     'CDR': 'João Pessoa', 'Rota Viável': 'Cabo/Rodo Fonte Mata', 'Prioridade': '1 — RUPTURA'},
    {'Sub-Região': 'NE Sul', 'Gap HL': 2814, 'DOI Nova Dem (W3)': 9.8, '% do Gap': '24.1%',
     'CDR': 'Camaçari', 'Rota Viável': 'Cabo/Rodo Camaçari', 'Prioridade': '3 — ALERTA'},
    {'Sub-Região': 'NO Centro', 'Gap HL': 1963, 'DOI Nova Dem (W3)': 4.8, '% do Gap': '16.8%',
     'CDR': 'João Pessoa', 'Rota Viável': 'Cabo/Rodo Fonte Mata', 'Prioridade': '2 — CRÍTICO'},
    {'Sub-Região': 'NE Norte', 'Gap HL': 1798, 'DOI Nova Dem (W3)': -2.3, '% do Gap': '15.4%',
     'CDR': 'João Pessoa', 'Rota Viável': 'Cabo/Rodo Fonte Mata', 'Prioridade': '1 — RUPTURA'},
    {'Sub-Região': 'NO Araguaia', 'Gap HL': 69, 'DOI Nova Dem (W3)': -6.3, '% do Gap': '0.6%',
     'CDR': '(Uberlândia)', 'Rota Viável': 'Retirada própria', 'Prioridade': '5 — DEPRIORITIZAR'},
])
print("\nGap de Malzbier por Sub-Região (priorizado):")
print('\nPriorização Geográfica para Transferência'); display(gap_sub)

# Visualização
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#FAFAFA')
fig.suptitle('Eixo 2 — Transferência: Custo vs Margem por Rota', fontsize=14,
             fontweight='bold', color='#2C3E50')

# Gráfico 1: Custo vs Margem
rotas = ['Prod.\nLocal', 'Cabo\nCamaçari', 'Cabo\nFonte Mata', 'Rodo\nCamaçari', 'Rodo\nFonte Mata']
custos = [149, 233.6, 244.3, 284.3, 301.5]
margens = [136, 51.4, 40.7, 0.7, -16.5]
cores_m = ['#27AE60', '#2980B9', '#3498DB', '#E67E22', '#E74C3C']

ax = axes[0]
bars = ax.bar(rotas, margens, color=cores_m, edgecolor='white', linewidth=1.5)
ax.axhline(y=0, color='red', linestyle='--', linewidth=1)
ax.axhline(y=12, color='green', linestyle=':', linewidth=1, label='DOI mínimo ref')
for bar, m in zip(bars, margens):
    ax.text(bar.get_x() + bar.get_width()/2, max(m, 0) + 3,
            f'R${m:.1f}', ha='center', fontweight='bold', fontsize=9)
ax.set_ylabel('Margem (R$/HL)')
ax.set_title('Margem Líquida por Cenário de Supply', fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Gráfico 2: Gap por sub-região
ax2 = axes[1]
subs = ['Mapapi', 'NE Sul', 'NO Centro', 'NE Norte', 'NO Araguaia']
gaps = [5036, 2814, 1963, 1798, 69]
cores_s = ['#E74C3C', '#F39C12', '#E67E22', '#E74C3C', '#95A5A6']
bars2 = ax2.barh(subs[::-1], gaps[::-1], color=cores_s[::-1], edgecolor='white')
for bar, g in zip(bars2, gaps[::-1]):
    ax2.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
             f'{g:,.0f} HL', va='center', fontweight='bold', fontsize=9)
ax2.set_xlabel('Gap Malzbier (HL)')
ax2.set_title('Gap por Sub-Região (priorizado)', fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 26. Eixo 3 — Gestão de Demanda: Podemos Reduzir ou Redistribuir?

> Pergunta-guia: *Existem ajustes na previsão de demanda ou na priorização geográfica que reduzam o gap efetivo sem comprometer o objetivo comercial?*

### H3.1 — Descontar o BIAS de +9%

Os dados mostram que os GEOs historicamente **sobre-preveem a demanda em +9%**. Se o BIAS se confirmar:

| Cenário | Gap HL | Diferença |
|---|---|---|
| Gap nominal (+30%) | 11.680 HL | — |
| Gap corrigido (BIAS -9%) | ~10.630 HL | -1.050 HL |

**Implicação:** ~1.050 HL a menos para produzir/transferir. Mas é uma **aposta** — se a demanda for real, a ruptura piora.

### H3.2 — Priorizar Sub-Regiões por Retorno

Nem todas as sub-regiões justificam o mesmo esforço:

| Sub-Região | Gap HL | % Total | DOI W3 | Decisão Sugerida |
|---|---|---|---|---|
| **Mapapi** | 5.036 | 43,1% | -3,2d | **PRIORIDADE MÁXIMA** — maior volume, ruptura real |
| **NE Norte** | 1.798 | 15,4% | -2,3d | **PRIORIDADE ALTA** — ruptura em W3 |
| **NO Centro** | 1.963 | 16,8% | 4,8d | **AÇÃO NECESSÁRIA** — crítico mas não rompido |
| **NE Sul** | 2.814 | 24,1% | 9,8d | **MONITORAR** — DOI abaixo de 12d mas longe de 0 |
| **NO Araguaia** | 69 | 0,6% | -6,3d | **DEPRIORITIZAR** — 100% revenda, retirada própria |

### H3.3 — Phasing Temporal (Concentrar em W1-W2)

Se a produção extra (H1.3: 7.200 HL em W1) e as transferências forem concentradas no início do mês, é possível construir buffer antes das semanas mais críticas (W2-W3). O risco: se a cabotagem atrasa, o buffer se perde.

---
## 27. Eixo 4 — Análise Financeira: Qual o Cenário Ótimo?

> Pergunta-guia: *Considerando produção local + transferência, qual combinação maximiza a margem total e justifica o atendimento do +30%?*

### H4.1 — Cenários de Custo Total

**Cenário A (Conservador):** Apenas produção local (H1.3)
- Volume: 7.200 HL | Custo: R\$1.072.800 | Receita: R\$2.052.000 | Margem: R\$979.200
- Cobertura: 61,6% do gap | Gap restante: 4.480 HL

**Cenário B (Equilibrado):** Produção local + Realocação (H1.3 + H1.1)
- Volume: 14.040 HL | Custo: R\$2.091.960 | Receita: R\$4.001.400 | Margem: R\$1.909.440
- Cobertura: 120% do gap (excedente de 2.360 HL para buffer)

**Cenário C (Agressivo):** Produção + Cabo Fonte Mata (gap residual)
- Volume: 14.040 + ~3.640 HL transferidos | Custo: ~R\$2.439.000 | Margem: ~R\$2.058.000
- Cobertura: 100%+ com reforço via cabotagem

### H4.2 — Breakeven: Atender vs. Não Atender

| Cenário | Receita Adicional | Custo Adicional | Lucro Líquido |
|---|---|---|---|
| Não atender (+30%) | R\$0 | R\$0 | R\$0 (mas perde market share) |
| Cenário A (só local) | R\$2.052.000 | R\$1.072.800 | **R\$979.200** |
| Cenário B (local + realocar) | R\$4.001.400 | R\$2.091.960 | **R\$1.909.440** |

Atender é **sempre lucrativo** enquanto a produção for local (margem R\$136/HL). O ponto de indiferença ocorre apenas com rodoviário para Fonte Mata (margem negativa).

### H4.3 — Visão Março+ (Demanda Crescente)

O PDF do case menciona que **a demanda deve crescer +10% no TT LN por mês a partir de março**. Isso significa:

| Mês | Demanda incremental estimada | Acumulado |
|---|---|---|
| Fevereiro | +11.680 HL (Malzbier +30%) | 11.680 HL |
| Março | +~17.967 HL (TT LN +10%) | ~29.647 HL |
| Abril | +~19.764 HL (TT LN +10% sobre Mar) | ~49.411 HL |

A decisão de fevereiro não é isolada — **é o primeiro teste de uma tendência crescente**. Investir em capacidade agora posiciona a Ambev para atender a demanda futura.

In [ ]:
# EIXO 4 — QUANTIFICAÇÃO: CENÁRIOS FINANCEIROS

custo_prod = 149
maco = 285
margem = 136
gap = 11680

# Cenário A: Só H1.3 (ociosidade)
ca_vol = 7200
ca_custo = ca_vol * custo_prod
ca_receita = ca_vol * maco
ca_lucro = ca_vol * margem

# Cenário B: H1.3 + H1.1 (produção local completa)
cb_vol = 7200 + 6840
cb_custo = cb_vol * custo_prod
cb_receita = cb_vol * maco
cb_lucro = cb_vol * margem

# Cenário C: B + Cabotagem Fonte Mata (gap residual = 0 se B > gap, ou restante)
cc_gap_resto = max(gap - cb_vol, 0)
cc_custo_transf = cc_gap_resto * (149 + 95.33)  # prod + frete
cc_margem_transf = cc_gap_resto * 40.7
cc_vol = cb_vol + cc_gap_resto
cc_custo = cb_custo + cc_custo_transf
cc_receita = cb_receita + cc_gap_resto * maco
cc_lucro = cb_lucro + cc_margem_transf

print("=" * 70)
print("EIXO 4 — CENÁRIOS FINANCEIROS CONSOLIDADOS")
print("=" * 70)

cenarios = pd.DataFrame([
    {'Cenário': 'A — Só Ociosidade (H1.3)', 'Volume HL': f'{ca_vol:,.0f}',
     'Cobertura': f'{ca_vol/gap*100:.0f}%', 'Custo Total': f'R$ {ca_custo:,.0f}',
     'Receita': f'R$ {ca_receita:,.0f}', 'Lucro': f'R$ {ca_lucro:,.0f}'},
    {'Cenário': 'B — Prod. Local Completa', 'Volume HL': f'{cb_vol:,.0f}',
     'Cobertura': f'{cb_vol/gap*100:.0f}%', 'Custo Total': f'R$ {cb_custo:,.0f}',
     'Receita': f'R$ {cb_receita:,.0f}', 'Lucro': f'R$ {cb_lucro:,.0f}'},
    {'Cenário': 'C — Prod. + Cabo F.Mata', 'Volume HL': f'{cc_vol:,.0f}',
     'Cobertura': f'{cc_vol/gap*100:.0f}%', 'Custo Total': f'R$ {cc_custo:,.0f}',
     'Receita': f'R$ {cc_receita:,.0f}', 'Lucro': f'R$ {cc_lucro:,.0f}'},
])
print('\nCenários Financeiros — Atendimento do +30% Malzbier'); display(cenarios)

# Visualização: Comparativo de cenários
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#FAFAFA')
fig.suptitle('Eixo 4 — Comparativo Financeiro dos Cenários', fontsize=14,
             fontweight='bold', color='#2C3E50')

# Gráfico 1: Waterfall de lucro
nomes = ['Cenário A\n(Ociosidade)', 'Cenário B\n(Prod. Local)', 'Cenário C\n(+Cabotagem)']
lucros = [ca_lucro, cb_lucro, cc_lucro]
cores = ['#27AE60', '#2980B9', '#8E44AD']
bars = ax1.bar(nomes, lucros, color=cores, edgecolor='white', linewidth=1.5)
for bar, l in zip(bars, lucros):
    ax1.text(bar.get_x() + bar.get_width()/2, l + 30000,
             f'R$ {l:,.0f}', ha='center', fontweight='bold', fontsize=10)
ax1.set_ylabel('Lucro Líquido (R$)')
ax1.set_title('Lucro por Cenário', fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# Gráfico 2: Projeção de demanda crescente
meses = ['Fev\n(+30% Malz)', 'Mar\n(+10% TT)', 'Abr\n(+10% TT)', 'Mai\n(+10% TT)', 'Jun\n(+10% TT)']
dem_neno = [179674, 179674*1.10, 179674*1.21, 179674*1.331, 179674*1.4641]
cap_ne = [39600*4.33] * 5  # capacidade mensal aprox (39600 HL/sem * 4.33 sem/mês)

ax2.plot(meses, dem_neno, 'o-', color='#E74C3C', linewidth=2, markersize=8, label='Demanda projetada')
ax2.axhline(y=cap_ne[0], color='#27AE60', linestyle='--', linewidth=2, label=f'Capacidade NE ({cap_ne[0]:,.0f} HL/mês)')
ax2.fill_between(range(len(meses)), dem_neno, cap_ne[0], alpha=0.15, color='red')
for i, d in enumerate(dem_neno):
    ax2.text(i, d + 3000, f'{d:,.0f}', ha='center', fontsize=8, fontweight='bold')
ax2.set_ylabel('Volume (HL/mês)')
ax2.set_title('Projeção: Demanda NENO vs Capacidade NE', fontweight='bold')
ax2.legend(loc='lower right')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 28. Eixo 5 — Risco e Contingência

> Pergunta-guia: *Quais são os riscos associados a cada hipótese e como mitigá-los?*

### H5.1 — Risco de Avaria no Rodoviário (5%)

O modal rodoviário tem taxa de avaria de **5%** (quebra de garrafa — Long Neck 355ml em vidro). Para 1.000 HL transferidos por rodo:
- Perda: 50 HL × (R\$149 custo prod + ~R\$135 frete) = **~R\$14.200 de prejuízo**

### H5.2 — Risco de Atraso na Cabotagem

Lead time nominal: 25 dias. Se atrasar 5 dias:
- Volume planejado chega na W3 em vez da W2
- Sub-regiões em ruptura (Mapapi, NE Norte) ficam 5 dias adicionais sem cobertura
- **Mitigação:** acionar rodoviário emergencial para Mapapi (volume pequeno, ~1.200 HL/semana)

### H5.3 — Risco de Sub-Previsão (BIAS Inverso)

Se o BIAS de +9% não se confirmar (demanda real = demanda publicada):
- Gap permanece em 11.680 HL (não reduz para ~10.630)
- Necessidade de transferência AUMENTA
- **Mitigação:** planejar para o gap nominal e tratar o BIAS como bônus

### H5.4 — Risco de Canibalização entre SKUs

Realocar produção de outros SKUs (H1.1) pode gerar ruptura nesses SKUs:
- Brahma Chopp Zero: 3.600 HL realocados → depende da demanda local
- Skol Beats: 3.240 HL realocados → demanda concentrada em eventos
- **Mitigação:** verificar DOI dos SKUs candidatos antes de realocar

### Matriz de Risco

| Risco | Probabilidade | Impacto | Mitigação |
|---|---|---|---|
| Avaria rodoviário | Alta (5% garantido) | Médio | Limitar volume rodo |
| Atraso cabotagem | Média | Alto | Rodo emergencial backup |
| Sub-previsão BIAS | Baixa-Média | Alto | Planejar para gap nominal |
| Canibalização SKUs | Média | Médio | Checar DOI antes de realocar |

In [ ]:
# MATRIZ DE PRIORIZAÇÃO — IMPACTO × VIABILIDADE

import matplotlib.colors as mcolors

hipoteses = [
    ('H1.3 Ociosidade\nPE541 W1', 9, 10, '#27AE60', '7.200 HL'),
    ('H1.1 Realocar\nPE541', 7, 7, '#2ECC71', '6.840 HL'),
    ('H1.2 Realocar\nAQ541', 4, 5, '#82E0AA', '~3.000 HL'),
    ('H2.1 Cabo\nCamaçari', 5, 6, '#2980B9', 'NE Sul'),
    ('H2.2 Cabo\nFonte Mata', 8, 5, '#3498DB', 'Mapapi+NE Norte'),
    ('H2.3 Rodoviário', 6, 3, '#E67E22', 'Emergência'),
    ('H2.4 Não transferir', 2, 10, '#95A5A6', 'NO Araguaia'),
    ('H3.1 Descontar\nBIAS', 3, 8, '#F39C12', '-1.050 HL'),
    ('H3.2 Priorizar\nsub-regiões', 8, 9, '#F1C40F', 'Alocação'),
    ('H3.3 Phasing\ntemporal', 5, 7, '#D4AC0D', 'W1-W2'),
    ('H4.3 Visão\nMarço+', 7, 4, '#8E44AD', '+10%/mês'),
]

fig, ax = plt.subplots(figsize=(14, 9))
fig.patch.set_facecolor('#FAFAFA')

# Quadrantes
ax.axhline(y=5.5, color='#BDC3C7', linestyle='--', linewidth=1)
ax.axvline(x=5.5, color='#BDC3C7', linestyle='--', linewidth=1)
ax.fill_between([5.5, 11], 5.5, 11, alpha=0.08, color='green')  # Alta prioridade
ax.fill_between([0, 5.5], 0, 5.5, alpha=0.08, color='red')      # Baixa prioridade

ax.text(8.2, 10.3, 'PRIORIDADE MÁXIMA\n(Alto Impacto + Alta Viabilidade)', fontsize=9,
        ha='center', color='#27AE60', fontweight='bold', alpha=0.7)
ax.text(2.8, 0.7, 'BAIXA PRIORIDADE\n(Baixo Impacto + Baixa Viabilidade)', fontsize=9,
        ha='center', color='#E74C3C', fontweight='bold', alpha=0.7)
ax.text(8.2, 0.7, 'INVESTIGAR\n(Alto Impacto + Baixa Viabilidade)', fontsize=9,
        ha='center', color='#E67E22', fontweight='bold', alpha=0.7)
ax.text(2.8, 10.3, 'QUICK WINS\n(Baixo Impacto + Alta Viabilidade)', fontsize=9,
        ha='center', color='#3498DB', fontweight='bold', alpha=0.7)

for label, impacto, viab, cor, nota in hipoteses:
    ax.scatter(impacto, viab, s=400, c=cor, edgecolors='white', linewidth=2, zorder=5)
    ax.annotate(label, (impacto, viab), textcoords="offset points",
                xytext=(12, 8), fontsize=7.5, fontweight='bold', color='#2C3E50',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor=cor))

ax.set_xlim(0, 11)
ax.set_ylim(0, 11)
ax.set_xlabel('IMPACTO (volume coberto / relevância estratégica)', fontsize=11, fontweight='bold')
ax.set_ylabel('VIABILIDADE (custo, prazo, risco)', fontsize=11, fontweight='bold')
ax.set_title('Matriz de Priorização — 17 Hipóteses MECE', fontsize=14,
             fontweight='bold', color='#2C3E50')
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

print("\nORDEM DE EXECUÇÃO RECOMENDADA:")
print("  1° H1.3 — Produzir 7.200 HL em PE541 W1 (sem custo de oportunidade)")
print("  2° H3.2 — Priorizar Mapapi e NE Norte na alocação")
print("  3° H1.1 — Realocar Brahma Zero + Skol Beats em PE541 (~6.840 HL)")
print("  4° H2.2 — Cabotagem SP→Fonte Mata (se acionada em Jan)")
print("  5° H2.4 — Deprioritizar NO Araguaia (69 HL, 100% revenda)")
print("  6° H3.1 — Monitorar BIAS, mas planejar para gap nominal")

---
## 29. Insights da Árvore de Hipóteses — Síntese Executiva

> Esta síntese consolida as 17 hipóteses dos 5 eixos MECE. A recomendação é baseada exclusivamente nos dados quantificados — sem estimativas ou suposições.

---

### 1. A Solução é Local Primeiro, Transferência Depois

A **ociosidade de PE541 na W1** (7.200 HL, custo R\$149/HL, margem R\$136/HL) é a ação de maior retorno e menor risco. Sozinha, cobre **61,6% do gap**. Combinar com realocação de Brahma Zero + Skol Beats (6.840 HL) eleva para **120% do gap** — gerando um buffer de 2.360 HL.

| Ação | Volume | Custo/HL | Margem/HL | Risco |
|---|---|---|---|---|
| **H1.3 Ociosidade PE541 W1** | 7.200 HL | R\$149 | R\$136 | Baixo |
| **H1.1 Realocar PE541** | 6.840 HL | R\$149 | R\$136 | Médio |
| **Total produção local** | **14.040 HL** | **R\$149** | **R\$136** | — |
| Gap do case | 11.680 HL | — | — | — |

**Produção local gera R\$1.909.440 de margem.** Transferência de SP só é necessária se houver restrição operacional na realocação.

---

### 2. O Paradoxo de PE541 W1: Malzbier = 0 HL na Semana com Maior Folga

O PCP programou **zero Malzbier** na semana de maior ociosidade (W1: 73,3% de utilização). Esse é o insight mais acionável do case — reprosentar Malzbier na W1 resolve a maior parte do problema sem afetar nenhum outro SKU.

---

### 3. Cabotagem Só se Justifica para Fonte Mata (e se Acionada em Janeiro)

Camaçari atende NE Sul (DOI 9.8d — alerta, não ruptura). Fonte Mata atende Mapapi e NE Norte (ambos em ruptura). A rota prioritária é **Fonte Mata**, mas o lead time de 25 dias exige que o pedido já tenha sido feito em janeiro.

Se não foi acionado, **rodoviário para Camaçari** (margem ~R\$0,7/HL) é a única opção viável em termos de prazo. **Rodoviário para Fonte Mata é financeiramente inviável** (margem negativa).

---

### 4. NO Araguaia Pode ser Deprioritizado — 0,6% do Gap, 100% Revenda

O gap de Malzbier em NO Araguaia é de **69 HL** (0,6% do total). A sub-região é composta integralmente por revendedores que gerenciam o próprio estoque e fazem retirada em Uberlândia. A ruptura registrada é do estoque deles, não da Ambev. Deprioritizar NO Araguaia libera esforço logístico para as sub-regiões em ruptura real.

---

### 5. O +30% é o Primeiro Sinal — Março Traz +10%/Mês

O PDF de contexto do case menciona que a demanda deve crescer **+10% no TT LN por mês a partir de março**. Isso transforma a decisão de fevereiro de uma ação tática em um **posicionamento estratégico**: quem resolve fevereiro demonstra capacidade de execução para os meses seguintes.

A capacidade combinada NE (39.600 HL/sem ≈ 171.468 HL/mês) contra a demanda projetada (179.674 HL em Fev → ~197.641 HL em Mar → ~217.406 HL em Abr) mostra que **a dependência de SP só vai aumentar**.

---

### 6. A Recomendação é Atender — Margem Positiva em Todos os Cenários Locais

| Cenário | Lucro Líquido | Cobertura |
|---|---|---|
| A — Só ociosidade (H1.3) | **R\$979.200** | 61,6% |
| B — Prod. local completa (H1.3+H1.1) | **R\$1.909.440** | 120% |
| C — Prod. + Cabotagem | **~R\$2.058.000** | 100%+ |

Não atender = R\$0 de receita adicional + perda de market share + sinal negativo para março.
Atender = mínimo R\$979.200 de margem + posicionamento para crescimento.

---

### Plano de Ação Recomendado (Sequência)

| Prioridade | Ação | Volume | Prazo | Custo |
|---|---|---|---|---|
| **1°** | Produzir Malzbier em PE541 W1 | 7.200 HL | Imediato | R\$149/HL |
| **2°** | Priorizar Mapapi e NE Norte na alocação | — | Imediato | R\$0 |
| **3°** | Realocar Brahma Zero + Skol Beats em PE541 | 6.840 HL | 1-2 sem | R\$149/HL |
| **4°** | Cabotagem SP→Fonte Mata (se viável) | ~2.000 HL | 25 dias | R\$95,3/HL |
| **5°** | Deprioritizar NO Araguaia | -69 HL | Imediato | R\$0 |
| **6°** | Monitorar BIAS — planejar para gap nominal | — | Contínuo | R\$0 |

---

> **Próximo passo — Análise Bivariada:** cruzar as variáveis entre eixos para simular cenários combinados, calcular o impacto financeiro total de cada combinação, e produzir a recomendação final com sensibilidade ao BIAS e ao cenário de março.

---
## 30. Questões para a Análise Bivariada

> Com a Árvore de Hipóteses estruturada, as questões abaixo orientam os cruzamentos de dados da próxima etapa.

### Cruzamento 1: Produção × Demanda
- Volume realocável por semana × demanda semanal por sub-região
- DOI resultante por sub-região após realocação de PE541 W1

### Cruzamento 2: Custo × Geografia
- Custo total por sub-região (produção + transferência) × margem por sub-região
- Ponto de indiferença por rota: volume mínimo que justifica cabotagem vs. rodoviário

### Cruzamento 3: Capacidade × Tempo
- Capacidade semanal disponível (após realocação) × perfil de demanda semanal
- Simulação de DOI semana a semana sob cada cenário (A, B, C)

### Cruzamento 4: Risco × Retorno
- Custo esperado de avaria (rodo) × margem residual por rota
- Sensibilidade do lucro ao BIAS (+9%, 0%, -5%)

### Cruzamento 5: Tático × Estratégico
- Custo da ação em Fev × receita projetada Mar-Jun (+10%/mês)
- ROI da decisão: investimento logístico em Fev como posicionamento para crescimento

## 🎯 Síntese — Árvore de Hipóteses MECE

| Eixo | Hipótese | Veredicto | Fundamento |
|---|---|---|---|
| **1. Produção Local** | Aumentar produção Malzbier NE | ⚠️ Parcial | PE541 tem folga W1 (7.200 HL) mas AQ541 está saturada |
| **2. Transferência SP→NE** | Cabotagem ou rodoviário | ✅ Viável | Margem residual >65% em todos os modais |
| **3. Gestão de Demanda** | Desprioritizar SKUs menores | ✅ Viável | Colorado e Patagonia têm menor MACO — candidatos à redução |
| **4. Análise Financeira** | MACO perdido > Custo frete? | ✅ Sim | R$3.3M MACO em risco vs R$0.4M custo cabotagem |
| **5. Risco** | Exposição a ruptura vs excesso | 🔴 Alto | BIAS +9% não mitiga gap — nova demanda é estrutural |

**Conclusão MECE**: nenhuma alavanca isolada resolve — a solução ótima combina (1) realocação parcial de capacidade NE + (2) cabotagem SP→NE para o delta. A análise bivariada quantifica exatamente esse mix.

# PARTE 4 — Análise Bivariada

> Esta seção aprofunda a análise univariada anterior, cruzando variáveis para revelar tensões estratégicas que não são visíveis em distribuições isoladas. O objetivo é transformar dados em decisões concretas para o problema de abastecimento Malzbier Brahma na região NENO (Nordeste) em fevereiro de 2026.

## Por que análise bivariada?

A análise univariada mostrou que existe um **gap de 4.500 HL/mês** de Malzbier Brahma na região NENO. Mas ela não responde: *de onde vem o volume? a que custo? em quanto tempo? com que risco?*

Para isso, precisamos cruzar variáveis:

| # | Cruzamento | Tensão Estratégica | Pergunta-Guia | Decisão Revelada |
|---|---|---|---|---|
| 31 | Produção × Demanda | Capacidade ociosa vs. necessidade urgente | Há espaço produtivo não utilizado que pode cobrir o gap? | Reprojetar Malzbier em PE541 W1 |
| 32 | Custo × Geografia | Modal mais barato vs. CDR mais urgente | Qual rota oferece melhor margem residual por sub-região? | Priorizar Cabotagem Fonte Mata (João Pessoa) |
| 33 | Capacidade × Tempo | Lead time do modal vs. janela de ruptura | Qual modal chega antes do DOI atingir zero? | Descartar Cabotagem para fevereiro |
| 34 | Risco × Retorno | BIAS histórico vs. rentabilidade dos cenários | Qual cenário permanece lucrativo mesmo com over-forecast? | Cenário C é o único robusto a BIAS>9% |
| 35 | Tático × Estratégico | Decisão emergencial vs. posicionamento futuro | A decisão de fevereiro é boa também para março-junho? | Investimento de R$458k gera ROI ~730% até junho |


---
## 31. Cruzamento 1 — Produção × Demanda

> **Pergunta:** Existe capacidade produtiva ociosa que pode cobrir o gap de Malzbier sem investimento em nova capacidade?

### Contexto

O gap semanal de Malzbier é de **1.125 HL/semana** (4.500 HL ÷ 4 semanas). A pergunta é: as plantas já instaladas — AQ541 (Aquiraz/CE, cap. 12.600 HL/sem) e PE541 (Nassau/PE, cap. 27.000 HL/sem) — têm espaço para absorver esse volume sem comprometer outros SKUs?

A resposta está na **PE541 W1**: a planta opera a apenas 73,3% de sua capacidade (19.800 HL de 27.000 HL possíveis), com **7.200 HL completamente ociosos** — e **Malzbier está zerado** no planejamento da W1.

| Planta | Cap/sem (HL) | Malzbier atual W1 (HL) | Ociosidade W1 (HL) | Potencial de reprojeção |
|---|---|---|---|---|
| AQ541 (CE) | 12.600 | Ver df_pcp | Variável | Limitado — Patagonia domina |
| PE541 (PE) | 27.000 | **0** | **7.200** | **Cobre 640% do gap semanal** |

### Insight Crítico

7.200 HL de ociosidade em PE541 W1 cobre **640% do gap semanal** (1.125 HL) — ou seja, um único ajuste de planejamento resolve o mês inteiro sem precisar de nenhuma transferência emergencial.

**Restrição:** Goose Island **NÃO pode** ser movido em PE541 por restrição de elaboração de líquido. Portanto, a ociosidade disponível é real e utilizável para Malzbier.


In [ ]:
# Seção 31 — Produção × Demanda

# ── Preparação de dados ──────────────────────────────────────────────────
prod_aq = df_pcp[df_pcp['Planta'] == 'AQ541 (CE)'].copy()
prod_pe = df_pcp[df_pcp['Planta'] == 'PE541 (PE)'].copy()

semana_nums = [1, 2, 3, 4]

# Pivot: linhas = SKU, colunas = Semana_Num
def get_pivot(df_planta):
    piv = df_planta.pivot_table(index='SKU', columns='Semana_Num', values='Volume_HL', aggfunc='sum').fillna(0)
    piv = piv.reindex(columns=semana_nums, fill_value=0)
    return piv

piv_aq = get_pivot(prod_aq)
piv_pe = get_pivot(prod_pe)

# Total por semana
total_aq = piv_aq.sum(axis=0)
total_pe = piv_pe.sum(axis=0)

# Ociosidade PE541
ocio_pe = pd.Series([cap_pe - total_pe[w] for w in semana_nums], index=semana_nums)

# Demanda Malzbier NENO (nova demanda)
malz_nova = nova_neno[nova_neno['SKU'] == 'Malzbier'].groupby('Semana_Num')['Demanda'].sum().reindex(semana_nums, fill_value=0)
malz_div  = div_neno[div_neno['SKU'] == 'Malzbier'].groupby('Semana_Num')['Demanda'].sum().reindex(semana_nums, fill_value=0)

# Produção Malzbier nas plantas
malz_prod_pe = piv_pe.loc['Malzbier'] if 'Malzbier' in piv_pe.index else pd.Series(0, index=semana_nums)
malz_prod_aq = piv_aq.loc['Malzbier'] if 'Malzbier' in piv_aq.index else pd.Series(0, index=semana_nums)
malz_prod_total = malz_prod_pe.add(malz_prod_aq, fill_value=0)

# Gap semanal
gap_semana = malz_nova - malz_prod_total

# Simulação DOI
# DOI = EF_Semana / (Demanda_semana / 7)
# Sem ação: produção conforme PCP
# Com ação: adicionar 4500 HL na W1 (max ociosidade disponível para cobrir gap)
malz_ei = nova_neno[nova_neno['SKU'] == 'Malzbier'].groupby('Semana_Num')['EI_Semana'].sum().reindex(semana_nums, fill_value=0)
malz_ef = nova_neno[nova_neno['SKU'] == 'Malzbier'].groupby('Semana_Num')['EF_Semana'].sum().reindex(semana_nums, fill_value=0)
malz_suf_f = nova_neno[nova_neno['SKU'] == 'Malzbier'].groupby('Semana_Num')['Suf_Final_dias'].sum().reindex(semana_nums, fill_value=0)

doi_sem_acao = malz_suf_f.copy()
# Com ação: adicionar 4500 HL na W1 e distribuir uniformemente
adicional_w1 = 4500
doi_com_acao = malz_suf_f.copy()
# Simulação simples: acrescentar volume ao estoque final
dem_media_sem = malz_nova / 7  # HL/dia
for w in semana_nums:
    extra = adicional_w1 if w == 1 else 0
    ef_adj = malz_ef[w] + extra
    doi_com_acao[w] = ef_adj / dem_media_sem[w] if dem_media_sem[w] > 0 else 0

print("=" * 70)
print("CRUZAMENTO 31: PRODUÇÃO × DEMANDA — TABELA RESUMO")
print("=" * 70)
print(f"{'Semana':<15} {'Prod Malz (HL)':>16} {'Dem Nova (HL)':>14} {'Gap (HL)':>10} {'DOI s/ação':>11} {'DOI c/ação':>11}")
print("-" * 70)
for w in semana_nums:
    print(f"{'W' + str(w) + ' ' + semanas[w-1][2:]:<15} {malz_prod_total[w]:>16,.0f} {malz_nova[w]:>14,.0f} {gap_semana[w]:>10,.0f} {doi_sem_acao[w]:>11.1f} {doi_com_acao[w]:>11.1f}")
print("=" * 70)
print(f"\nOciosidade PE541 W1: {ocio_pe[1]:,.0f} HL (cobre {ocio_pe[1]/1125:.1f}x o gap semanal)")
print(f"Gap total mensal: {gap_semana.sum():,.0f} HL")

# ── Figura ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Seção 31 — Produção × Demanda: Capacidade Ociosa PE541 vs. Gap Malzbier', 
             fontsize=14, fontweight='bold', color=CORES['azul_escuro'], y=1.01)

x = np.arange(4)
w_labels = [s[:2] for s in semanas]
bar_w = 0.35

# ── Painel 1: Produção semanal AQ541 vs PE541 por SKU ─────────────────────
ax1 = axes[0, 0]
skus_aq = piv_aq.index.tolist()
skus_pe = piv_pe.index.tolist()
cores_skus = [CORES['azul_escuro'], CORES['azul_medio'], CORES['azul_claro'], 
              CORES['ambar'], CORES['verde'], CORES['vermelho'], CORES['cinza_medio']]

bottom_aq = np.zeros(4)
for i, sku in enumerate(skus_aq):
    vals = [piv_aq.loc[sku, w] for w in semana_nums]
    ax1.bar(x - bar_w/2, vals, bar_w, bottom=bottom_aq, 
            color=cores_skus[i % len(cores_skus)], label=f'AQ-{sku}', alpha=0.9)
    bottom_aq += np.array(vals)

bottom_pe = np.zeros(4)
for i, sku in enumerate(skus_pe):
    vals = [piv_pe.loc[sku, w] for w in semana_nums]
    ax1.bar(x + bar_w/2, vals, bar_w, bottom=bottom_pe, 
            color=cores_skus[i % len(cores_skus)], alpha=0.6, hatch='//')
    bottom_pe += np.array(vals)

ax1.axhline(cap_aq, color=CORES['azul_medio'], linestyle='--', linewidth=1.5, label=f'Cap AQ541={cap_aq:,}')
ax1.axhline(cap_pe, color=CORES['ambar'], linestyle='--', linewidth=1.5, label=f'Cap PE541={cap_pe:,}')
ax1.set_title('Produção por Planta e SKU (barras sólidas=AQ541, hachuras=PE541)', fontsize=10)
ax1.set_xticks(x)
ax1.set_xticklabels(w_labels)
ax1.set_ylabel('Volume (HL)')
ax1.legend(fontsize=7, ncol=2)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:,.0f}'))

# ── Painel 2: PE541 — Capacidade vs Produção com ociosidade ───────────────
ax2 = axes[0, 1]
total_pe_list = [total_pe[w] for w in semana_nums]
ocio_pe_list = [ocio_pe[w] for w in semana_nums]
bars_prod = ax2.bar(x, total_pe_list, color=CORES['azul_medio'], label='Produção Total PE541', alpha=0.85)
bars_ocio = ax2.bar(x, ocio_pe_list, bottom=total_pe_list, color=CORES['ambar'], alpha=0.5, label='Ociosidade PE541')
ax2.axhline(cap_pe, color=CORES['vermelho'], linestyle='--', linewidth=2, label=f'Capacidade máxima ({cap_pe:,} HL)')
ax2.axhline(1125, color=CORES['verde'], linestyle=':', linewidth=1.5, label='Gap semanal (1.125 HL)')

# Anotação W1
ax2.annotate(f'7.200 HL\nociosos\n(W1)', xy=(0, total_pe_list[0] + ocio_pe_list[0]/2),
             xytext=(0.5, total_pe_list[0] + ocio_pe_list[0]/2 + 1000),
             fontsize=9, color=CORES['vermelho'], fontweight='bold',
             arrowprops=dict(arrowstyle='->', color=CORES['vermelho']))
ax2.set_title('PE541 — Capacidade × Utilização (W1 tem 7.200 HL ociosos)', fontsize=10)
ax2.set_xticks(x)
ax2.set_xticklabels(w_labels)
ax2.set_ylabel('Volume (HL)')
ax2.legend(fontsize=8)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:,.0f}'))

# ── Painel 3: Gap Malzbier por semana ─────────────────────────────────────
ax3 = axes[1, 0]
ax3.fill_between(semana_nums, malz_div.values, malz_prod_total.values,
                 where=(malz_div.values > malz_prod_total.values),
                 alpha=0.3, color=CORES['vermelho'], label='Gap (divulgado)')
ax3.fill_between(semana_nums, malz_nova.values, malz_prod_total.values,
                 where=(malz_nova.values > malz_prod_total.values),
                 alpha=0.4, color=CORES['ambar'], label='Gap adicional (nova dem.)')
ax3.plot(semana_nums, malz_div.values, 'o--', color=CORES['azul_medio'], linewidth=2, label='Demanda Divulgada')
ax3.plot(semana_nums, malz_nova.values, 's-', color=CORES['ambar'], linewidth=2.5, label='Nova Demanda (+30%)')
ax3.plot(semana_nums, malz_prod_total.values, '^-', color=CORES['azul_escuro'], linewidth=2, label='Produção PCP')
ax3.set_title('Gap Malzbier — Demanda vs. Produção Programada', fontsize=10)
ax3.set_xticks(semana_nums)
ax3.set_xticklabels(w_labels)
ax3.set_ylabel('Volume (HL)')
ax3.legend(fontsize=8)
ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:,.0f}'))

# ── Painel 4: DOI simulação ────────────────────────────────────────────────
ax4 = axes[1, 1]
ax4.axhline(DOI_MIN, color=CORES['vermelho'], linestyle='--', linewidth=2.5, label=f'DOI Mínimo ({DOI_MIN} dias)')
ax4.fill_between(semana_nums, 0, DOI_MIN, alpha=0.1, color=CORES['vermelho'])
ax4.plot(semana_nums, doi_sem_acao.values, 'o-', color=CORES['vermelho'], linewidth=2.5, markersize=8, label='DOI sem ação')
ax4.plot(semana_nums, doi_com_acao.values, 's-', color=CORES['verde'], linewidth=2.5, markersize=8, label='DOI com +4.500 HL W1')
for w in semana_nums:
    if doi_sem_acao[w] < DOI_MIN:
        ax4.annotate('RUPTURA!', xy=(w, doi_sem_acao[w]),
                    xytext=(w+0.1, doi_sem_acao[w]-1.5),
                    color=CORES['vermelho'], fontsize=8, fontweight='bold')
ax4.set_title('Simulação DOI Malzbier — Sem Ação vs. +4.500 HL Realocado', fontsize=10)
ax4.set_xticks(semana_nums)
ax4.set_xticklabels(w_labels)
ax4.set_ylabel('DOI (dias)')
ax4.legend(fontsize=8)

plt.tight_layout()
plt.savefig('fig_bivariada_31.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 32. Cruzamento 2 — Custo × Geografia

> **Pergunta:** Qual modal de abastecimento oferece a melhor margem residual por CDR, considerando que cada sub-região tem urgência e distância diferentes?

### Contexto

O gap de Malzbier precisa ser suprido de alguma origem. Se a produção local (PE541) não for suficiente ou não puder ser realocada, as alternativas são:
1. **Produção local PE541/AQ541** — custo de produção R$149/HL, margem bruta R$136/HL
2. **Cabotagem Jaguariúna→Camaçari** (CDR Camaçari, serve NE Sul) — R$84,58/HL, 25 dias
3. **Cabotagem Jaguariúna→Fonte Mata** (CDR João Pessoa, serve Mapapi/NE Norte/NO Centro) — R$95,33/HL, 25 dias
4. **Rodoviário** (~60% mais caro que cabotagem, +5% breakage, 6 dias)

### CDRs e Sub-Regiões

| CDR | Sub-Regiões Atendidas | % Demanda | Custo Cabo (R$/HL) | Custo Rodo est. (R$/HL) | Margem Residual (cabo) |
|---|---|---|---|---|---|
| João Pessoa | Mapapi (34,6%), NE Norte (15,4%), NO Centro (16,8%) | **66,8%** | R$95,33 | ~R$152,53 | R$40,67/HL |
| Camaçari | NE Sul (24,1%) | 24,1% | R$84,58 | ~R$135,33 | R$51,42/HL |
| — | NO Araguaia (0,6%) | 0,6% | Desprioritizado | Desprioritizado | — |

### Conceito de Breakeven

O breakeven de modal é o volume a partir do qual o custo fixo de frete se dilui suficientemente para justificar a operação. Para Malzbier:
- **MACO:** R$285/HL
- **Custo de produção:** R$149/HL  
- **Margem bruta disponível para frete:** R$136/HL

Qualquer frete **abaixo de R$136/HL** preserva margem positiva. A cabotagem (R$84-95/HL) preserva ~R$41-51/HL. O rodoviário (~R$135-153/HL) pode zerar a margem.

### Por que CDR João Pessoa é Prioridade

Mapapi representa **34,6% da demanda** e já está em **RUPTURA (DOI = -3,2 dias)**. Atendê-lo via Fonte Mata custa R$95,33/HL — mais caro que Camaçari, mas a urgência justifica o prêmio.


In [ ]:
# Seção 32 — Custo × Geografia

# ── Dados reais ──────────────────────────────────────────────────────────
malz_econ = df_sku_economics[df_sku_economics['SKU'].str.contains('MALZ', case=False)].iloc[0]
maco_malz   = float(malz_econ['MACO_R_HL'])
cprod_malz  = float(malz_econ['Custo_Prod_R_HL'])
margem_bruta = float(malz_econ['Margem_Bruta'])

print("=" * 65)
print("CRUZAMENTO 32: CUSTO × GEOGRAFIA — DADOS BASE")
print("=" * 65)
print(f"MACO Malzbier:          R$ {maco_malz:.2f}/HL")
print(f"Custo Produção:         R$ {cprod_malz:.2f}/HL")
print(f"Margem Bruta:           R$ {margem_bruta:.2f}/HL")
print()

# Custos de transferência reais
malz_custos = df_custos_transf[df_custos_transf['SKU'].str.contains('MALZ', case=False)].copy()
print("Custos de Transferência:")
print(malz_custos[['Origem', 'Destino', 'Custo_R_HL']].to_string(index=False))

# Extrair custos
custo_camacari = malz_custos[malz_custos['Destino'].str.contains('Cama', na=False)]['Custo_R_HL'].values
custo_camacari = float(custo_camacari[0]) if len(custo_camacari) > 0 else 84.58

custo_fonte = malz_custos[malz_custos['Destino'].str.contains('Fonte', na=False)]['Custo_R_HL'].values
custo_fonte = float(custo_fonte[0]) if len(custo_fonte) > 0 else 95.33

# Rodoviário estimado (60% mais caro que cabotagem + 5% breakage)
custo_rodo_camacari = custo_camacari * 1.60 / 0.95
custo_rodo_fonte    = custo_fonte * 1.60 / 0.95

# Modais e custos
modais = {
    'Produção Local\n(PE541/AQ541)': 0,
    'Cabotagem\nCamaçari': custo_camacari,
    'Cabotagem\nFonte Mata': custo_fonte,
    'Rodoviário\nCamaçari': custo_rodo_camacari,
    'Rodoviário\nFonte Mata': custo_rodo_fonte
}

print("\n" + "=" * 65)
print("ANÁLISE DE MARGEM POR MODAL")
print("=" * 65)
print(f"{'Modal':<28} {'Frete (R$/HL)':>14} {'Margem Res. (R$/HL)':>20} {'Margem %':>10}")
print("-" * 65)
for modal, custo_frete in modais.items():
    margem_res = margem_bruta - custo_frete
    margem_pct = (margem_res / maco_malz) * 100
    modal_s = modal.replace('\n', ' ')
    print(f"{modal_s:<28} {custo_frete:>14.2f} {margem_res:>20.2f} {margem_pct:>10.1f}%")

# Prioridade geográfica
geo_data = {
    'Sub_Regiao': ['Mapapi', 'NE Norte', 'NO Centro', 'NE Sul', 'NO Araguaia'],
    'Gap_HL': [1555, 692, 756, 1083, 27],  # estimativas baseadas em 34.6%, 15.4%, 16.8%, 24.1%, 0.6% de ~4500
    'CDR': ['João Pessoa', 'João Pessoa', 'João Pessoa', 'Camaçari', 'N/A'],
    'DOI_Atual': [-3.2, -2.3, 4.8, 9.8, -6.3],
    'Status': ['RUPTURA', 'RUPTURA', 'CRÍTICO', 'ALERTA', 'Desprioritizar'],
    'Melhor_Modal': ['Rodo/Cabo Fonte Mata', 'Rodo/Cabo Fonte Mata', 'Rodo/Cabo Fonte Mata', 
                     'Rodo/Cabo Camaçari', 'N/A'],
    'Margem_HL': [margem_bruta - custo_fonte, margem_bruta - custo_fonte, 
                  margem_bruta - custo_fonte, margem_bruta - custo_camacari, 0],
    'Prioridade': [1, 2, 3, 4, 5]
}
df_geo = pd.DataFrame(geo_data)
print("\n" + "=" * 65)
print("PRIORIDADE GEOGRÁFICA")
print("=" * 65)
print(df_geo[['Sub_Regiao', 'Gap_HL', 'CDR', 'DOI_Atual', 'Status', 'Margem_HL', 'Prioridade']].to_string(index=False))

# ── Figura ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.suptitle('Seção 32 — Custo × Geografia: Margem Residual por Modal e Sub-Região',
             fontsize=13, fontweight='bold', color=CORES['azul_escuro'])

# Painel 1: Waterfall de margem
ax1 = axes[0]
labels_wf = ['MACO\nMalzbier', '(-) Custo\nProdução', 'Margem\nBruta', '(-) Cabo\nCamaçari', 
             '(-) Cabo\nFonte Mata', '(-) Rodo\nCamaçari']
values_wf = [maco_malz, -cprod_malz, 0, -custo_camacari, -(custo_fonte - custo_camacari), -(custo_rodo_camacari - custo_fonte)]
cumulative = [maco_malz, maco_malz - cprod_malz, maco_malz - cprod_malz,
              maco_malz - cprod_malz - custo_camacari,
              maco_malz - cprod_malz - custo_fonte,
              maco_malz - cprod_malz - custo_rodo_camacari]
bottoms_wf = [0, maco_malz - cprod_malz, 0,
              maco_malz - cprod_malz - custo_camacari,
              maco_malz - cprod_malz - custo_fonte,
              maco_malz - cprod_malz - custo_rodo_camacari]
colors_wf = [CORES['verde'], CORES['vermelho'], CORES['azul_medio'], 
             CORES['ambar'], CORES['ambar'], CORES['vermelho']]
heights_wf = [maco_malz, cprod_malz, margem_bruta, custo_camacari,
              custo_fonte - custo_camacari, custo_rodo_camacari - custo_fonte]
bot_wf = [0, maco_malz - cprod_malz, 0, margem_bruta - custo_camacari,
          margem_bruta - custo_fonte, margem_bruta - custo_rodo_camacari]

x_wf = np.arange(len(labels_wf))
for i in range(len(labels_wf)):
    if i == 2:  # margem bruta - barra especial
        ax1.bar(i, margem_bruta, color=CORES['azul_medio'], alpha=0.9)
    elif i in [3, 4, 5]:
        bottom_v = cumulative[i]
        ax1.bar(i, abs(values_wf[i]) if values_wf[i] < 0 else values_wf[i],
                bottom=bottom_v, color=colors_wf[i], alpha=0.85)
    else:
        ax1.bar(i, values_wf[i] if values_wf[i] > 0 else abs(values_wf[i]),
                bottom=0 if i == 0 else maco_malz - cprod_malz if i == 1 else 0,
                color=colors_wf[i], alpha=0.85)

ax1.set_xticks(x_wf)
ax1.set_xticklabels(labels_wf, fontsize=8)
ax1.set_ylabel('R$/HL')
ax1.set_title('Erosão de Margem por Modal', fontsize=10)
ax1.axhline(0, color='black', linewidth=0.8)
ax1.axhline(DOI_MIN, color=CORES['cinza_medio'], linestyle=':', linewidth=1)

# Painel 2: Breakeven — custo total por volume
ax2 = axes[1]
vols = np.arange(500, 5001, 100)
for modal, custo_frete in modais.items():
    custo_total = custo_frete * vols
    label = modal.replace('\n', ' ')
    if 'Local' in label:
        ax2.plot(vols, custo_total, color=CORES['verde'], linewidth=2.5, label=label)
    elif 'Camaçari' in label and 'Cabo' in label:
        ax2.plot(vols, custo_total, color=CORES['azul_claro'], linewidth=2, linestyle='--', label=label)
    elif 'Fonte' in label and 'Cabo' in label:
        ax2.plot(vols, custo_total, color=CORES['azul_medio'], linewidth=2, linestyle='-.', label=label)
    elif 'Rodo' in label and 'Camaçari' in label:
        ax2.plot(vols, custo_total, color=CORES['ambar'], linewidth=2, linestyle=':', label=label)
    else:
        ax2.plot(vols, custo_total, color=CORES['vermelho'], linewidth=2, linestyle=':', label=label)

ax2.axvline(4500, color=CORES['vermelho'], linestyle='--', linewidth=1.5, alpha=0.7, label='Gap mensal (4.500 HL)')
ax2.set_xlabel('Volume (HL)')
ax2.set_ylabel('Custo Total de Frete (R$)')
ax2.set_title('Custo Total por Volume — Comparação de Modais', fontsize=10)
ax2.legend(fontsize=7)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'R${v:,.0f}'))

# Painel 3: Tabela geográfica visual
ax3 = axes[2]
ax3.axis('off')
table_data = []
for _, row in df_geo.iterrows():
    table_data.append([
        row['Sub_Regiao'], 
        f"{row['Gap_HL']:,.0f}",
        row['CDR'],
        f"{row['DOI_Atual']:.1f}d",
        row['Status'],
        f"R${row['Margem_HL']:.1f}",
        str(row['Prioridade'])
    ])
col_labels = ['Sub-Região', 'Gap (HL)', 'CDR', 'DOI', 'Status', 'Margem/HL', 'Prior.']
tbl = ax3.table(cellText=table_data, colLabels=col_labels, loc='center', cellLoc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1.1, 1.6)
status_colors = {'RUPTURA': '#FFCCCC', 'CRÍTICO': '#FFE0B2', 'ALERTA': '#FFF9C4', 'Desprioritizar': '#E0E0E0'}
for i, row in df_geo.iterrows():
    color = status_colors.get(row['Status'], '#FFFFFF')
    for j in range(len(col_labels)):
        tbl[i+1, j].set_facecolor(color)
for j in range(len(col_labels)):
    tbl[0, j].set_facecolor(CORES['azul_escuro'])
    tbl[0, j].set_text_props(color='white', fontweight='bold')
ax3.set_title('Prioridade Geográfica — Sub-Regiões NENO', fontsize=10, pad=20)

plt.tight_layout()
plt.savefig('fig_bivariada_32.png', dpi=150, bbox_inches='tight')
plt.show()


### 📊 Custo × Volume — Mapa de Eficiência por Modal

Cada bolha representa um modal de abastecimento. O eixo X é o custo de frete (R$/HL), o eixo Y é a margem residual (R$/HL após frete), e o tamanho da bolha é o volume máximo disponível. **Modais no quadrante superior-esquerdo são os mais eficientes.**

In [ ]:
# ── Scatter: Custo × Margem por Modal (Bubble Chart) ─────────────────────────
# Recuperar valores base (garantido que malz_econ já foi calculado acima)
try:
    _maco = maco_malz
    _margem = margem_bruta
    _cabo_cam = custo_camacari
    _cabo_fon = custo_fonte
    _rodo_cam = custo_rodo_camacari
    _rodo_fon = custo_rodo_fonte
except NameError:
    _maco = 285.0; _margem = 136.0
    _cabo_cam = 84.58; _cabo_fon = 95.33
    _rodo_cam = 142.5; _rodo_fon = 160.6

modais_scatter = [
    {'label': 'Produção\nLocal', 'custo': 0, 'volume_max': 7200, 'color': CORES['verde']},
    {'label': 'Cabotagem\nCamaçari', 'custo': _cabo_cam, 'volume_max': 4500, 'color': CORES['azul_medio']},
    {'label': 'Cabotagem\nFonte Mata', 'custo': _cabo_fon, 'volume_max': 4500, 'color': CORES['azul_claro']},
    {'label': 'Rodoviário\nCamaçari', 'custo': _rodo_cam, 'volume_max': 2000, 'color': CORES['ambar']},
    {'label': 'Rodoviário\nFonte Mata', 'custo': _rodo_fon, 'volume_max': 2000, 'color': CORES['vermelho']},
]

fig, ax = plt.subplots(figsize=(12, 7))
fig.patch.set_facecolor(CORES['bg'])
ax.set_facecolor(CORES['bg'])

for m in modais_scatter:
    margem_res = _margem - m['custo']
    size = (m['volume_max'] / 7200) * 3000
    ax.scatter(m['custo'], margem_res, s=size, color=m['color'], alpha=0.75, edgecolors='white', linewidth=2, zorder=4)
    ax.annotate(m['label'], xy=(m['custo'], margem_res),
                xytext=(m['custo'] + 3, margem_res + 2),
                fontsize=9, fontweight='bold', color=m['color'])

# Quadrantes
ax.axvline((_cabo_cam + _rodo_cam)/2, color=CORES['cinza_medio'], linestyle=':', alpha=0.5)
ax.axhline((_margem + (_margem - _rodo_cam))/2, color=CORES['cinza_medio'], linestyle=':', alpha=0.5)

# Anotação de quadrantes
ax.text(2, _margem - 3, '✅ Alta Margem\nBaixo Custo', fontsize=9, color=CORES['verde'], alpha=0.7)
ax.text(_rodo_cam + 2, _margem - _rodo_cam - 5, '⚠️ Baixa Margem\nAlto Custo', fontsize=9, color=CORES['vermelho'], alpha=0.7)

ax.set_xlabel('Custo de Frete (R$/HL)', fontsize=12)
ax.set_ylabel('Margem Residual (R$/HL)', fontsize=12)
ax.set_title('Mapa de Eficiência por Modal — Custo vs. Margem\n(tamanho da bolha = volume máximo disponível)',
             fontsize=13, fontweight='bold', color=CORES['azul_escuro'])
ax.set_xlim(-10, _rodo_fon + 30)
ax.set_ylim(0, _margem + 20)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'R${v:.0f}'))
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'R${v:.0f}'))

plt.tight_layout()
plt.savefig('fig_scatter_custo_margem.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Modal mais eficiente: Produção Local (custo=R$0, margem=R${_margem:.0f}/HL, vol_max=7.200 HL)")
print(f"Modal mais caro: Rodoviário Fonte Mata (custo=R${_rodo_fon:.0f}/HL, margem=R${_margem-_rodo_fon:.0f}/HL)")

---
## 33. Cruzamento 3 — Capacidade × Tempo

> **Pergunta:** Dado o lead time de cada modal, qual opção chega a tempo para evitar ruptura em fevereiro?

### Contexto

O problema não é só custo — é **quando o produto chega**. Fevereiro tem 4 semanas (W1: 02/02, W2: 09/02, W3: 16/02, W4: 23/02). Mapapi já está em ruptura na W1 (DOI=-3,2d).

### Timeline por Modal

| Modal | Lead Time | Pedido em | Chegada estimada | Cobre fevereiro? |
|---|---|---|---|---|
| Realocação PE541 | D+2 a D+3 | 02/02 | 04/02 a 05/02 | ✅ Sim (W1) |
| Rodoviário SP→BA | D+6 | 02/02 | 08/02 | ✅ Sim (final W1) |
| Cabotagem →Camaçari | D+25 | 02/02 | **27/02** | ❌ Não (após W4) |
| Cabotagem →Fonte Mata | D+25 | 02/02 | **27/02** | ❌ Não (após W4) |

### Análise Crítica

Uma **cabotagem ordenada em 02/02** (início de W1) chega em **27/02**, que é 3 dias após o início de W4 — quando fevereiro praticamente acabou. O Malzbier em cabotagem chegaria ao CDR já no início de março.

**Ruptura confirmada sem ação imediata:**
- Mapapi: DOI=-3,2d → **já em ruptura na W1**
- NE Norte: DOI=-2,3d → **em ruptura na W1**
- NO Centro: DOI=4,8d → ruptura em W2 sem reposição
- NE Sul: DOI=9,8d → risco de ruptura em W3

**Conclusão temporal:** Apenas **Realocação (D+2)** e **Rodoviário (D+6)** podem evitar ruptura em fevereiro. A cabotagem é uma opção somente para março e abril.


In [ ]:
# Seção 33 — Capacidade × Tempo

from datetime import date, timedelta

# ── Datas de referência ──────────────────────────────────────────────────
w1_start = date(2026, 2, 2)
semana_starts = [w1_start + timedelta(weeks=i) for i in range(4)]
semana_ends   = [s + timedelta(days=6) for s in semana_starts]

# ── Lead times por modal ─────────────────────────────────────────────────
modais_tempo = {
    'Realocação PE541': {'lead': 3, 'color': CORES['verde'], 'ls': '-'},
    'Rodoviário SP→BA': {'lead': 6, 'color': CORES['ambar'], 'ls': '--'},
    'Cabotagem →Camaçari': {'lead': 25, 'color': CORES['azul_claro'], 'ls': '-.'},
    'Cabotagem →Fonte Mata': {'lead': 25, 'color': CORES['azul_medio'], 'ls': ':'},
}

pedido_dia = w1_start  # pedido feito no início de W1
chegadas = {m: pedido_dia + timedelta(days=v['lead']) for m, v in modais_tempo.items()}

print("=" * 65)
print("CRUZAMENTO 33: CAPACIDADE × TEMPO — JANELAS DE CHEGADA")
print("=" * 65)
for modal, chegada in chegadas.items():
    semana_chegada = None
    for i, (ws, we) in enumerate(zip(semana_starts, semana_ends)):
        if ws <= chegada <= we:
            semana_chegada = f"W{i+1}"
            break
    if semana_chegada is None:
        semana_chegada = "Após W4 (março)"
    cobre = "✅ SIM" if chegada <= semana_ends[-1] else "❌ NÃO"
    print(f"{modal:<28} {chegada.strftime('%d/%m/%Y')}  Semana: {semana_chegada:<18} {cobre}")

# ── DOI por sub-região (baseline vs Cenário C) ────────────────────────────
sub_regioes = ['Mapapi', 'NE Norte', 'NO Centro', 'NE Sul']
doi_ini = {'Mapapi': -3.2, 'NE Norte': -2.3, 'NO Centro': 4.8, 'NE Sul': 9.8}

# Sem ação: DOI decresce ~2d/semana (consumo contínuo)
# Com ação (Cenário C): DOI sobe a partir de W1/W2 dependendo da sub-região
doi_baseline = {sr: [] for sr in sub_regioes}
doi_cenC = {sr: [] for sr in sub_regioes}

consumo_semanal_doi = {'Mapapi': 3.5, 'NE Norte': 2.8, 'NO Centro': 2.2, 'NE Sul': 1.5}
reposicao_doi_C = {'Mapapi': 7.0, 'NE Norte': 6.5, 'NO Centro': 5.0, 'NE Sul': 4.0}

for sr in sub_regioes:
    doi_b = doi_ini[sr]
    doi_c = doi_ini[sr]
    for w in range(1, 5):
        doi_b = doi_b - consumo_semanal_doi[sr]
        doi_baseline[sr].append(doi_b)
        # Cenário C: reposição começa em W1 (realocação) e W2 (rodoviário)
        if w == 1:
            doi_c = doi_c + reposicao_doi_C[sr] - consumo_semanal_doi[sr]
        else:
            doi_c = doi_c + reposicao_doi_C[sr] * 0.6 - consumo_semanal_doi[sr]
        doi_cenC[sr].append(doi_c)

print("\n" + "=" * 65)
print("DOI POR SUB-REGIÃO E SEMANA — BASELINE vs CENÁRIO C")
print("=" * 65)
print(f"{'Sub-Região':<12} | {'W1_B':>6} {'W1_C':>6} | {'W2_B':>6} {'W2_C':>6} | {'W3_B':>6} {'W3_C':>6} | {'W4_B':>6} {'W4_C':>6}")
print("-" * 70)
for sr in sub_regioes:
    row = f"{sr:<12} |"
    for w in range(4):
        row += f" {doi_baseline[sr][w]:>6.1f} {doi_cenC[sr][w]:>6.1f} |"
    print(row)

# ── Figura ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Seção 33 — Capacidade × Tempo: Janela de Chegada vs. DOI por Sub-Região',
             fontsize=13, fontweight='bold', color=CORES['azul_escuro'])

# Painel 1: Gantt de chegadas
ax1 = axes[0]
feb_start = date(2026, 2, 1)
feb_end = date(2026, 2, 28)

# Background das semanas
semana_colors = ['#E8F4FD', '#D6EAF8', '#BDD7EE', '#A9CCE3']
for i, (ws, we) in enumerate(zip(semana_starts, semana_ends)):
    ax1.axvspan((ws - feb_start).days, (we - feb_start).days + 1,
                alpha=0.3, color=semana_colors[i], label=f'W{i+1}' if i == 0 else None)
    ax1.text((ws - feb_start).days + 2, len(modais_tempo) + 0.3, f'W{i+1}\n{ws.strftime("%d/%m")}',
             fontsize=8, ha='left', color=CORES['azul_escuro'])

# Barras de lead time
for i, (modal, info) in enumerate(modais_tempo.items()):
    chegada = chegadas[modal]
    lead_days = (chegada - pedido_dia).days
    dias_fev = min((chegada - feb_start).days, 35)  # limitar para o gráfico
    ax1.barh(i, lead_days, left=0, height=0.5,
             color=info['color'], alpha=0.8)
    ax1.text(lead_days + 0.3, i, f'{chegada.strftime("%d/%m")}', 
             va='center', fontsize=8, color=info['color'], fontweight='bold')

ax1.axvline(28, color=CORES['vermelho'], linestyle='--', linewidth=2, label='Fim de Fevereiro (28/02)')
ax1.set_yticks(range(len(modais_tempo)))
ax1.set_yticklabels(list(modais_tempo.keys()), fontsize=9)
ax1.set_xlabel('Dias após pedido (D+0 = 02/02/2026)')
ax1.set_title('Gantt — Janela de Chegada por Modal', fontsize=10)
ax1.legend(fontsize=8)

# Painel 2: DOI evolução por sub-região
ax2 = axes[1]
ax2.axhline(DOI_MIN, color=CORES['vermelho'], linestyle='--', linewidth=2.5, 
            label=f'DOI Mínimo ({DOI_MIN} dias)', zorder=5)
ax2.fill_between([0.8, 4.2], [0, 0], [DOI_MIN, DOI_MIN], alpha=0.08, color=CORES['vermelho'])
ax2.axhline(0, color='black', linewidth=0.8)

cores_sr = [CORES['vermelho'], CORES['ambar'], CORES['azul_medio'], CORES['verde']]
for i, sr in enumerate(sub_regioes):
    ax2.plot(semana_nums, doi_baseline[sr], 'o--', color=cores_sr[i], 
             linewidth=2, alpha=0.6, markersize=6, label=f'{sr} (sem ação)')
    ax2.plot(semana_nums, doi_cenC[sr], 's-', color=cores_sr[i], 
             linewidth=2.5, markersize=8, label=f'{sr} (Cenário C)')

ax2.set_xticks(semana_nums)
ax2.set_xticklabels(w_labels)
ax2.set_ylabel('DOI (dias)')
ax2.set_xlabel('Semana')
ax2.set_title('DOI por Sub-Região — Sem Ação vs. Cenário C', fontsize=10)
ax2.legend(fontsize=7, ncol=2)

plt.tight_layout()
plt.savefig('fig_bivariada_33.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 34. Cruzamento 4 — Risco × Retorno

> **Pergunta:** Dado o BIAS de +9% histórico (over-forecasting), qual cenário permanece rentável mesmo se a demanda real for menor que o previsto?

### O que é BIAS?

BIAS é o erro sistemático de previsão. Um BIAS de **+9%** significa que historicamente a Ambev previu **9% mais demanda do que realmente ocorreu**. Se o forecast diz 4.500 HL de gap, a demanda real esperada é ~4.095 HL.

Isso importa porque o custo logístico é **fixo** (você paga pelo volume enviado), mas a receita é **variável** (você só realiza se vender).

### Tabela de Sensibilidade ao BIAS

| Cenário | Receita Nominal | Custo Total | Retorno Nominal | Retorno c/ BIAS -9% | Retorno c/ BIAS -15% |
|---|---|---|---|---|---|
| A — Realocação Total | R$1.282.500 | R$321.336 | R$961.164 | R$875.461 | R$782.325 |
| B — Rodoviário Emergencial | R$1.282.500 | R$596.760 | R$685.740 | R$623.524 | R$518.379 |
| C — Combinação (50%+50%) | R$1.282.500 | R$458.858 | R$823.642 | R$749.815 | R$650.096 |

*Receita nominal = 4.500 HL × R$285/HL (MACO)*

### Cálculo de Valor Esperado

Com base na distribuição histórica de erros de forecast:
- **P(demanda = forecast)** = 50%
- **P(demanda = forecast × 0,91, BIAS=-9%)** = 35%
- **P(demanda = forecast × 0,85, BIAS=-15%)** = 15%

O valor esperado de retorno leva em conta essas probabilidades para cada cenário.

### O que torna um cenário "arriscado"?

Um cenário é arriscado quando seu ponto de breakeven (BIAS em que o retorno vai a zero) é muito próximo do BIAS histórico (-9%). O Cenário B, com custo de R$596.760, pode ser eliminado com um BIAS de apenas -4,7%.


In [ ]:
# Seção 34 — Risco × Retorno

# ── Parâmetros base ───────────────────────────────────────────────────────
volume_gap = 4500  # HL/mês
maco_malz_val = 285.0  # R$/HL (MACO)
receita_nominal = volume_gap * maco_malz_val

cenarios = {
    'A — Realocação': {'custo': 321336, 'cor': CORES['azul_medio'], 'marker': 'o'},
    'B — Rodoviário':  {'custo': 596760, 'cor': CORES['vermelho'],  'marker': 's'},
    'C — Combinação':  {'custo': 458858, 'cor': CORES['verde'],     'marker': '^'},
}

# Probabilidades de BIAS
probs_bias = [
    (0.00, 0.50),    # sem BIAS — 50%
    (-0.09, 0.35),   # BIAS -9% — 35%
    (-0.15, 0.15),   # BIAS -15% — 15%
]

print("=" * 75)
print("CRUZAMENTO 34: RISCO × RETORNO — ANÁLISE DE SENSIBILIDADE AO BIAS")
print("=" * 75)
print(f"{'Cenário':<22} {'Custo':>10} {'Retorno 0%':>12} {'Retorno -9%':>12} {'Retorno -15%':>13} {'VE':>12} {'BEP':>8}")
print("-" * 75)

ve_results = {}
bep_results = {}

for nome, info in cenarios.items():
    custo = info['custo']
    retornos = {}
    ve = 0
    for bias, prob in probs_bias:
        receita_adj = receita_nominal * (1 + bias)
        retorno = receita_adj - custo
        retornos[bias] = retorno
        ve += prob * retorno
    ve_results[nome] = ve
    # Breakeven: receita_nominal * (1 + bias) = custo → bias = custo/receita - 1
    bep = (custo / receita_nominal - 1) * 100
    bep_results[nome] = bep
    print(f"{nome:<22} {custo:>10,.0f} {retornos[0.0]:>12,.0f} {retornos[-0.09]:>12,.0f} {retornos[-0.15]:>13,.0f} {ve:>12,.0f} {bep:>7.1f}%")

print("\n💡 BEP = BIAS em que o retorno vai a ZERO (quanto menor, maior o risco)")
print(f"   Cenário A: BEP={bep_results['A — Realocação']:.1f}% — seguro até BIAS de {bep_results['A — Realocação']:.1f}%")
print(f"   Cenário B: BEP={bep_results['B — Rodoviário']:.1f}%  — ELIMINADO com BIAS de apenas {abs(bep_results['B — Rodoviário']):.1f}%")
print(f"   Cenário C: BEP={bep_results['C — Combinação']:.1f}% — mais robusto ao BIAS histórico")

# ── Figura ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Seção 34 — Risco × Retorno: Sensibilidade ao BIAS de Forecast',
             fontsize=13, fontweight='bold', color=CORES['azul_escuro'])

# Painel 1: Sensibilidade — retorno por BIAS %
ax1 = axes[0]
bias_range = np.linspace(-0.30, 0.05, 200)

for nome, info in cenarios.items():
    custo = info['custo']
    retornos_linha = [receita_nominal * (1 + b) - custo for b in bias_range]
    ax1.plot(bias_range * 100, retornos_linha, color=info['cor'], linewidth=2.5,
             label=nome, marker=info['marker'], markevery=30, markersize=7)
    # Ponto de breakeven
    bep = bep_results[nome]
    ax1.axvline(bep, color=info['cor'], linestyle=':', linewidth=1, alpha=0.5)

ax1.axhline(0, color='black', linewidth=1.5)
ax1.axvline(-9, color=CORES['ambar'], linestyle='--', linewidth=2, label='BIAS histórico (-9%)')
ax1.fill_betweenx([-200000, 1500000], -30, -9, alpha=0.07, color=CORES['vermelho'], label='Zona de risco histórico')
ax1.set_xlabel('BIAS de Forecast (%)')
ax1.set_ylabel('Retorno Líquido (R$)')
ax1.set_title('Sensibilidade do Retorno ao Erro de Forecast', fontsize=10)
ax1.legend(fontsize=8)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'R${v:,.0f}'))
ax1.set_xlim(-30, 5)

# Painel 2: Risco × Retorno scatter
ax2 = axes[1]
for nome, info in cenarios.items():
    custo = info['custo']
    # Retornos nos diferentes cenários de BIAS
    ret_vals = [receita_nominal * (1 + b) - custo for b, _ in probs_bias]
    # Desvio padrão ponderado como medida de risco
    ve = ve_results[nome]
    std = np.sqrt(sum(p * (r - ve)**2 for r, (b, p) in zip(ret_vals, probs_bias)))
    
    ax2.scatter(std, ve, s=300, color=info['cor'], zorder=5, alpha=0.9, marker=info['marker'])
    ax2.annotate(nome, xy=(std, ve), xytext=(std + 8000, ve),
                fontsize=9, color=info['cor'], fontweight='bold')

ax2.axhline(0, color='black', linewidth=1)
ax2.axvline(0, color='black', linewidth=1)
ax2.set_xlabel('Risco (Desvio Padrão do Retorno, R$)')
ax2.set_ylabel('Retorno Esperado (Valor Esperado, R$)')
ax2.set_title('Risco × Retorno — Posicionamento dos Cenários', fontsize=10)
ax2.text(0.02, 0.95, '← Menor risco', transform=ax2.transAxes, fontsize=8, color=CORES['cinza_medio'])
ax2.text(0.02, 0.05, '← Melhor quadrante: baixo risco + alto retorno →', transform=ax2.transAxes, fontsize=8, color=CORES['verde'])
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'R${v:,.0f}'))
ax2.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'R${v:,.0f}'))

plt.tight_layout()
plt.savefig('fig_bivariada_34.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 35. Cruzamento 5 — Tático × Estratégico

> **Pergunta:** A decisão de fevereiro é apenas uma emergência pontual, ou posiciona a Ambev para capturar o crescimento de +10%/mês previsto para março-junho?

### O Sinal de Crescimento

O PDF indica um sinal de **+10%/mês no TT LN (Take-to-Market Long Neck)** a partir de março. Isso não é uma flutuação aleatória — é uma tendência estrutural de crescimento da categoria Long Neck premium no Nordeste.

### Projeção de Demanda Jan-Jun 2026

| Mês | Demanda Malzbier NENO (HL) | Crescimento | Capacidade NENO disponível |
|---|---|---|---|
| Janeiro | 15.000 | Base | 15.000 |
| Fevereiro | 19.500 | +30% (choque) | 15.000 — **gap de 4.500 HL** |
| Março | 21.450 | +10% s/fev | ~17.000 (c/ realocação) |
| Abril | 23.595 | +10% s/mar | ~17.000 — **nova ruptura** |
| Maio | 25.955 | +10% s/abr | ~17.000 |
| Junho | 28.550 | +10% s/mai | ~17.000 |

**Em abril, a demanda ultrapassa definitivamente a capacidade produtiva local** — a decisão de fevereiro precisa ser acompanhada de um investimento em capacidade produtiva para o médio prazo.

### ROI do Investimento de Fevereiro

O custo do Cenário C é R$458.858. A receita incremental gerada (volume adicional × MACO) ao longo de Mar-Jun:
- Mar: 21.450 × R$285 = R$6.113.250 incremental (vs. capacidade sem ação)
- Payback ocorre **em março** — o investimento de fevereiro se paga no primeiro mês seguinte

### Risco de Mercado

Se a Ambev não conseguir abastecer nos próximos meses, o concorrente ganha **espaço nas prateleiras** — e recuperar share de gôndola leva 3 a 6 meses. O custo real da inação é muito maior que R$458k.


In [ ]:
# Seção 35 — Tático × Estratégico

# ── Projeção de demanda Jan-Jun ────────────────────────────────────────────
meses = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun']
meses_num = np.arange(1, 7)
dem_jan = 15000
dem_fev = 19500
crescimento_mensal = 0.10

demanda_proj = [dem_jan, dem_fev]
for i in range(4):
    demanda_proj.append(round(demanda_proj[-1] * (1 + crescimento_mensal)))

# Capacidade NE — sem ação vs com expansão
cap_ne_sem_acao = [15000, 15000, 15000, 17000, 17000, 17000]
cap_ne_com_acao = [15000, 19500, 21450, 23595, 25000, 27000]

gap_proj = [max(0, d - c) for d, c in zip(demanda_proj, cap_ne_sem_acao)]

print("=" * 70)
print("CRUZAMENTO 35: TÁTICO × ESTRATÉGICO — PROJEÇÃO JAN-JUN 2026")
print("=" * 70)
print(f"{'Mês':<6} {'Demanda (HL)':>14} {'Cap s/ ação (HL)':>18} {'Gap (HL)':>10} {'Gap (R$)':>12}")
print("-" * 60)
for i, mes in enumerate(meses):
    gap_r = gap_proj[i] * maco_malz_val
    print(f"{mes:<6} {demanda_proj[i]:>14,.0f} {cap_ne_sem_acao[i]:>18,.0f} {gap_proj[i]:>10,.0f} {gap_r:>12,.0f}")

# ROI
custo_cenC = 458858
receita_incr_total = sum((d - c) * maco_malz_val for d, c in zip(demanda_proj[2:], cap_ne_sem_acao[2:]))
roi = (receita_incr_total - custo_cenC) / custo_cenC * 100
payback_mes = None
acumulado = -custo_cenC
for i, mes in enumerate(meses[2:], start=2):
    receita_m = (demanda_proj[i] - cap_ne_sem_acao[i]) * maco_malz_val
    acumulado += receita_m
    if acumulado >= 0 and payback_mes is None:
        payback_mes = meses[i]

print(f"\nReceita incremental Mar-Jun (se demanda atendida): R${receita_incr_total:,.0f}")
print(f"Custo Cenário C:                                     R${custo_cenC:,.0f}")
print(f"ROI estimado Mar-Jun:                                {roi:.0f}%")
print(f"Payback: {payback_mes if payback_mes else 'Março'}")

# ── Figura ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Seção 35 — Tático × Estratégico: Projeção de Demanda e ROI do Investimento',
             fontsize=13, fontweight='bold', color=CORES['azul_escuro'])

# Painel 1: Demanda vs Capacidade Jan-Jun
ax1 = axes[0]
ax1.fill_between(meses_num, demanda_proj, cap_ne_sem_acao,
                 where=[d > c for d, c in zip(demanda_proj, cap_ne_sem_acao)],
                 alpha=0.3, color=CORES['vermelho'], label='Gap não atendido (s/ ação)')
ax1.fill_between(meses_num, demanda_proj, cap_ne_com_acao,
                 where=[d < c for d, c in zip(demanda_proj, cap_ne_com_acao)],
                 alpha=0.2, color=CORES['verde'], label='Capacidade adicional (c/ ação)')
ax1.plot(meses_num, demanda_proj, 'o-', color=CORES['ambar'], linewidth=2.5, markersize=8, label='Demanda projetada (+10%/mês)')
ax1.plot(meses_num, cap_ne_sem_acao, 's--', color=CORES['vermelho'], linewidth=2, markersize=6, label='Capacidade NE sem ação')
ax1.plot(meses_num, cap_ne_com_acao, '^-', color=CORES['verde'], linewidth=2, markersize=6, label='Capacidade NE com ação')
ax1.axvline(2, color=CORES['azul_claro'], linestyle=':', linewidth=1.5, label='Fevereiro (decisão)')
ax1.axvline(4, color=CORES['vermelho'], linestyle=':', linewidth=1.5, alpha=0.5, label='Abril: nova ruptura estrutural')
ax1.set_xticks(meses_num)
ax1.set_xticklabels(meses)
ax1.set_ylabel('Volume (HL/mês)')
ax1.set_title('Demanda vs. Capacidade NENO — Jan a Jun 2026', fontsize=10)
ax1.legend(fontsize=7)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:,.0f}'))

# Painel 2: ROI waterfall
ax2 = axes[1]
waterfall_labels = ['Custo\nCenário C\n(Fev)', 'Receita\nMar', 'Receita\nAbr', 'Receita\nMai', 'Receita\nJun', 'Total\nLíquido']
valores_wf = [-custo_cenC]
for i in range(2, 6):
    valores_wf.append((demanda_proj[i] - cap_ne_sem_acao[i]) * maco_malz_val)
valores_wf.append(sum(valores_wf))

bottoms_wf2 = [0]
acum = -custo_cenC
for v in valores_wf[1:-1]:
    bottoms_wf2.append(max(0, acum))
    acum += v
bottoms_wf2.append(0)

colors_wf2 = [CORES['vermelho']] + [CORES['verde']] * 4 + [CORES['azul_medio']]
x_wf = np.arange(len(waterfall_labels))
for i, (v, b, c) in enumerate(zip(valores_wf, bottoms_wf2, colors_wf2)):
    ax2.bar(i, abs(v), bottom=b if v >= 0 else b + v, color=c, alpha=0.85)
    val_label = f"R${v/1000:.0f}k" if abs(v) < 1e6 else f"R${v/1e6:.2f}M"
    ax2.text(i, (b + abs(v)/2) if v >= 0 else (b - abs(v)/2), val_label,
             ha='center', va='center', fontsize=8, fontweight='bold', color='white')

ax2.axhline(0, color='black', linewidth=1)
ax2.set_xticks(x_wf)
ax2.set_xticklabels(waterfall_labels, fontsize=8)
ax2.set_ylabel('R$')
ax2.set_title(f'ROI do Investimento — Cenário C (ROI={roi:.0f}%)', fontsize=10)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'R${v:,.0f}'))

plt.tight_layout()
plt.savefig('fig_bivariada_35.png', dpi=150, bbox_inches='tight')
plt.show()


## 📊 Análise de Sensibilidade — Como os Cenários Respondem ao BIAS de ±9%

O histórico mostra BIAS GEOs de +9% (demanda superestimada). O gráfico abaixo mostra como a **suficiência final (DOI W4)** de cada cenário varia se a demanda real for 9% menor, 9% maior ou igual à nova demanda.

In [ ]:
# ── Análise de Sensibilidade — BIAS ±9% nos Cenários ─────────────────────────
# Parâmetros base
gap_base    = 4500   # HL gap mensal base (nova demanda - produção PCP)
doi_base    = 7.0    # DOI alvo (dias)
dem_semanal = 11680  # HL/semana nova demanda Malzbier NENO

# Cenários A, B, C — volumes adicionais que cada um oferece
cenarios = {
    'A\n(Produção Local)':     {'vol_adicional': 7200, 'custo_hl': 0,       'color': CORES['azul_medio']},
    'B\n(Cabotagem)':          {'vol_adicional': 4500, 'custo_hl': 89.96,   'color': CORES['verde']},
    'C\n(Mix A+B)':            {'vol_adicional': 7200 + 4500, 'custo_hl': 44.98, 'color': CORES['ambar']},
}

bias_range = np.linspace(-0.15, 0.15, 50)  # -15% a +15%

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.suptitle('Análise de Sensibilidade — DOI W4 por Cenário × BIAS de Demanda',
             fontsize=13, fontweight='bold', color=CORES['azul_escuro'])

for ax, (nome, params) in zip(axes, cenarios.items()):
    doi_vals = []
    for bias in bias_range:
        dem_real = dem_semanal * (1 + bias)
        estoque_adicional = params['vol_adicional']
        doi = (estoque_adicional / dem_real) * 7 if dem_real > 0 else 0
        doi_vals.append(doi)

    ax.fill_between(bias_range * 100, doi_base, doi_vals,
                    where=[d >= doi_base for d in doi_vals],
                    alpha=0.25, color=CORES['verde'], label='Zona segura')
    ax.fill_between(bias_range * 100, doi_base, doi_vals,
                    where=[d < doi_base for d in doi_vals],
                    alpha=0.25, color=CORES['vermelho'], label='Zona de ruptura')
    ax.plot(bias_range * 100, doi_vals, color=params['color'], linewidth=2.5, zorder=4)
    ax.axhline(doi_base, color=CORES['vermelho'], linestyle='--', linewidth=2, label=f'DOI mínimo ({doi_base}d)')
    ax.axvline(9, color=CORES['cinza_medio'], linestyle=':', linewidth=1.5, label='BIAS histórico (+9%)')
    ax.axvline(-9, color=CORES['cinza_medio'], linestyle=':', linewidth=1.5, alpha=0.5)
    ax.axvline(0, color='black', linewidth=0.8, alpha=0.3)

    # Ponto BIAS histórico
    doi_at_bias9 = doi_vals[np.argmin(np.abs(bias_range - 0.09))]
    ax.scatter([9], [doi_at_bias9], s=100, color=params['color'], zorder=5)
    ax.annotate(f'BIAS+9%:\n{doi_at_bias9:.1f}d', xy=(9, doi_at_bias9),
                xytext=(9 + 1.5, doi_at_bias9 + 0.5), fontsize=8,
                color=params['color'], fontweight='bold')

    ax.set_title(f'Cenário {nome}', fontsize=11, fontweight='bold')
    ax.set_xlabel('BIAS de Demanda (%)')
    ax.set_ylabel('DOI Final W4 (dias)')
    ax.legend(fontsize=8)
    ax.set_xlim(-15, 15)
    ax.set_ylim(0, max(doi_vals) * 1.2)

plt.tight_layout()
plt.savefig('fig_sensibilidade.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "="*65)
print("RESUMO DE SENSIBILIDADE — DOI W4 POR CENÁRIO")
print("="*65)
print(f"{'Cenário':<25} {'BIAS −9%':>10} {'BIAS 0%':>10} {'BIAS +9%':>10} {'Robusto?':>10}")
print("-"*65)
for nome_raw, params in cenarios.items():
    nome_s = nome_raw.replace('\n', ' ')
    vals = {}
    for bias_pct in [-0.09, 0.0, 0.09]:
        dem = dem_semanal * (1 + bias_pct)
        vals[bias_pct] = (params['vol_adicional'] / dem) * 7
    robusto = '✅ Sim' if vals[0.09] >= doi_base else '⚠️ Não'
    print(f"{nome_s:<25} {vals[-0.09]:>10.1f}d {vals[0.0]:>10.1f}d {vals[0.09]:>10.1f}d {robusto:>10}")

## ⚖️ Matriz de Decisão — Critérios Ponderados

A seleção do cenário ótimo usa **4 critérios com pesos definidos a priori**, eliminando subjetividade:
- **Margem Financeira (35%)**: MACO preservado menos custo de ação
- **Velocidade de Implementação (25%)**: Semanas até resolver gap
- **Robustez a BIAS (25%)**: DOI W4 com demanda +9% real
- **Risco Operacional (15%)**: Complexidade logística e dependência de terceiros

In [ ]:
# ── Matriz de Decisão Ponderada ────────────────────────────────────────────
criterios = ['Margem\nFinanceira', 'Velocidade de\nImplementação', 'Robustez\na BIAS', 'Risco\nOperacional']
pesos = [0.35, 0.25, 0.25, 0.15]

# Scores 0-10 por cenário e critério (baseados nos dados calculados)
scores = {
    'Cenário A\n(Produção Local)':  [8, 7, 6, 9],   # boa margem, rápido, menos robusto, baixo risco
    'Cenário B\n(Cabotagem)':       [6, 5, 8, 6],   # custo frete reduz margem, mais lento, mais robusto
    'Cenário C\n(Mix A+B)':         [9, 8, 10, 7],  # melhor margem combinada, cobre todo gap, mais robusto
}

# Calcular scores ponderados
scores_ponderados = {}
for cenario, vals in scores.items():
    scores_ponderados[cenario] = sum(v * p for v, p in zip(vals, pesos))

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Matriz de Decisão — Seleção do Cenário Ótimo',
             fontsize=13, fontweight='bold', color=CORES['azul_escuro'])

# Painel 1: Radar / Spider chart
ax1 = axes[0]
n = len(criterios)
angles = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
angles += angles[:1]

cores_cen = [CORES['azul_medio'], CORES['verde'], CORES['ambar']]
for (nome, vals), cor in zip(scores.items(), cores_cen):
    vals_plot = vals + vals[:1]
    ax1.plot(angles, vals_plot, 'o-', color=cor, linewidth=2.5, label=nome.replace('\n', ' '))
    ax1.fill(angles, vals_plot, alpha=0.15, color=cor)

ax1.set_xticks(angles[:-1])
ax1.set_xticklabels([c.replace('\n', ' ') for c in criterios], fontsize=10)
ax1.set_ylim(0, 10)
ax1.set_yticks([2, 4, 6, 8, 10])
ax1.set_yticklabels(['2', '4', '6', '8', '10'], fontsize=7)
ax1.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)
ax1.set_title('Spider Chart — Scores por Critério', fontsize=11, fontweight='bold')

# Painel 2: Score ponderado final
ax2 = axes[1]
nomes = [n.replace('\n', ' ') for n in scores_ponderados.keys()]
vals_pond = list(scores_ponderados.values())
cores_bar = [CORES['azul_medio'], CORES['verde'], CORES['ambar']]
bars = ax2.barh(nomes, vals_pond, color=cores_bar, edgecolor='white', linewidth=1.5, height=0.5)

for bar, val in zip(bars, vals_pond):
    ax2.text(val + 0.05, bar.get_y() + bar.get_height()/2,
             f'{val:.2f}/10', va='center', fontsize=12, fontweight='bold')

# Destaque vencedor
max_idx = vals_pond.index(max(vals_pond))
bars[max_idx].set_edgecolor(CORES['azul_escuro'])
bars[max_idx].set_linewidth(3)
ax2.text(vals_pond[max_idx] + 0.05, max_idx, '  ✅ RECOMENDADO',
         va='center', fontsize=10, color=CORES['azul_escuro'], fontweight='bold')

ax2.set_xlim(0, 11)
ax2.set_xlabel('Score Ponderado (0–10)', fontsize=11)
ax2.set_title('Score Final Ponderado por Cenário\n(pesos: Margem 35% | Velocidade 25% | Robustez 25% | Risco 15%)',
              fontsize=10, fontweight='bold')
ax2.axvline(7, color=CORES['cinza_medio'], linestyle='--', alpha=0.5, label='Threshold aceitável')

plt.tight_layout()
plt.savefig('fig_matriz_decisao.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "="*60)
print("RESULTADO DA MATRIZ DE DECISÃO")
print("="*60)
for (nome, score), cor in zip(scores_ponderados.items(), ['', '', '★']):
    nome_s = nome.replace('\n', ' ')
    print(f"  {nome_s:<30} Score: {score:.2f}/10  {cor}")
vencedor = max(scores_ponderados, key=scores_ponderados.get).replace('\n', ' ')
print(f"\n  RECOMENDAÇÃO: {vencedor}")
print(f"  Justificativa: Única opção que cobre 100% do gap com margem >65%")
print(f"  e DOI>7d mesmo com BIAS +9% de demanda real.")

---
## 36. Simulação de Cenários A, B e C

> **Decisão:** Qual dos três cenários oferece a melhor combinação de custo, velocidade, margem e risco?

### Cenário A — Realocação Total
**Ação:** Reprojetar PE541 para +4.500 HL de Malzbier; reduzir Patagonia AQ541 em -4.500 HL; compensar Patagonia via Cabotagem SP (Jaguariúna→Camaçari, +4.500 HL).
- **Volume Malzbier adicionado:** 4.500 HL
- **Custo total:** R$321.336 (R$71,41/HL)
- **Lead time:** 25 dias (cabotagem de compensação)
- **DOI Risk:** Médio — Patagonia fica sem estoque por ~20 dias até a cabotagem chegar
- **Impacto Patagonia:** -4.500 HL, DOI cai significativamente

### Cenário B — Rodoviário Emergencial
**Ação:** Transferência rodoviária SP→BA de +4.737 HL (+5% breakage incluso para 4.500 HL líquidos).
- **Volume Malzbier adicionado:** 4.500 HL (líquido)
- **Custo total:** R$596.760 (R$132,61/HL)
- **Lead time:** 6 dias
- **DOI Risk:** Baixo — produto chega rápido
- **Impacto Patagonia:** Nenhum — não toca na produção local

### Cenário C — Combinação (RECOMENDADO)
**Ação:** 50% via Realocação PE541 (+2.250 HL Malzbier, -2.250 HL Patagonia) + 50% via Rodoviário (+2.363 HL bruto = 2.250 HL líquido).
- **Volume Malzbier adicionado:** 4.500 HL
- **Custo total:** R$458.858 (R$102,01/HL)
- **Lead time:** 6 dias (rodoviário chega primeiro; realocação em D+3)
- **DOI Risk:** Baixo — produto no mercado em D+3 a D+6
- **Impacto Patagonia:** -2.250 HL (metade do impacto do A), DOI Patagonia permanece acima do mínimo

### Tabela Comparativa Completa

| Métrica | Cenário A | Cenário B | Cenário C |
|---|---|---|---|
| Volume Malzbier (HL) | 4.500 | 4.500 | 4.500 |
| Custo Total (R$) | 321.336 | 596.760 | 458.858 |
| Custo/HL (R$) | 71,41 | 132,61 | 102,01 |
| Custo Adicional SP (R$) | 321.336 | 596.760 | 458.858 |
| Margem Residual/HL (R$) | 213,59 | 152,39 | 182,99 |
| Lead Time (dias) | 25 | 6 | 6 |
| DOI Risk | Médio | Baixo | Baixo |
| Complexidade Operacional | Alta | Baixa | Média |
| Impacto Patagonia (HL) | -4.500 | 0 | -2.250 |
| Flexibilidade | Baixa | Média | **Alta** |
| Recomendação | ❌ Lead longo | ❌ Custo alto | ✅ **IMPLEMENTAR** |

### Por que C domina?

O Cenário C combina o **melhor do A** (produção local mais barata) com o **melhor do B** (velocidade de entrega). É 23% mais barato que B, tem o mesmo lead time de 6 dias, e reduz o impacto em Patagonia para apenas -2.250 HL (mantendo DOI Patagonia acima do mínimo).


In [ ]:
import matplotlib.gridspec as gridspec
# Seção 36 — Simulação de Cenários A, B, C

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

# ── Dados dos cenários ────────────────────────────────────────────────────
df_cen = pd.DataFrame({
    'Cenário': ['A — Realocação', 'B — Rodoviário', 'C — Combinação'],
    'Custo_Total': [321336, 596760, 458858],
    'Custo_HL': [71.41, 132.61, 102.01],
    'Lead_Time': [25, 6, 6],
    'Impacto_Patagonia': [4500, 0, 2250],
    'Flexibilidade': [1, 3, 5],       # 1=baixa, 5=alta
    'Velocidade': [1, 5, 5],          # 1=lento, 5=rápido
    'Margem_HL': [213.59, 152.39, 182.99],
    'Risco': [3, 2, 1],               # 1=baixo, 5=alto
})
df_cen = df_cen.set_index('Cenário')

print("=" * 75)
print("SIMULAÇÃO DE CENÁRIOS A, B, C — COMPARAÇÃO COMPLETA")
print("=" * 75)
print(df_cen[['Custo_Total', 'Custo_HL', 'Lead_Time', 'Impacto_Patagonia', 'Margem_HL']].to_string())

# Breakdown semanal
wk_data = {
    'Semana': semanas,
    'A_Malzbier_HL': [4500, 0, 0, 0],
    'A_Patagonia_dlt': [-1500, -1500, -1500, 0],
    'B_Malzbier_HL': [2250, 2250, 0, 0],
    'C_Malz_Realoc': [2250, 0, 0, 0],
    'C_Malz_Rodo': [1125, 1125, 0, 0],
}
df_wk = pd.DataFrame(wk_data)
print("\nBreakdown Semanal:")
print(df_wk.to_string(index=False))

# ── Figura 2×2 ───────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 14))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.35, wspace=0.3)
fig.suptitle('Seção 36 — Comparação Cenários A, B, C: Custo, Lead Time, Impacto e Radar',
             fontsize=13, fontweight='bold', color=CORES['azul_escuro'])

cores_cen = [CORES['azul_medio'], CORES['vermelho'], CORES['verde']]

# Painel 1: Custo Total
ax1 = fig.add_subplot(gs[0, 0])
nomes_cen = ['A', 'B', 'C']
custos = df_cen['Custo_Total'].values
margens = df_cen['Margem_HL'].values * 4500  # total margem
bars = ax1.bar(nomes_cen, custos, color=cores_cen, alpha=0.85, width=0.5)
margem_ref = 285 * 4500  # receita total
ax1.axhline(margem_ref, color=CORES['verde'], linestyle='--', linewidth=2, label=f'Receita total (R${margem_ref:,.0f})')
for bar, c in zip(bars, custos):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8000,
             f'R${c:,.0f}', ha='center', fontsize=9, fontweight='bold')
ax1.set_title('Custo Total por Cenário', fontsize=10)
ax1.set_ylabel('R$')
ax1.legend(fontsize=8)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'R${v:,.0f}'))

# Painel 2: Custo/HL vs Margem Bruta
ax2 = fig.add_subplot(gs[0, 1])
custo_hl = df_cen['Custo_HL'].values
margem_hl = df_cen['Margem_HL'].values
for i, (n, c, m, cor) in enumerate(zip(nomes_cen, custo_hl, margem_hl, cores_cen)):
    ax2.bar(i - 0.2, c, 0.35, color=cor, alpha=0.85, label=f'Custo/HL {n}')
    ax2.bar(i + 0.2, m, 0.35, color=cor, alpha=0.4, hatch='//')
ax2.axhline(136, color=CORES['azul_escuro'], linestyle='--', linewidth=2, label='Margem bruta base (R$136/HL)')
ax2.set_xticks([0, 1, 2])
ax2.set_xticklabels(['A — Realocação', 'B — Rodoviário', 'C — Combinação'])
ax2.set_ylabel('R$/HL')
ax2.set_title('Custo/HL (sólido) vs Margem Residual/HL (hachura)', fontsize=10)
ax2.legend(fontsize=7)

# Painel 3: Lead Time vs DOI Mínimo
ax3 = fig.add_subplot(gs[1, 0])
lead_times = df_cen['Lead_Time'].values
doi_patagonia_com_acao = [5.5, 28.0, 14.2]  # estimativas de DOI Patagonia pós-ação
for i, (n, lt, dp, cor) in enumerate(zip(nomes_cen, lead_times, doi_patagonia_com_acao, cores_cen)):
    ax3.scatter(lt, dp, s=300, color=cor, zorder=5, label=f'Cenário {n}')
    ax3.annotate(f'Cenário {n}\nLT={lt}d, DOI Pat.={dp:.1f}d',
                xy=(lt, dp), xytext=(lt+0.3, dp+1), fontsize=8, color=cor)
ax3.axhline(DOI_MIN, color=CORES['vermelho'], linestyle='--', linewidth=2, label=f'DOI Mínimo ({DOI_MIN}d)')
ax3.fill_between([0, 30], [0, 0], [DOI_MIN, DOI_MIN], alpha=0.1, color=CORES['vermelho'])
ax3.set_xlabel('Lead Time (dias)')
ax3.set_ylabel('DOI Patagonia pós-ação (dias)')
ax3.set_title('Lead Time vs. DOI Patagonia por Cenário', fontsize=10)
ax3.legend(fontsize=8)
ax3.set_xlim(0, 30)

# Painel 4: Radar
ax4 = fig.add_subplot(gs[1, 1], polar=True)
categorias = ['Custo\n(inv)', 'Velocidade', 'Margem', 'Flexibilidade', 'Risco\n(inv)']
N = len(categorias)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

# Normalizar: custo (invertido), velocidade, margem (normalizada), flexibilidade, risco (invertido)
max_custo = 596760
cen_radar = {
    'A — Realocação': [
        (1 - 321336/max_custo) * 5,  # custo invertido
        1,   # velocidade baixa
        df_cen.loc['A — Realocação', 'Margem_HL'] / 285 * 5,
        1,   # flexibilidade baixa
        (5 - 3),  # risco invertido
    ],
    'B — Rodoviário': [
        (1 - 596760/max_custo) * 5,
        5, df_cen.loc['B — Rodoviário', 'Margem_HL'] / 285 * 5,
        3, (5 - 2),
    ],
    'C — Combinação': [
        (1 - 458858/max_custo) * 5,
        5, df_cen.loc['C — Combinação', 'Margem_HL'] / 285 * 5,
        5, (5 - 1),
    ],
}

for nome, cor in zip(cen_radar.keys(), cores_cen):
    vals = cen_radar[nome] + [cen_radar[nome][0]]
    ax4.plot(angles, vals, color=cor, linewidth=2.5, label=nome)
    ax4.fill(angles, vals, color=cor, alpha=0.15)

ax4.set_xticks(angles[:-1])
ax4.set_xticklabels(categorias, fontsize=8)
ax4.set_ylim(0, 5)
ax4.set_title('Radar — Cenários em 5 Dimensões', fontsize=10, pad=15)
ax4.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=8)

plt.savefig('fig_bivariada_36.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 37. Recomendação Final — Cenário C: Combinação

> **✅ IMPLEMENTAR CENÁRIO C — COMBINAÇÃO: 50% Realocação PE541 + 50% Rodoviário SP→BA**
>
> **Custo: R$458.858 | Lead: 6 dias | DOI Risk: Baixo | ROI Mar-Jun: ~730%**

### 5 Razões para o Cenário C

1. **Velocidade:** Lead time de 6 dias garante produto em fevereiro — antes que Mapapi e NE Norte zerem o estoque
2. **Custo:** 23% mais barato que o Rodoviário puro (B), economizando R$137.902
3. **Margem:** R$182,99/HL de margem residual vs. R$152,39/HL no B
4. **Resiliência ao BIAS:** Mantém retorno positivo até BIAS de -18% (vs. -4,7% no B)
5. **Patagonia protegida:** Reduz Patagonia apenas em -2.250 HL (metade do A), DOI permanece acima de 12 dias

### O que acontece se NÃO agirmos?

| Sub-Região | Ação (Cenário C) | Inação |
|---|---|---|
| Mapapi (34,6%) | DOI recuperado para 15+ dias em W1 | **Ruptura imediata (DOI=-3,2d)** — perda de R$443k |
| NE Norte (15,4%) | DOI recuperado para 14+ dias em W1 | **Ruptura imediata (DOI=-2,3d)** — perda de R$197k |
| NO Centro (16,8%) | DOI estabilizado em 14+ dias | Ruptura em W2 — perda de R$215k |
| NE Sul (24,1%) | DOI mantido em 16+ dias | Alerta W3, ruptura W4 — perda de R$308k |

**Custo total da inação estimado: R$1.163.000 em receita perdida (4.500 HL × R$285/HL × % não atendido)**

### Plano de Ação Semanal

| Semana | Data | Ação | Responsável | Entregável |
|---|---|---|---|---|
| W0 | 02/02 (D+0) | Aprovar Cenário C em S&OP emergencial | Diretoria NENO | Ata de aprovação |
| W0 | 02/02 (D+0) | Emitir NF rodoviária SP→BA (2.363 HL bruto) | Logística/PCP | NF emitida |
| W0 | 03/02 (D+1) | Reprojetar PE541: +2.250 HL Malzbier, -2.250 HL Patagonia W1 | PCP PE541 | Plano PCP atualizado |
| W1 | 05/02 (D+3) | Receber realocação em CDR João Pessoa (2.250 HL) | Logística NENO | Recebimento confirmado |
| W1 | 08/02 (D+6) | Receber rodoviário em CDR Camaçari/João Pessoa (2.250 HL liq.) | Logística NENO | Recebimento confirmado |
| W2 | 09/02 | Verificar DOI por sub-região — todos devem estar >12d | Comercial/Financeiro | Relatório DOI |
| W2-W4 | 16-28/02 | Monitorar DOI semanal e ajustar se necessário | S&OP NENO | Dashboard semanal |
| W4+ | 28/02+ | Avaliar demanda real vs forecast — recalibrar BIAS | Finance/S&OP | Análise de BIAS |

### KPIs de Controle

| KPI | Meta | Alerta | Crítico |
|---|---|---|---|
| DOI Malzbier NENO | ≥ 15 dias | < 12 dias | < 7 dias |
| DOI Patagonia NENO | ≥ 12 dias | < 10 dias | < 7 dias |
| Custo frete realizado | ≤ R$458.858 | > R$500k | > R$600k |
| Volume Malzbier entregue | ≥ 4.500 HL | < 4.000 HL | < 3.500 HL |
| BIAS realizado fev. | < ±5% | 5%-10% | > 10% |

### Registro de Riscos

| Risco | Probabilidade | Impacto | Mitigação |
|---|---|---|---|
| Patagonia abaixo do DOI mínimo | Média | Alto | Monitorar diariamente; priorizar cabotagem compensação em março |
| Rodoviário com mais de 5% de quebra | Baixa | Médio | Contratar seguro de carga; inspecionar veículos |
| Demanda real 15% abaixo do forecast | Média | Médio | Revisar pedidos semanalmente; flexibilidade de cancelamento D+2 |
| Concorrente ocupa gôndola durante ruptura | Alta (sem ação) | Muito Alto | Garantir estoque mínimo em PDV críticos |
| Goose Island sem produção em PE541 | Baixa | Muito Alto | Confirmar restrição de elaboração; plano B: AQ541 |

### Posicionamento para Março e Além

A demanda de +10%/mês a partir de março significa que em abril (Mar × 1,1) a demanda já supera toda a capacidade NE disponível. A **próxima decisão estratégica** é:
1. Contratar capacidade adicional na PE541 (expansão de linha)
2. Negociar acordo de fornecimento inter-regional permanente com SP
3. Revisar o mix de produção PE541 para maximizar Malzbier nos meses de alta demanda


In [ ]:
import matplotlib.gridspec as gridspec
# Seção 37 — Dashboard Executivo Final — Cenário C

fig = plt.figure(figsize=(20, 14))
gs_dash = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)
fig.suptitle('Seção 37 — Dashboard Executivo: Recomendação Final — Cenário C',
             fontsize=14, fontweight='bold', color=CORES['azul_escuro'])
fig.patch.set_facecolor(CORES['bg'])

# ── Painel 1 (topo esquerdo): KPI Box ────────────────────────────────────
ax1 = fig.add_subplot(gs_dash[0, 0])
ax1.set_facecolor(CORES['azul_escuro'])
ax1.axis('off')

kpis = [
    ('CENÁRIO RECOMENDADO', 'C — COMBINAÇÃO', CORES['ambar']),
    ('Volume Malzbier', '4.500 HL/mês', CORES['branco']),
    ('Custo Total', 'R$ 458.858', CORES['branco']),
    ('Custo/HL', 'R$ 102,01', CORES['branco']),
    ('Lead Time', '6 dias (D+6)', CORES['verde']),
    ('DOI Risk', 'BAIXO', CORES['verde']),
    ('Margem Residual/HL', 'R$ 182,99', CORES['verde']),
    ('ROI Mar-Jun', '~730%', CORES['ambar']),
    ('Payback', 'Março', CORES['ambar']),
]

for i, (label, val, color) in enumerate(kpis):
    y = 0.95 - i * 0.105
    ax1.text(0.03, y, label + ':', transform=ax1.transAxes,
             fontsize=8.5, color=CORES['cinza_medio'], va='top')
    ax1.text(0.97, y, val, transform=ax1.transAxes,
             fontsize=9, color=color, va='top', ha='right', fontweight='bold')
ax1.set_title('KPIs — Cenário C', fontsize=10, color=CORES['branco'],
              pad=5, fontweight='bold')

# ── Painel 2 (topo centro): Comparação cenários ───────────────────────────
ax2 = fig.add_subplot(gs_dash[0, 1])
cen_names = ['A', 'B', 'C']
custos_all = [321336, 596760, 458858]
margens_all = [(285 - 71.41) * 4500, (285 - 132.61) * 4500, (285 - 102.01) * 4500]

x_c = np.arange(3)
bars_c = ax2.bar(x_c, custos_all, 0.5, color=cores_cen, alpha=0.85, label='Custo Total')
bars_m = ax2.bar(x_c, margens_all, 0.5, bottom=custos_all, color=cores_cen, alpha=0.35, hatch='//', label='Margem Residual')
for i, (c, m) in enumerate(zip(custos_all, margens_all)):
    ax2.text(i, c/2, f'R${c/1000:.0f}k', ha='center', va='center', fontsize=8, color='white', fontweight='bold')
    ax2.text(i, c + m/2, f'R${m/1000:.0f}k', ha='center', va='center', fontsize=7, color=cores_cen[i])
ax2.set_xticks(x_c)
ax2.set_xticklabels(['A — Realoc.', 'B — Rodo', 'C — Comb.'])
ax2.set_ylabel('R$')
ax2.set_title('Custo (sólido) + Margem (hachura)', fontsize=10)
ax2.legend(fontsize=8)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'R${v:,.0f}'))

# ── Painel 3 (topo direito): Mapa geográfico por tratamento ──────────────
ax3 = fig.add_subplot(gs_dash[0, 2])
ax3.axis('off')
geo_info = [
    ['Sub-Região', '% Dem', 'DOI W1', 'Ação Cenário C', 'Prioridade'],
    ['Mapapi', '34,6%', '-3,2d', 'Realoc. + Rodo → JP', '🔴 1'],
    ['NE Norte', '15,4%', '-2,3d', 'Realoc. + Rodo → JP', '🔴 2'],
    ['NO Centro', '16,8%', '4,8d', 'Rodo → JP', '🟠 3'],
    ['NE Sul', '24,1%', '9,8d', 'Rodo → Camaçari', '🟡 4'],
    ['NO Araguaia', '0,6%', '-6,3d', 'DESPRIORITIZAR', '⚪ 5'],
]
tbl3 = ax3.table(cellText=geo_info[1:], colLabels=geo_info[0], loc='center', cellLoc='center')
tbl3.auto_set_font_size(False)
tbl3.set_fontsize(8.5)
tbl3.scale(1.1, 1.5)
row_colors3 = ['#FFCCCC', '#FFCCCC', '#FFE0B2', '#FFF9C4', '#F5F5F5']
for i, rc in enumerate(row_colors3):
    for j in range(5):
        tbl3[i+1, j].set_facecolor(rc)
for j in range(5):
    tbl3[0, j].set_facecolor(CORES['azul_escuro'])
    tbl3[0, j].set_text_props(color='white', fontweight='bold')
ax3.set_title('Ações por Sub-Região — Cenário C', fontsize=10, pad=20)

# ── Painel 4 (baixo esquerdo): DOI evolução ───────────────────────────────
ax4 = fig.add_subplot(gs_dash[1, 0])
sub_regioes_plot = ['Mapapi', 'NE Norte', 'NO Centro', 'NE Sul']
for i, sr in enumerate(sub_regioes_plot):
    ax4.plot(semana_nums, doi_baseline[sr], 'o--', color=cores_sr[i], linewidth=1.5, alpha=0.5)
    ax4.plot(semana_nums, doi_cenC[sr], 's-', color=cores_sr[i], linewidth=2.5, label=sr)
ax4.axhline(DOI_MIN, color=CORES['vermelho'], linestyle='--', linewidth=2, label=f'DOI Min ({DOI_MIN}d)')
ax4.fill_between([0.8, 4.2], [0, 0], [DOI_MIN, DOI_MIN], alpha=0.07, color=CORES['vermelho'])
ax4.set_xticks(semana_nums)
ax4.set_xticklabels(w_labels)
ax4.set_ylabel('DOI (dias)')
ax4.set_title('DOI — Sem Ação (tracejado) vs Cenário C (sólido)', fontsize=9)
ax4.legend(fontsize=7, ncol=2)

# ── Painel 5 (baixo centro): Gantt de ações ───────────────────────────────
ax5 = fig.add_subplot(gs_dash[1, 1])
acoes = [
    ('Aprovação S&OP', 0, 1, CORES['azul_escuro']),
    ('Emitir NF Rodoviário', 0, 1, CORES['ambar']),
    ('Reprojetar PE541', 1, 2, CORES['azul_medio']),
    ('Produção Malzbier PE541', 2, 5, CORES['azul_claro']),
    ('Transporte Rodoviário SP→BA', 1, 6, CORES['ambar']),
    ('Recebimento JP (Realoc.)', 3, 4, CORES['verde']),
    ('Recebimento CDRs (Rodo)', 6, 7, CORES['verde']),
    ('Distribuição PDV', 7, 14, CORES['verde']),
    ('Monitoramento DOI W2-W4', 7, 28, CORES['cinza_medio']),
    ('Revisão BIAS pós-fev.', 28, 32, CORES['azul_escuro']),
]
for i, (nome, inicio, fim, cor) in enumerate(acoes):
    ax5.barh(i, fim - inicio, left=inicio, height=0.5, color=cor, alpha=0.85)
    ax5.text(inicio + (fim - inicio) / 2, i, nome, ha='center', va='center',
             fontsize=6.5, color='white', fontweight='bold')
ax5.axvline(0, color=CORES['azul_escuro'], linewidth=1.5, linestyle='-', label='D+0 (02/02)')
ax5.axvline(6, color=CORES['verde'], linewidth=1.5, linestyle='--', label='D+6: Rodo chega')
ax5.axvline(28, color=CORES['ambar'], linewidth=1.5, linestyle=':', label='D+28: Fim fev.')
ax5.set_xlabel('Dias após D+0 (02/02/2026)')
ax5.set_yticks([])
ax5.set_title('Timeline de Ações — D+0 a D+32', fontsize=9)
ax5.legend(fontsize=7)

# ── Painel 6 (baixo direito): Árvore de decisão textual ──────────────────
ax6 = fig.add_subplot(gs_dash[1, 2])
ax6.set_facecolor('#F8F9FA')
ax6.axis('off')
decision_text = (
    "ÁRVORE DE DECISÃO\n"
    "━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n"
    "Gap = 4.500 HL/mês (confirmado)\n"
    "        │\n"
    "        ├─ Lead time aceitável (<7d)?\n"
    "        │        └─ SIM → Rodo ou Combinar\n"
    "        │\n"
    "        ├─ Margem > R$100/HL necessária?\n"
    "        │        └─ SIM → NÃO Rodo puro (B)\n"
    "        │\n"
    "        ├─ DOI Patagonia seguro?\n"
    "        │        └─ SIM → NÃO Realoc. pura (A)\n"
    "        │\n"
    "        └─ RESULTADO: CENÁRIO C\n"
    "             50% Realoc. + 50% Rodo\n"
    "             Custo: R$458.858\n"
    "             Lead: 6 dias\n"
    "             DOI Risk: BAIXO\n"
    "             ✅ IMPLEMENTAR"
)
ax6.text(0.05, 0.95, decision_text, transform=ax6.transAxes,
         fontsize=8.5, va='top', ha='left', family='monospace',
         color=CORES['azul_escuro'],
         bbox=dict(boxstyle='round,pad=0.5', facecolor=CORES['cinza_claro'], alpha=0.8))
ax6.set_title('Árvore de Decisão — Lógica do Cenário C', fontsize=9)

plt.savefig('fig_bivariada_37.png', dpi=150, bbox_inches='tight')
plt.show()


## ✅ Recomendação Final — Cenário C (Mix Produção Local + Cabotagem)

### Plano de Ação em 3 Horizontes

| Horizonte | Ação | Volume | Prazo | Responsável |
|---|---|---|---|---|
| **Imediato (W1)** | Realocar 4.500 HL capacidade ociosa PE541 para Malzbier | 4.500 HL | Semana 1 | PCP PE541 |
| **Curto Prazo (W2–W3)** | Embarcar cabotagem SP→Camaçari (4.500 HL/mês) | 4.500 HL | Semanas 2–3 | Logística |
| **Médio Prazo (W4+)** | Reduzir Colorado/Patagonia NENO em 15% para liberar capacidade | Permanente | W4 em diante | S&OP |

### Impacto Financeiro Consolidado

| Métrica | Valor |
|---|---|
| MACO em risco (sem ação) | R$ 3.328.800 |
| Custo cabotagem (4.500 HL × R$89,96) | R$ 404.820 |
| **Margem líquida preservada** | **R$ 2.923.980** |
| ROI da ação | **~722%** |

### Premissas e Sensibilidade

- Resultado robusto mesmo com BIAS +9% de demanda real (DOI W4 > 7 dias em Cenário C)
- Se demanda vier −9% abaixo: estoque excedente de ~650 HL — risco gerenciável
- Se cabotagem atrasar >1 semana: Mapapi e NE Norte entram em ruptura — monitorar diariamente